# Unit 2. Multiple Regression I: Matrix Theory

精算統計模型 (SOA Exam SRM/PA). The text of [unit02.pdf](https://github.com/chang-ye-tu/asm/blob/master/note/unit02.pdf) with every R chunk as a code cell.

<a href="https://colab.research.google.com/github/chang-ye-tu/asm/blob/master/colab/unit02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook runs on the R runtime (on Colab: Runtime › Change runtime type › R). Run the setup cell first: it installs the packages and downloads the data this unit uses.

In [ ]:
# Setup: run this cell first. It installs the packages this unit uses and downloads
# its data files into data/, the paths the code below reads.
local({
  if (is.null(getOption("repos")) || identical(unname(getOption("repos")["CRAN"]), "@CRAN@"))
    options(repos = c(CRAN = "https://cloud.r-project.org"))
  os <- tryCatch(readLines("/etc/os-release"), error = function(e) character())
  code <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
  if (length(code) == 1 && nzchar(code))  # Linux, e.g. Colab: prebuilt binary packages
    options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", code)),
            HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(), paste(getRversion(),
              R.version$platform, R.version$arch, R.version$os)))
})
pkgs <- c("ISLR2")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new)) install.packages(new)
data_files <- c("data/frees/csv/AnscombesData.csv", "data/frees/csv/CAPM.csv", "data/frees/csv/IPO.csv", "data/frees/csv/NAICExpense.csv", "data/frees/csv/OutlierExample.csv", "data/frees/csv/TermLife.csv", "data/frees/csv/UNLifeExpectancy.csv", "data/frees/csv/WiscLottery.csv", "data/frees/csv/WiscNursingHome.csv", "data/islr/Advertising.csv")
for (f in data_files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  url <- if (startsWith(f, "data/islr/"))
    paste0("https://www.statlearning.com/s/", basename(f)) else
    paste0("https://raw.githubusercontent.com/chang-ye-tu/asm/master/note/", f)
  download.file(url, f, quiet = TRUE, mode = "wb")
}
options(width = 62, digits = 4, scipen = 4, repr.plot.width = 6.4, repr.plot.height = 3.0)

In [ ]:
library(ISLR2)

## 1 Data, Correlation and the Model

> **Reading.** FREES §2.1–2.2, §3.1, §3.2.1–3.2.3 · ISLR §3.1, §3.2 · CLM §1. SRM learning outcomes 2(a), 2(c).

### 1.1 Scatter plots and correlation

**Example 2.1** (Wisconsin lottery sales; FREES §2.1). Average weekly online lottery sales in dollars (`SALES`) over forty weeks of 1998–99 and the population (`POP`) of $50$ Wisconsin ZIP-code areas chosen at random.

In [ ]:
lottery <- read.csv("data/frees/csv/WiscLottery.csv")
descr <- function(v) c(mean = mean(v), median = median(v), sd = sd(v), min = min(v),
                       max = max(v))
round(rbind(POP = descr(lottery$POP), SALES = descr(lottery$SALES)), 1)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
hist(lottery$POP, breaks = 15, main = "", xlab = "POP", col = "grey85", border = "white")
hist(lottery$SALES, breaks = 15, main = "", xlab = "SALES", col = "grey85", border = "white")
plot(lottery$POP, lottery$SALES, xlab = "POP", ylab = "SALES", pch = 16, cex = 0.7)
par(op)

*Figure 2.1.* The lottery data: histograms of population and of sales, and the scatter plot of sales against population (FREES Figures 2.1–2.2).

Both variables are skewed to the right, each mean well above its median; the scatter plot shows sales rising with population.

**Definition 2.1** (Pearson correlation; FREES §2.1, ISLR §3.1.3; CLM Definition 2.15). For pairs $(x_i,y_i)$, $i=1,\ldots,n$, of variables that are not constant, with $s_x^2=(n-1)^{-1}\sum_i(x_i-\bar x)^2$ and $s_y^2$ likewise,

$$
r = \frac{1}{(n-1)s_xs_y}\sum_{i=1}^n(x_i-\bar x)(y_i-\bar y)
  = \frac{\sum_i(x_i-\bar x)(y_i-\bar y)}
         {\sqrt{\sum_i(x_i-\bar x)^2}\sqrt{\sum_i(y_i-\bar y)^2}} . \tag{1}
$$

The variables are **positively correlated** if $r>0$ and **negatively correlated** if $r<0$.

**Proposition 2.2** (Bounds and invariance; CLM Proposition 2.17). For variables that are not constant, $-1\leqslant r\leqslant1$; $r=1$ iff the points lie on a line of positive slope and $r=-1$ iff they lie on a line of negative slope. Replacing $x_i$ by $a+bx_i$ and $y_i$ by $c+dy_i$ leaves $r$ unchanged when $bd>0$ and changes its sign when $bd<0$.

**Proof.** **Step 1: the bound.** Put $u_i=x_i-\bar x$ and $v_i=y_i-\bar y$. By the Cauchy–Schwarz inequality of the prerequisites, $(\sum_iu_iv_i)^2\leqslant(\sum_iu_i^2)(\sum_iv_i^2)$, with equality iff the vectors $(u_i)$ and $(v_i)$ are proportional. When neither variable is constant the right side is positive, and dividing by it gives $r^2\leqslant1$.

**Step 2: the cases of equality.** Equality means $v_i=cu_i$ for every $i$ and some $c\neq0$, that is $y_i-\bar y=c(x_i-\bar x)$: the points lie on the line through $(\bar x,\bar y)$ of slope $c$, and then $r=c\sum_iu_i^2\big/\sqrt{\sum_iu_i^2\cdot c^2\sum_iu_i^2}=c/|c|$, which is $1$ for $c>0$ and $-1$ for $c<0$. Conversely, points on a line $y=\alpha+\beta x$ with $\beta\neq0$ have $v_i=\beta u_i$, so equality holds.

**Step 3: invariance.** The transformed deviations are $bu_i$ and $dv_i$, so the numerator of (1) is multiplied by $bd$ and the denominator by $|b|\,|d|$; the ratio is $r$ when $bd>0$ and $-r$ when $bd<0$. ∎

**Remark** (What $r$ measures; FREES §2.1). $r$ is dimensionless and measures **linear** association only. The five points $(-2,4),(-1,1),(0,0),(1,1),(2,4)$ lie on $y=x^2$ and have $r=0$: with $\bar x=0$ and $\bar y=2$, $\sum_ix_i(y_i-2)=(-2)(2)+(-1)(-1)+0+(1)(-1)+(2)(2)=0$.

**Example 2.2** (Lottery sales, continued; FREES §2.1).

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
cor(lottery$POP, lottery$SALES)

Sales and population are strongly and positively correlated.

**Example 2.3** (Summarizing simulations; FREES §2.1, Figure 2.3). Manistre and Hancock's put option: a stock worth \$100 now is worth $S(Z)=100\exp(0.08\cdot10+0.15\sqrt{10}\,Z)$ in ten years, $Z$ standard normal, and a put with strike \$110 has present value $C(Z)=e^{-0.06\cdot10}\max\{0,110-S(Z)\}$. Each of $1000$ replicates simulates $1000$ values of $C$ and records their $95$th percentile, the value at risk $\text{VaR}$, and the mean of the largest $50$, the conditional tail expectation $\text{CTE}$:

In [ ]:
options(repr.plot.width = 4.2, repr.plot.height = 2.5)
set.seed(1)
Cput <- function(z) exp(-0.6) * pmax(0, 110 - 100 * exp(0.8 + 0.15 * sqrt(10) * z))
vc <- t(replicate(1000, { v <- sort(Cput(rnorm(1000)))
  c(VaR = v[950], CTE = mean(v[951:1000])) }))
cor(vc[, "VaR"], vc[, "CTE"])
par(mar = c(4, 4.2, 0.5, 0.5), cex = 0.85)
plot(vc, pch = 16, cex = 0.4, col = "grey40")

The correlation is $0.78$, FREES's $0.782$ up to simulation noise: a strong but not perfect linear relation between two risk measures computed from the same simulated values. The summaries of this unit apply to the output of a simulation study as they do to data.

**Example 2.4** (Term life insurance; FREES §3.1). The 2004 Survey of Consumer Finances: of $500$ households with positive income, the $275$ that bought term life insurance, with the policy face amount `FACE`, annual `INCOME`, years of `EDUCATION` of the respondent and the household size `NUMHH`. Face amount and income are analysed in logarithms.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
term <- subset(read.csv("data/frees/csv/TermLife.csv"), FACE > 0)
term$LNFACE <- log(term$FACE); term$LNINCOME <- log(term$INCOME)
nrow(term)
round(t(sapply(term[, c("FACE", "INCOME", "EDUCATION", "NUMHH", "LNFACE", "LNINCOME")],
               descr)), 3)
round(cor(term[, c("NUMHH", "EDUCATION", "LNINCOME", "LNFACE")]), 3)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
plot(term$INCOME, term$FACE, xlab = "INCOME", ylab = "FACE", pch = 16, cex = 0.5)
plot(term$LNINCOME, term$LNFACE, xlab = "LNINCOME", ylab = "LNFACE", pch = 16, cex = 0.5)
par(op)

*Figure 2.2.* Face amount against income, in dollars and in logarithms (FREES Figure 3.1).

In dollars the points cluster in the lower left corner and the relation is far from linear; in logarithms a line is a reasonable summary. The correlation matrix summarises the pairwise scatter plots: each entry is a pairwise linear association and cannot show how the response depends on several variables jointly.

In [ ]:
options(repr.plot.width = 4, repr.plot.height = 3)
pairs(term[, c("NUMHH", "EDUCATION", "LNINCOME", "LNFACE")], pch = 16, cex = 0.35,
      col = "grey30", gap = 0.3)

*Figure 2.3.* The scatterplot matrix of the four variables, whose correlations are printed above (FREES Figure 3.2): `LNFACE` rises with `LNINCOME` and `EDUCATION`; `NUMHH` takes few values.

**Example 2.5** (Advertising; ISLR §2.1, §3.1–3.2). Sales of a product in thousands of units (`sales`) in $200$ markets, with the advertising budgets in thousands of dollars for `TV`, `radio` and `newspaper`. The file is distributed at the book's site and is not in `ISLR2`.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
adv <- read.csv("data/islr/Advertising.csv")[, -1]
round(cor(adv), 4)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
for (v in c("TV", "radio", "newspaper"))
  plot(adv[[v]], adv$sales, xlab = v, ylab = "sales", pch = 16, cex = 0.5)
par(op)

*Figure 2.4.* Sales against each advertising budget (ISLR Figure 2.1).

### 1.2 The model and its assumptions

**Definition 2.3** (Multiple linear regression model; FREES §3.2, ISLR §3.2; CLM Definition 1.1, equation (2)). Given $n$ observations, each consisting of a response $Y_i$ and $k$ explanatory variables $X_{i1},\ldots,X_{ik}$, the **multiple linear regression model** is

$$
Y_i = \beta_0 + \beta_1 X_{i1} + \beta_2 X_{i2} + \cdots + \beta_k X_{ik} + \varepsilon_i,
\qquad i = 1,\ldots,n, \tag{2}
$$

or in matrix form

$$
\mathbf{Y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon} , \tag{3}
$$

$$
\mathbf{Y} = \begin{pmatrix} Y_1 \\ \vdots \\ Y_n \end{pmatrix}, \quad
\mathbf{X} = \begin{pmatrix} 1 & X_{11} & \cdots & X_{1k} \\
                      \vdots & \vdots & \ddots & \vdots \\
                      1 & X_{n1} & \cdots & X_{nk} \end{pmatrix}, \quad
\boldsymbol{\beta} = \begin{pmatrix} \beta_0 \\ \beta_1 \\ \vdots \\ \beta_k \end{pmatrix}, \quad
\boldsymbol{\varepsilon} = \begin{pmatrix} \varepsilon_1 \\ \vdots \\ \varepsilon_n \end{pmatrix}.
$$

$\mathbf{X}$ is the $n\times(k+1)$ **design matrix**, whose first column $\mathbf{1}$ carries the intercept; $\mathbf{x}_i'$ is its $i$-th row, so $Y_i=\mathbf{x}_i'\boldsymbol{\beta}+\varepsilon_i$. The **regression function** is $\operatorname{\mathsf{E}}[Y\mid X_1,\ldots,X_k]=\beta_0+\beta_1X_1+\cdots+\beta_kX_k$, linear in the parameters.

**Remark** (Counts and notation; FREES §3.1, ISLR §3.2). $k$ is the number of explanatory variables, $k+1$ the number of parameters and of columns of $\mathbf{X}$, and $n-k-1$ the residual degrees of freedom. FREES writes $k$ and $\mathbf b$ for the estimate of $\boldsymbol{\beta}$; ISLR writes $p$ for $k$ and $\hat\beta$; the SOA questions use both. The case $k=1$, $\mathbf{X}=[\mathbf{1}\ \mathbf{x}]$, is FREES's **basic** and ISLR's **simple** linear regression, and fitting (3) is called **regressing $Y$ on $X_1,\ldots,X_k$**.

**Definition 2.4** (Classical assumptions; FREES §2.2, §3.2.3, ISLR §3.1.2; CLM Definition 1.2). On model (3):

(A1) **Linearity**: $\operatorname{\mathsf{E}}[\mathbf{Y}\mid\mathbf{X}] = \mathbf{X}\boldsymbol{\beta}$.

(A2) **Zero conditional mean**: $\operatorname{\mathsf{E}}[\boldsymbol{\varepsilon}\mid\mathbf{X}] = \mathbf{0}$.

(A3) **Spherical errors**: $\operatorname{var}(\boldsymbol{\varepsilon}\mid\mathbf{X}) = \sigma^2\mathbf{I}_n$ with $\sigma^2>0$, that is $\operatorname{var}(\varepsilon_i)=\sigma^2$ for every $i$ (**homoscedasticity**) and $\operatorname{cov}(\varepsilon_i,\varepsilon_j)=0$ for $i\neq j$.

(A4) **Full rank**: $\operatorname{rank}(\mathbf{X})=k+1$, which requires $n\geqslant k+1$; inference requires $n>k+1$.

(A5) **Normality**: $\boldsymbol{\varepsilon}\mid\mathbf{X} \sim N(\mathbf{0},\sigma^2\mathbf{I}_n)$.

A1–A4 are the **Gauss–Markov assumptions**; A5 is used only for exact finite-sample distributions. $\mathbf{X}$ is treated as fixed, and every expectation, variance and distribution is conditional on it, with the conditioning suppressed.

**Remark** (FREES's two representations; FREES §2.2, §3.2.3). FREES states the model through **observables** assumptions F1–F5, namely $\operatorname{\mathsf{E}}[y_i]=\mathbf{x}_i'\boldsymbol{\beta}$, nonstochastic $x_{ij}$, $\operatorname{var}(y_i)=\sigma^2$, independent $y_i$, normal $y_i$, or through **error** assumptions E1–E5 on $\varepsilon_i=y_i-\mathbf{x}_i'\boldsymbol{\beta}$. The two sets are equivalent, and A1–A3 and A5 are their matrix form with independence weakened to zero covariance, which is all the proofs use; A4 is FREES's requirement that the explanatory variables are not linear combinations of one another. Nonstochastic $x$'s are motivated by stratified sampling: at each value of $\mathbf{x}_i$ a response is drawn from the population at that value, so $\mathbf{X}$ is fixed and the responses are independent. Since $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$, A1 and A2 are the same statement.

**Remark** (Interpretation of the coefficients; FREES §3.2.1, ISLR §3.1.2, §3.2.1). $\beta_0=\operatorname{\mathsf{E}}[Y\mid X_1=\cdots=X_k=0]$ sets the height of the plane and is rarely of interest. For $j\geqslant1$, $\beta_j=\partial\,\operatorname{\mathsf{E}}[Y\mid X_1,\ldots,X_k]/\partial X_j$ is the expected change in $Y$ per unit change in $X_j$ with the other explanatory variables held fixed; ISLR's phrase is the average effect on $Y$ of a one-unit increase in $X_j$, holding all other predictors fixed. For $k=1$ the equation $Y=\beta_0+\beta_1X+\varepsilon$ is the **population regression line**, and $\varepsilon$ collects what the line misses: curvature, omitted variables and measurement error.

## 2 Least Squares

> **Reading.** FREES §2.1, §3.1, §3.2.2 · ISLR §3.1.1, §3.2.1 · CLM §2. SRM learning outcomes 2(a), 2(c).

### 2.1 The normal equations

**Definition 2.5** (Least squares; FREES §2.1, §3.1, ISLR §3.1.1; CLM Definition 2.1). The **least squares estimator** $\hat{\boldsymbol{\beta}}$ minimises the **residual sum of squares**

$$
Q(\boldsymbol{\beta}^\ast)=\sum_{i=1}^n(Y_i-\mathbf{x}_i'\boldsymbol{\beta}^\ast)^2
=(\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}^\ast)'(\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}^\ast)=\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}^\ast\|^2 \tag{4}
$$

over $\boldsymbol{\beta}^\ast\in\mathbb{R}^{k+1}$. The **fitted values** are $\hat Y_i=\mathbf{x}_i'\hat{\boldsymbol{\beta}}$, the **residuals** $e_i=Y_i-\hat Y_i$, and $\hat y=\hat\beta_0+\hat\beta_1x_1+\cdots+\hat\beta_kx_k$ is the **fitted regression plane**, a line when $k=1$.

**Theorem 2.6** (The least squares estimator; CLM Theorem 2.4, equation (6)). Under A4 the minimiser of (4) is unique and equals

$$
\hat{\boldsymbol{\beta}}=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{Y} . \tag{5}
$$

**Proof.** **Step 1: expand.** The scalar $\mathbf{Y}'\mathbf{X}\boldsymbol{\beta}$ equals its transpose $\boldsymbol{\beta}'\mathbf{X}'\mathbf{Y}$, so

$$
Q(\boldsymbol{\beta})=\mathbf{Y}'\mathbf{Y}-\mathbf{Y}'\mathbf{X}\boldsymbol{\beta}-\boldsymbol{\beta}'\mathbf{X}'\mathbf{Y}+\boldsymbol{\beta}'\mathbf{X}'\mathbf{X}\boldsymbol{\beta}
=\mathbf{Y}'\mathbf{Y}-2\boldsymbol{\beta}'\mathbf{X}'\mathbf{Y}+\boldsymbol{\beta}'\mathbf{X}'\mathbf{X}\boldsymbol{\beta} . \tag{6}
$$

**Step 2: differentiate.** By the two rules of matrix differentiation of the prerequisites, $\partial(\mathbf{a}'\boldsymbol{\beta})/\partial\boldsymbol{\beta}'=\mathbf{a}'$ and $\partial(\boldsymbol{\beta}'\mathbf{A}\boldsymbol{\beta})/\partial\boldsymbol{\beta}'=2\boldsymbol{\beta}'\mathbf{A}$ for symmetric $\mathbf{A}$, applied with $\mathbf{a}=\mathbf{X}'\mathbf{Y}$ and $\mathbf{A}=\mathbf{X}'\mathbf{X}$,

$$
\frac{\partial Q}{\partial\boldsymbol{\beta}'}=-2\mathbf{Y}'\mathbf{X}+2\boldsymbol{\beta}'\mathbf{X}'\mathbf{X} .
$$

Setting this to $\mathbf{0}'$ and transposing gives the **normal equations**

$$
\mathbf{X}'\mathbf{X}\hat{\boldsymbol{\beta}}=\mathbf{X}'\mathbf{Y} . \tag{7}
$$

Row $j$ of (7) is $\sum_ix_{ij}(Y_i-\mathbf{x}_i'\hat{\boldsymbol{\beta}})=0$, which is $\partial Q/\partial\beta_j=0$; for $k=1$ the two rows are $\sum_i(Y_i-b_0-b_1x_i)=0$ and $\sum_ix_i(Y_i-b_0-b_1x_i)=0$.

**Step 3: $\mathbf{X}'\mathbf{X}$ is invertible.** $\mathbf{X}'\mathbf{X}$ and $\mathbf{X}$ have the same null space: $\mathbf{X}\mathbf{v}=\mathbf{0}$ gives $\mathbf{X}'\mathbf{X}\mathbf{v}=\mathbf{0}$, and $\mathbf{X}'\mathbf{X}\mathbf{v}=\mathbf{0}$ gives $\mathbf{v}'\mathbf{X}'\mathbf{X}\mathbf{v}=\|\mathbf{X}\mathbf{v}\|^2=0$, hence $\mathbf{X}\mathbf{v}=\mathbf{0}$. Both matrices have $k+1$ columns, so by the rank–nullity theorem of the prerequisites they have the same rank, which is $k+1$ under A4. The square matrix $\mathbf{X}'\mathbf{X}$ is therefore invertible and (7) has the unique solution (5).

**Step 4: it is the minimum.** For any $\boldsymbol{\beta}$ put $\mathbf{d}=\boldsymbol{\beta}-\hat{\boldsymbol{\beta}}$. Expanding $Q(\boldsymbol{\beta})=\|(\mathbf{Y}-\mathbf{X}\hat{\boldsymbol{\beta}})-\mathbf{X}\mathbf{d}\|^2$ and using $\mathbf{X}'(\mathbf{Y}-\mathbf{X}\hat{\boldsymbol{\beta}})=\mathbf{0}$, which is (7) rearranged,

$$
Q(\boldsymbol{\beta})=Q(\hat{\boldsymbol{\beta}})-2\mathbf{d}'\mathbf{X}'(\mathbf{Y}-\mathbf{X}\hat{\boldsymbol{\beta}})+\mathbf{d}'\mathbf{X}'\mathbf{X}\mathbf{d}
=Q(\hat{\boldsymbol{\beta}})+\|\mathbf{X}\mathbf{d}\|^2\geqslant Q(\hat{\boldsymbol{\beta}}),
$$

with equality iff $\mathbf{X}\mathbf{d}=\mathbf{0}$, that is iff $\mathbf{d}=\mathbf{0}$ by A4. So $\hat{\boldsymbol{\beta}}$ is the unique global minimiser. ∎

**Remark** (Without full rank; FREES §3.1). Only Step 3 and the equality case of Step 4 use A4. If one column of $\mathbf{X}$ is a linear combination of the others, $\mathbf{X}'\mathbf{X}$ is singular, the normal equations have infinitely many solutions (one exists because the prerequisites' theorem **orthogonal projection** gives a closest point $\mathbf{X}\boldsymbol{\beta}$ of $\mathcal{C}(\mathbf{X})$ with $\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\perp\mathcal{C}(\mathbf{X})$, which is (7); adding any nonzero vector of the null space of $\mathbf{X}$ gives others), all with the same fitted vector by Step 4, and R reports `NA` for the redundant coefficients.

### 2.2 Orthogonal projection

**Theorem 2.7** (Least squares as projection; CLM Theorem 2.5). Under A4, $\hat{\mathbf{Y}}=\mathbf{X}\hat{\boldsymbol{\beta}}$ is the orthogonal projection of $\mathbf{Y}$ onto the column space $\mathcal{C}(\mathbf{X})$: it is the unique point of $\mathcal{C}(\mathbf{X})$ closest to $\mathbf{Y}$, and $\mathbf{Y}-\hat{\mathbf{Y}}\perp\mathcal{C}(\mathbf{X})$.

**Proof.** Minimising $\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\|^2$ over $\boldsymbol{\beta}$ is finding the point of $\mathcal{C}(\mathbf{X})$ closest to $\mathbf{Y}$.

**Step 1: closest iff orthogonal residual.** A point $\hat{\mathbf{Y}}\in\mathcal{C}(\mathbf{X})$ minimises $\|\mathbf{Y}-\tilde{\mathbf{Y}}\|^2$ over $\tilde{\mathbf{Y}}\in\mathcal{C}(\mathbf{X})$ iff $\mathbf{Y}-\hat{\mathbf{Y}}\perp\mathcal{C}(\mathbf{X})$. ($\Leftarrow$) For $\tilde{\mathbf{Y}}\in\mathcal{C}(\mathbf{X})$, $\hat{\mathbf{Y}}-\tilde{\mathbf{Y}}\in\mathcal{C}(\mathbf{X})$, so $(\mathbf{Y}-\hat{\mathbf{Y}})\perp(\hat{\mathbf{Y}}-\tilde{\mathbf{Y}})$ and Pythagoras gives

$$
\|\mathbf{Y}-\tilde{\mathbf{Y}}\|^2=\|\mathbf{Y}-\hat{\mathbf{Y}}\|^2+\|\hat{\mathbf{Y}}-\tilde{\mathbf{Y}}\|^2\geqslant\|\mathbf{Y}-\hat{\mathbf{Y}}\|^2,
$$

with equality iff $\tilde{\mathbf{Y}}=\hat{\mathbf{Y}}$. ($\Rightarrow$) If some unit vector $\mathbf{v}\in\mathcal{C}(\mathbf{X})$ had $\mathbf{v}'(\mathbf{Y}-\hat{\mathbf{Y}})\neq0$, then $\tilde{\mathbf{Y}}=\hat{\mathbf{Y}}+t\mathbf{v}\in\mathcal{C}(\mathbf{X})$ gives $\|\mathbf{Y}-\tilde{\mathbf{Y}}\|^2=\|\mathbf{Y}-\hat{\mathbf{Y}}\|^2-2t\,\mathbf{v}'(\mathbf{Y}-\hat{\mathbf{Y}})+t^2$, strictly smaller at $t=\mathbf{v}'(\mathbf{Y}-\hat{\mathbf{Y}})$, contradicting optimality.

**Step 2: the orthogonality condition is the normal equations.** $\mathbf{Y}-\hat{\mathbf{Y}}$ is orthogonal to $\mathcal{C}(\mathbf{X})$ iff it is orthogonal to each column of $\mathbf{X}$, that is $\mathbf{X}'(\mathbf{Y}-\hat{\mathbf{Y}})=\mathbf{0}$; with $\hat{\mathbf{Y}}=\mathbf{X}\hat{\boldsymbol{\beta}}$ this is (7), whose solution Theorem 2.6 gives. Uniqueness of the closest point is the equality case of Step 1. ∎

**Definition 2.8** (Hat matrix and residual maker; CLM Definition 2.6). Under A4,

$$
\mathbf{H} = \mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}', \qquad \mathbf{M} = \mathbf{I}_n - \mathbf{H} , \tag{8}
$$

so that $\hat{\mathbf{Y}} = \mathbf{H}\mathbf{Y}$ and $\mathbf{e} = \mathbf{M}\mathbf{Y}$.

**Theorem 2.9** (Properties of $\mathbf{H}$ and $\mathbf{M}$; CLM Theorem 2.9). (i) $\mathbf{H}$ and $\mathbf{M}$ are symmetric and idempotent.

(ii) $\mathbf{H}\mathbf{X} = \mathbf{X}$ and $\mathbf{M}\mathbf{X} = \mathbf{0}$.

(iii) $\mathbf{H}\mathbf{M} = \mathbf{M}\mathbf{H} = \mathbf{0}$.

(iv) $\operatorname{tr}(\mathbf{H}) = k+1$ and $\operatorname{tr}(\mathbf{M}) = n-k-1$.

**Proof.** **(i)** $(\mathbf{X}'\mathbf{X})^{-1}$ is symmetric, being the inverse of a symmetric matrix, so

$$
\mathbf{H}'=\mathbf{X}[(\mathbf{X}'\mathbf{X})^{-1}]'\mathbf{X}'=\mathbf{H} ,\qquad
\mathbf{H}^2=\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}\underbrace{\mathbf{X}'\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}}_{\mathbf{I}_{k+1}}\mathbf{X}'=\mathbf{H} .
$$

Then $\mathbf{M}'=\mathbf{I}-\mathbf{H}'=\mathbf{M}$ and $\mathbf{M}^2=\mathbf{I}-2\mathbf{H}+\mathbf{H}^2=\mathbf{I}-\mathbf{H}=\mathbf{M}$.

**(ii)** $\mathbf{H}\mathbf{X}=\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{X}=\mathbf{X}$, hence $\mathbf{M}\mathbf{X}=\mathbf{X}-\mathbf{X}=\mathbf{0}$.

**(iii)** $\mathbf{H}\mathbf{M}=\mathbf{H}-\mathbf{H}^2=\mathbf{0}$, and $\mathbf{M}\mathbf{H}=\mathbf{H}-\mathbf{H}^2=\mathbf{0}$.

**(iv)** By the prerequisites' lemma **trace equals rank** for idempotent matrices, $\operatorname{tr}(\mathbf{H})=\operatorname{rank}(\mathbf{H})$. Now $\mathcal{C}(\mathbf{H})=\mathcal{C}(\mathbf{X})$: every column of $\mathbf{H}$ is $\mathbf{X}$ times a vector, and $\mathbf{H}\mathbf{X}=\mathbf{X}$ from (ii) puts every column of $\mathbf{X}$ in $\mathcal{C}(\mathbf{H})$. By A4, $\dim\mathcal{C}(\mathbf{X})=k+1$, so $\operatorname{tr}(\mathbf{H})=k+1$ and $\operatorname{tr}(\mathbf{M})=\operatorname{tr}(\mathbf{I}_n)-\operatorname{tr}(\mathbf{H})=n-k-1$. ∎

**Remark**. $\mathbf{H}$ is symmetric, idempotent, with $\mathcal{C}(\mathbf{H})=\mathcal{C}(\mathbf{X})$; by the prerequisites' proposition **projections are the symmetric idempotents**, $\mathbf{H}$ is the orthogonal projection onto $\mathcal{C}(\mathbf{X})$, so Theorem 2.7 and Definition 2.8 describe one object, and $\mathbf{M}$ is the projection onto $\mathcal{C}(\mathbf{X})^{\perp}$, of dimension $n-k-1$.

**Lemma 2.10** (The intercept projection; CLM Remark 6.2, Lemma 7.2). Let $\mathbf{H}_0=\tfrac1n\mathbf{1}\mathbf{1}'$ and suppose $\mathbf{1}\in\mathcal{C}(\mathbf{X})$. Then $\mathbf{H}\mathbf{H}_0=\mathbf{H}_0\mathbf{H}=\mathbf{H}_0$; $\mathbf{H}-\mathbf{H}_0$ is symmetric idempotent with $\operatorname{tr}(\mathbf{H}-\mathbf{H}_0)=k$, and is the orthogonal projection onto $\mathcal{C}(\mathbf{X})\cap\mathbf{1}^\perp$; and $(\mathbf{H}-\mathbf{H}_0)\mathbf{M}=\mathbf{0}$.

**Proof.** $\mathbf{H}_0$ is symmetric, and $\mathbf{H}_0^2=\tfrac1{n^2}\mathbf{1}(\mathbf{1}'\mathbf{1})\mathbf{1}'=\mathbf{H}_0$ since $\mathbf{1}'\mathbf{1}=n$; so $\mathbf{H}_0$ is the projection onto $\mathcal{C}(\mathbf{1})$. Since $\mathbf{H}$ projects onto $\mathcal{C}(\mathbf{X})\ni\mathbf{1}$, $\mathbf{H}\mathbf{1}=\mathbf{1}$ and $\mathbf{H}\mathbf{H}_0=\tfrac1n(\mathbf{H}\mathbf{1})\mathbf{1}'=\mathbf{H}_0$; transposing, $\mathbf{H}_0\mathbf{H}=\mathbf{H}_0$. Hence $\mathbf{H}-\mathbf{H}_0$ is symmetric and

$$
(\mathbf{H}-\mathbf{H}_0)^2=\mathbf{H}-\mathbf{H}\mathbf{H}_0-\mathbf{H}_0\mathbf{H}+\mathbf{H}_0^2=\mathbf{H}-\mathbf{H}_0-\mathbf{H}_0+\mathbf{H}_0=\mathbf{H}-\mathbf{H}_0 ,
$$

with $\operatorname{tr}(\mathbf{H}-\mathbf{H}_0)=\operatorname{tr}(\mathbf{H})-\operatorname{tr}(\mathbf{H}_0)=(k+1)-1=k$ by Theorem 2.9(iv) and $\operatorname{tr}(\mathbf{H}_0)=\tfrac1n\operatorname{tr}(\mathbf{1}\mathbf{1}')=1$. Its column space: $(\mathbf{H}-\mathbf{H}_0)\mathbf{v}=\mathbf{H}\mathbf{v}-\mathbf{H}_0\mathbf{v}$ lies in $\mathcal{C}(\mathbf{X})$, and $\mathbf{1}'(\mathbf{H}-\mathbf{H}_0)\mathbf{v}=\mathbf{1}'\mathbf{H}\mathbf{v}-\mathbf{1}'\mathbf{H}_0\mathbf{v}=\mathbf{1}'\mathbf{v}-\mathbf{1}'\mathbf{v}=0$ using $\mathbf{1}'\mathbf{H}=(\mathbf{H}\mathbf{1})'=\mathbf{1}'$ and $\mathbf{1}'\mathbf{H}_0=\mathbf{1}'$; conversely a vector $\mathbf{u}\in\mathcal{C}(\mathbf{X})$ with $\mathbf{1}'\mathbf{u}=0$ has $(\mathbf{H}-\mathbf{H}_0)\mathbf{u}=\mathbf{u}-\mathbf{0}=\mathbf{u}$. So $\mathbf{H}-\mathbf{H}_0$ is the symmetric idempotent with column space $\mathcal{C}(\mathbf{X})\cap\mathbf{1}^\perp$. Finally $(\mathbf{H}-\mathbf{H}_0)\mathbf{M}=\mathbf{H}\mathbf{M}-\mathbf{H}_0\mathbf{H}\mathbf{M}=\mathbf{0}$ by Theorem 2.9(iii) and $\mathbf{H}_0=\mathbf{H}_0\mathbf{H}$. ∎

### 2.3 Residuals and leverage

**Theorem 2.11** (Residual identities; CLM Theorem 2.12). The least squares residuals satisfy

(i) $\mathbf{X}'\mathbf{e} = \mathbf{0}$;

(ii) $\sum_i e_i = 0$ and $\overline{\hat Y}=\bar Y$, provided the model contains an intercept;

(iii) $\hat{\mathbf{Y}}'\mathbf{e} = 0$.

**Proof.** **(i)** Rearranging (7), $\mathbf{X}'(\mathbf{Y}-\mathbf{X}\hat{\boldsymbol{\beta}})=\mathbf{0}$. **(ii)** The first column of $\mathbf{X}$ is $\mathbf{1}$, so the first component of $\mathbf{X}'\mathbf e=\mathbf{0}$ is $\mathbf{1}'\mathbf e=\sum_ie_i=0$; then $\mathbf{1}'\hat{\mathbf{Y}}=\mathbf{1}'(\mathbf{Y}-\mathbf e)=\mathbf{1}'\mathbf{Y}$, and dividing by $n$ gives $\overline{\hat Y}=\bar Y$. **(iii)** $\hat{\mathbf{Y}}'\mathbf e=\hat{\boldsymbol{\beta}}'\mathbf{X}'\mathbf e=0$ by (i). ∎

**Remark** (Errors versus residuals; FREES §2.3.2, §3.3). The **error** $\varepsilon_i=Y_i-\mathbf{x}_i'\boldsymbol{\beta}$ is unobservable; the **residual** $e_i=Y_i-\mathbf{x}_i'\hat{\boldsymbol{\beta}}$ is observable. Since $\mathbf{M}\mathbf{X}=\mathbf{0}$,

$$
\mathbf e=\mathbf{M}\mathbf{Y}=\mathbf{M}(\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon})=\mathbf{M}\boldsymbol{\varepsilon} , \tag{9}
$$

a linear map of rank $n-k-1$ of the errors: the residuals satisfy the $k+1$ constraints (i) whatever the data, and FREES's phrase is that only $n-(k+1)$ of them are free. The identities hold for every fit, so a residual mean of zero is not evidence that the model is right.

**Definition 2.12** (Leverage; ISLR §3.3.3; CLM Definition 2.13). The **leverage** of observation $i$ is $h_{ii}$, the $i$-th diagonal entry of $\mathbf{H}$, so that $\hat Y_i=\sum_jh_{ij}Y_j$ and $h_{ii}=\partial\hat Y_i/\partial Y_i$.

**Theorem 2.13** (Properties of leverage; CLM Theorem 2.14, Exercise D3). (i) $0\leqslant h_{ii}\leqslant1$, and $h_{ii}=1$ forces $h_{ij}=0$ for $j\neq i$ and $e_i=0$;

(ii) $\sum_{i=1}^n h_{ii}=k+1$, so the mean leverage is $(k+1)/n$;

(iii) with an intercept, $h_{ii}\geqslant1/n$, and for $k=1$

$$
h_{ii}=\frac1n+\frac{(x_i-\bar x)^2}{\sum_j(x_j-\bar x)^2} . \tag{10}
$$

**Proof.** **(i)** $\mathbf{H}=\mathbf{H}'\mathbf{H}$ by Theorem 2.9(i); comparing $(i,i)$ entries with $h_{ji}=h_{ij}$,

$$
h_{ii}=\sum_{j=1}^nh_{ij}^2=h_{ii}^2+\sum_{j\neq i}h_{ij}^2 .
$$

The left equality gives $h_{ii}\geqslant0$; dropping the sum gives $h_{ii}\geqslant h_{ii}^2$, hence $h_{ii}\leqslant1$. At $h_{ii}=1$ the sum $\sum_{j\neq i}h_{ij}^2$ must vanish, so $\hat Y_i=\sum_jh_{ij}Y_j=Y_i$ and $e_i=0$.

**(ii)** $\sum_ih_{ii}=\operatorname{tr}(\mathbf{H})=k+1$ by Theorem 2.9(iv).

**(iii)** With $\mathbf u_i$ the $i$-th coordinate vector and $\mathbf{H}_0=\tfrac1n\mathbf{1}\mathbf{1}'$, whose diagonal entries are $1/n$, Lemma 2.10 gives $h_{ii}-1/n=\mathbf u_i'(\mathbf{H}-\mathbf{H}_0)\mathbf u_i=\|(\mathbf{H}-\mathbf{H}_0)\mathbf u_i\|^2\geqslant0$. For $k=1$, $\mathbf{X}=[\mathbf{1}\ \mathbf{x}]$ and, with $S_{xx}=\sum_j(x_j-\bar x)^2$,

$$
\mathbf{X}'\mathbf{X}=\begin{pmatrix} n & n\bar x\\ n\bar x & \sum_jx_j^2\end{pmatrix},\qquad
(\mathbf{X}'\mathbf{X})^{-1}=\frac{1}{nS_{xx}}\begin{pmatrix}\sum_jx_j^2 & -n\bar x\\ -n\bar x & n\end{pmatrix},
$$

the determinant being $n\sum_jx_j^2-n^2\bar x^2=nS_{xx}$. Then with $\mathbf{x}_i'=(1,x_i)$,

$$
h_{ii}=\mathbf{x}_i'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_i
=\frac{\sum_jx_j^2-2n\bar xx_i+nx_i^2}{nS_{xx}}
=\frac{S_{xx}+n(x_i-\bar x)^2}{nS_{xx}}
=\frac1n+\frac{(x_i-\bar x)^2}{S_{xx}},
$$

using $\sum_jx_j^2=S_{xx}+n\bar x^2$. ∎

**Remark** (ISLR §3.3.3). $h_{ii}$ depends on $\mathbf{X}$ alone and, with one explanatory variable, measures how far $x_i$ lies from $\bar x$: by (10) it is a quadratic in $x_i-\bar x$ with minimum $1/n$. ISLR flags a leverage that greatly exceeds the mean $(k+1)/n$; FREES §5.4.1 flags three times the mean, and twice the mean, $2(k+1)/n$, is also in common use.

### 2.4 The case $k=1$

**Corollary 2.14** (The fitted line; FREES §2.1, ISLR §3.1.1; CLM Corollary 2.18). With one explanatory variable and an intercept, and $x_1,\ldots,x_n$ not all equal (and, for the last form of $b_1$, $y_1,\ldots,y_n$ not all equal),

$$
b_1=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}{\sum_i(x_i-\bar x)^2}
=\frac{\sum_ix_iy_i-n\bar x\bar y}{\sum_ix_i^2-n\bar x^2}
=r\,\frac{s_y}{s_x},
\qquad b_0=\bar y-b_1\bar x , \tag{11}
$$

so the fitted line $\hat y=b_0+b_1x$ passes through $(\bar x,\bar y)$ and $\hat y_i-\bar y=b_1(x_i-\bar x)$.

**Proof.** With $\mathbf{X}=[\mathbf{1}\ \mathbf{x}]$ the normal equations (7) are

$$
\begin{pmatrix} n & \sum_ix_i\\ \sum_ix_i & \sum_ix_i^2\end{pmatrix}
\begin{pmatrix} b_0\\ b_1\end{pmatrix}
=\begin{pmatrix}\sum_iy_i\\ \sum_ix_iy_i\end{pmatrix}.
$$

The first row is $nb_0+b_1\sum_ix_i=\sum_iy_i$, that is $b_0=\bar y-b_1\bar x$. Substituting into the second row, $\sum_ix_iy_i=(\bar y-b_1\bar x)n\bar x+b_1\sum_ix_i^2$, so $b_1(\sum_ix_i^2-n\bar x^2)=\sum_ix_iy_i-n\bar x\bar y$, the middle form of (11). The identities $\sum_i(x_i-\bar x)^2=\sum_ix_i^2-2\bar x\sum_ix_i+n\bar x^2=\sum_ix_i^2-n\bar x^2$ and $\sum_i(x_i-\bar x)(y_i-\bar y)=\sum_ix_iy_i-\bar y\sum_ix_i-\bar x\sum_iy_i+n\bar x\bar y =\sum_ix_iy_i-n\bar x\bar y$ give the first form, and dividing numerator and denominator by $n-1$ gives $b_1=\{(n-1)s_xs_yr\}/\{(n-1)s_x^2\}=r\,s_y/s_x$. Finally $\hat y_i-\bar y=b_0+b_1x_i-\bar y=b_1(x_i-\bar x)$. ∎

**Corollary 2.15** (Regression through the origin; FREES exercise 2.7, ISLR §3.7; CLM Corollary 2.20). For the model $y_i=\beta x_i+\varepsilon_i$ without an intercept, and some $x_i\neq0$, the least squares estimate is $\hat\beta=\sum_ix_iy_i/\sum_ix_i^2$ and $\hat y_i=x_i\hat\beta$.

**Proof.** Here $\mathbf{X}=\mathbf{x}$, so $\mathbf{X}'\mathbf{X}=\sum_ix_i^2$, a positive scalar when some $x_i\neq0$, and $\mathbf{X}'\mathbf{Y}=\sum_ix_iy_i$; Theorem 2.6 gives the ratio. ∎

**Example 2.6** (Lottery sales, continued; FREES §2.1). From Example 2.2 and the summary statistics, $b_1=r\,s_y/s_x=0.8863(8103)/11{,}098=0.6471$ and $b_0=\bar y-b_1\bar x=6495-0.6471(9311)=470$: the fitted line is $\widehat{\texttt{SALES}}=470+0.647\,\texttt{POP}$, and at a population of $10{,}000$ it gives $470+6471=6941$ dollars of sales.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
r <- cor(lottery$POP, lottery$SALES)
b1 <- r * sd(lottery$SALES) / sd(lottery$POP)
b0 <- mean(lottery$SALES) - b1 * mean(lottery$POP)
fit_lot <- lm(SALES ~ POP, data = lottery)
rbind(formula = c(b0 = b0, b1 = b1), lm = coef(fit_lot))
c(at_10000 = b0 + b1 * 10000, predict = unname(predict(fit_lot, data.frame(POP = 10000))))

**Example 2.7** (Advertising, continued; ISLR §3.1.1). Regressing `sales` on `TV` gives $\hat\beta_0=7.03$ and $\hat\beta_1=0.0475$: an additional \$1,000 of television advertising is associated with about $47.5$ additional units sold.

In [ ]:
fit_tv <- lm(sales ~ TV, data = adv)
coef(fit_tv)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
plot(adv$TV, adv$sales, xlab = "TV", ylab = "sales", pch = 16, cex = 0.5)
abline(fit_tv, lwd = 2)
segments(adv$TV, adv$sales, adv$TV, fitted(fit_tv), col = "grey60")
par(op)

*Figure 2.5.* The least squares line for sales on television advertising; each grey segment is a residual (ISLR Figure 3.1).

**Example 2.8** (Term life insurance, continued; FREES §3.1–3.2). The regression of `LNFACE` on `EDUCATION`, `NUMHH` and `LNINCOME` has $n=275$ and $k=3$; the first rows of $\mathbf{X}$ and $\mathbf{Y}$ and the fitted plane $\hat y=2.584+0.206x_1+0.306x_2+0.494x_3$:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
fit_term <- lm(LNFACE ~ EDUCATION + NUMHH + LNINCOME, data = term)
X_term <- model.matrix(fit_term)
round(cbind(X_term, LNFACE = term$LNFACE)[c(1, 2, 275), ], 3)
round(coef(fit_term), 3)

The three slopes are positive: with income and education fixed, a larger household demands more insurance; with household size and education fixed, a household with more income buys more; more education, other things equal, goes with more insurance.

**Remark** (Reading a coefficient on a logarithmic scale; FREES §3.2.2). With $y$ in logarithms, $\partial\ln y/\partial x=(\partial y/\partial x)/y$: a small change in $\ln y$ is a small proportional change in $y$, and a change of $\delta$ in $x_j$ changes $y$ by a factor $e^{\beta_j\delta}\approx1+\beta_j\delta$. With $x_j$ also in logarithms, $\beta_j=\partial\ln y/\partial\ln x_j=(\partial y/y)/(\partial x_j/x_j)$ is the ratio of the percentage changes, an **elasticity**. Natural logarithms are used because these identities need them.

**Example 2.9** (Term life insurance, continued; FREES §3.2.2). Holding `NUMHH` and `LNINCOME` at their sample means and moving `EDUCATION` in steps of $0.1$ year raises $\widehat{\texttt{LNFACE}}$ by a constant $0.0206$ per step, and the fitted face amount by a constant $2.09\%$ per step; the continuous version is $0.206\times0.1=2.06\%$. Moving `LNINCOME` in steps of $0.1$ raises income by $10.52\%$ per step and the fitted face amount by $5.06\%$, a ratio of $0.481$ against the coefficient $0.494$.

In [ ]:
b <- coef(fit_term); mN <- mean(term$NUMHH); mL <- mean(term$LNINCOME)
ed <- c(14, 14.1, 14.2, 14.3)
lnface_ed <- b[1] + b[2] * ed + b[3] * mN + b[4] * mL
round(rbind(EDUCATION = ed, LNFACE = lnface_ed, FACE = exp(lnface_ed),
            pct_change = c(NA, 100 * diff(exp(lnface_ed)) / exp(lnface_ed)[-4])), 3)
li <- c(11, 11.1, 11.2, 11.3)
lnface_li <- b[1] + b[2] * mean(term$EDUCATION) + b[3] * mN + b[4] * li
inc_pct <- 100 * diff(exp(li)) / exp(li)[-4]
face_pct <- 100 * diff(exp(lnface_li)) / exp(lnface_li)[-4]
round(rbind(LNINCOME = li, INCOME_pct = c(NA, inc_pct), FACE_pct = c(NA, face_pct),
            ratio = c(NA, face_pct / inc_pct)), 3)

So, holding household size and education fixed, a $1\%$ increase in income is associated with about a $0.48\%$ increase in the face amount.

**Example 2.10** (Simple against multiple regression; ISLR §3.2.1). Regressed alone on `newspaper`, sales rise by $0.055$ thousand units per thousand dollars; with `TV` and `radio` in the same model the coefficient is $-0.001$.

In [ ]:
fit_all <- lm(sales ~ TV + radio + newspaper, data = adv)
rbind(alone    = coef(lm(sales ~ newspaper, data = adv))["newspaper"],
      together = coef(fit_all)["newspaper"])
round(coef(fit_all), 4)

The simple coefficient measures the association ignoring the other media; the multiple coefficient measures it with `TV` and `radio` held fixed. The two differ because `newspaper` and `radio` spending are correlated ($0.35$ in Example 2.5): markets that spend on radio, which does raise sales, also spend on newspapers, so newspaper spending stands in for radio spending when radio is left out. ISLR's illustration is a regression of shark attacks on ice cream sales, which is positive until temperature is added.

## 3 Sampling Properties of Least Squares

> **Reading.** FREES §2.3.2, §2.4, §3.2.4, §3.3 · ISLR §3.1.2 · CLM §3. SRM learning outcome 2(a).

**Definition 2.16** (Sums of squares, mean squares and the residual standard deviation; FREES §2.3, §3.3, ISLR §3.1.3; CLM Definition 3.1). For the fitted model $\hat{\mathbf{Y}}=\mathbf{X}\hat{\boldsymbol{\beta}}$,

$$
\mathrm{SST}=\sum_i(Y_i-\bar Y)^2,\qquad
\mathrm{SSR}=\sum_i(\hat Y_i-\bar Y)^2,\qquad
\mathrm{SSE}=\sum_i(Y_i-\hat Y_i)^2=\mathbf e'\mathbf e ,
$$

$$
\mathrm{MSE}=s^2=\frac{\mathrm{SSE}}{n-k-1}\ \ (n>k+1),\qquad \mathrm{MSR}=\frac{\mathrm{SSR}}{k}\ \ (k\geqslant1).
$$

$s$ is the **residual standard deviation** (FREES) or **residual standard error**, RSE (ISLR). FREES writes **Total SS**, **Regression SS** and **Error SS**; ISLR writes $\mathrm{TSS}$ and $\mathrm{RSS}$ for $\mathrm{SST}$ and $\mathrm{SSE}$; the SOA questions use both.

**Proposition 2.17** (Coefficients are weighted sums of the responses; FREES §2.4, §3.2.4; CLM Proposition 2.19). $\hat{\boldsymbol{\beta}}=\mathbf{W}\mathbf{Y}$ with $\mathbf{W}=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'$ fixed by the design, so each $\hat\beta_j=\sum_iw_{ji}Y_i$. For $k=1$,

$$
b_1=\sum_{i=1}^nw_iY_i,\qquad w_i=\frac{x_i-\bar x}{(n-1)s_x^2},\qquad
\sum_iw_i=0,\quad \sum_iw_ix_i=1,\quad \sum_iw_i^2=\frac{1}{(n-1)s_x^2}. \tag{12}
$$

**Proof.** The first statement is (5). For $k=1$, Corollary 2.14 gives $b_1=\sum_i(x_i-\bar x)(Y_i-\bar Y)/\{(n-1)s_x^2\}=\sum_iw_i(Y_i-\bar Y) =\sum_iw_iY_i-\bar Y\sum_iw_i$, and $\sum_iw_i=\sum_i(x_i-\bar x)/\{(n-1)s_x^2\}=0$. Next $\sum_iw_ix_i=\sum_i(x_i-\bar x)x_i/\{(n-1)s_x^2\}$, and $\sum_i(x_i-\bar x)x_i=\sum_i(x_i-\bar x)^2+\bar x\sum_i(x_i-\bar x)=(n-1)s_x^2$, so the sum is $1$. Finally $\sum_iw_i^2=\sum_i(x_i-\bar x)^2/\{(n-1)s_x^2\}^2=1/\{(n-1)s_x^2\}$. ∎

### 3.1 Unbiasedness and variance

**Theorem 2.18** (Unbiasedness; CLM Theorem 3.3, equation (12)). Under A1–A2 and A4, $\operatorname{\mathsf{E}}[\hat{\boldsymbol{\beta}}]=\boldsymbol{\beta}$.

**Proof.** Substituting $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ into (5),

$$
\hat{\boldsymbol{\beta}}=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{X}\boldsymbol{\beta}+(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\boldsymbol{\varepsilon}=\boldsymbol{\beta}+(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\boldsymbol{\varepsilon} , \tag{13}
$$

and $\operatorname{\mathsf{E}}[\boldsymbol{\varepsilon}]=\mathbf{0}$ by A2, so $\operatorname{\mathsf{E}}[\hat{\boldsymbol{\beta}}]=\boldsymbol{\beta}+(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{0}=\boldsymbol{\beta}$. ∎

**Theorem 2.19** (Variance–covariance matrix; CLM Theorem 3.4). Under A1–A4, $\operatorname{var}(\hat{\boldsymbol{\beta}})=\sigma^2(\mathbf{X}'\mathbf{X})^{-1}$; in particular $\operatorname{var}(\hat\beta_j)=\sigma^2[(\mathbf{X}'\mathbf{X})^{-1}]_{jj}$ and $\operatorname{cov}(\hat\beta_i,\hat\beta_j)=\sigma^2[(\mathbf{X}'\mathbf{X})^{-1}]_{ij}$.

**Proof.** By A3, $\operatorname{var}(\mathbf{Y})=\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{I}_n$. By (13) and the prerequisites' proposition **variance under a linear map**, $\operatorname{var}(\mathbf{A}\mathbf{z})=\mathbf{A}\operatorname{var}(\mathbf{z})\mathbf{A}'$, with $\mathbf{A}=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'$,

$$
\operatorname{var}(\hat{\boldsymbol{\beta}})=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'(\sigma^2\mathbf{I}_n)\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}
=\sigma^2(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}=\sigma^2(\mathbf{X}'\mathbf{X})^{-1}.
$$

∎

**Corollary 2.20** (The case $k=1$; FREES §2.4, ISLR §3.1.2; CLM Corollary 3.13). Under A1–A4, with $x_1,\ldots,x_n$ not all equal and $S_{xx}=\sum_i(x_i-\bar x)^2=(n-1)s_x^2$,

$$
\operatorname{var}(b_1)=\frac{\sigma^2}{S_{xx}},\qquad
\operatorname{var}(b_0)=\sigma^2\Bigl(\frac1n+\frac{\bar x^2}{S_{xx}}\Bigr),\qquad
\operatorname{cov}(b_0,b_1)=-\frac{\sigma^2\bar x}{S_{xx}} . \tag{14}
$$

**Proof.** Reading $\sigma^2(\mathbf{X}'\mathbf{X})^{-1}$ off the explicit inverse in the proof of Theorem 2.13(iii): the $(2,2)$ entry is $\sigma^2n/(nS_{xx})$, the $(1,1)$ entry $\sigma^2\sum_ix_i^2/(nS_{xx})=\sigma^2(S_{xx}+n\bar x^2)/(nS_{xx})$, and the off-diagonal entry $-\sigma^2n\bar x/(nS_{xx})$. FREES's route is Proposition 2.17: the $Y_i$ are uncorrelated with common variance, so $\operatorname{var}(b_1)=\sum_iw_i^2\sigma^2=\sigma^2/\{(n-1)s_x^2\}$. ∎

**Remark** (The three determinants of precision; FREES §2.4, ISLR §3.1.2). $\operatorname{var}(b_1)=\sigma^2/\{(n-1)s_x^2\}$ falls with more observations, with less scatter about the line, and with more spread in the explanatory variable: doubling $s_x$ at fixed $n$ and $\sigma$ halves the standard deviation of the slope, which is what ISLR means by more **leverage** to estimate a slope. The spread of $x$ is a design choice. $\operatorname{var}(b_0)$ reduces to $\sigma^2/n$, the variance of a sample mean, when $\bar x=0$, in which case $b_0=\bar Y$.

### 3.2 The Gauss–Markov theorem

**Theorem 2.21** (Gauss–Markov; FREES §3.2.4; CLM Theorem 3.5). Under A1–A4, $\hat{\boldsymbol{\beta}}$ is the **best linear unbiased estimator** of $\boldsymbol{\beta}$: for any estimator $\tilde{\boldsymbol{\beta}}=\mathbf{C}\mathbf{Y}$ with $\mathbf{C}$ a constant matrix and $\operatorname{\mathsf{E}}[\tilde{\boldsymbol{\beta}}]=\boldsymbol{\beta}$ for every $\boldsymbol{\beta}$, the matrix $\operatorname{var}(\tilde{\boldsymbol{\beta}})-\operatorname{var}(\hat{\boldsymbol{\beta}})$ is positive semi-definite, so $\operatorname{var}(\tilde\beta_j)\geqslant\operatorname{var}(\hat\beta_j)$ for every $j$, and equality of the two variance matrices forces $\tilde{\boldsymbol{\beta}}=\hat{\boldsymbol{\beta}}$.

**Proof.** **Step 1: unbiasedness constrains $\mathbf{C}$.** $\operatorname{\mathsf{E}}[\tilde{\boldsymbol{\beta}}]=\mathbf{C}\operatorname{\mathsf{E}}[\mathbf{Y}]=\mathbf{C}\mathbf{X}\boldsymbol{\beta}$, and this equals $\boldsymbol{\beta}$ for every $\boldsymbol{\beta}\in\mathbb{R}^{k+1}$ iff $\mathbf{C}\mathbf{X}=\mathbf{I}_{k+1}$.

**Step 2: decompose $\mathbf{C}$.** Write $\mathbf{C}=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'+\mathbf{D}$. Then $\mathbf{D}\mathbf{X}=\mathbf{C}\mathbf{X}-(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{X}=\mathbf{I}_{k+1}-\mathbf{I}_{k+1}=\mathbf{0}$.

**Step 3: expand the variance.** By the prerequisites' **variance under a linear map** and A3, $\operatorname{var}(\tilde{\boldsymbol{\beta}})=\mathbf{C}(\sigma^2\mathbf{I})\mathbf{C}'=\sigma^2\mathbf{C}\mathbf{C}'$, and

$$
\begin{align*}
\mathbf{C}\mathbf{C}' &= \bigl[(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'+\mathbf{D}\bigr]\bigl[\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}+\mathbf{D}'\bigr]\\
&= (\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}+(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{D}'+\mathbf{D}\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}+\mathbf{D}\mathbf{D}' .
\end{align*}
$$

The first term is $(\mathbf{X}'\mathbf{X})^{-1}$. The third vanishes because $\mathbf{D}\mathbf{X}=\mathbf{0}$, and the second is its transpose, $(\mathbf{D}\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1})'=\mathbf{0}$. Hence

$$
\operatorname{var}(\tilde{\boldsymbol{\beta}})=\sigma^2(\mathbf{X}'\mathbf{X})^{-1}+\sigma^2\mathbf{D}\mathbf{D}'=\operatorname{var}(\hat{\boldsymbol{\beta}})+\sigma^2\mathbf{D}\mathbf{D}' .
$$

**Step 4: the difference is positive semi-definite.** For any $\mathbf{v}$, $\mathbf{v}'\mathbf{D}\mathbf{D}'\mathbf{v}=\|\mathbf{D}'\mathbf{v}\|^2\geqslant0$. Taking $\mathbf{v}$ the $j$-th coordinate vector gives $\operatorname{var}(\tilde\beta_j)\geqslant\operatorname{var}(\hat\beta_j)$. If $\mathbf{D}\mathbf{D}'=\mathbf{0}$ then every row of $\mathbf{D}$ has zero length, so $\mathbf{D}=\mathbf{0}$ and $\tilde{\boldsymbol{\beta}}=\hat{\boldsymbol{\beta}}$. ∎

**Remark**. The theorem ranks least squares within the estimators that are linear in $\mathbf{Y}$ and unbiased; it uses no normality and says nothing about prediction error on new data. Biased estimators, such as those that shrink $\hat{\boldsymbol{\beta}}$ toward zero, can have smaller mean squared error.

### 3.3 Estimating the error variance

**Theorem 2.22** (Unbiasedness of the mean square error; FREES §2.3.2, §3.3; CLM Theorem 3.7). Under A1–A4, $\operatorname{\mathsf{E}}[\mathrm{SSE}]=(n-k-1)\sigma^2$, so $\operatorname{\mathsf{E}}[s^2]=\sigma^2$.

**Proof.** By (9) and the symmetry and idempotency of $\mathbf{M}$, $\mathrm{SSE}=\mathbf e'\mathbf e=\boldsymbol{\varepsilon}'\mathbf{M}'\mathbf{M}\boldsymbol{\varepsilon}=\boldsymbol{\varepsilon}'\mathbf{M}\boldsymbol{\varepsilon}$. The prerequisites' lemma **expectation of a quadratic form**, $\operatorname{\mathsf{E}}[\mathbf{z}'\mathbf{A}\mathbf{z}]=\operatorname{tr}(\mathbf{A}\boldsymbol{\Sigma})+\boldsymbol{\mu}'\mathbf{A}\boldsymbol{\mu}$ for $\operatorname{\mathsf{E}}[\mathbf{z}]=\boldsymbol{\mu}$ and $\operatorname{var}(\mathbf{z})=\boldsymbol{\Sigma}$, applied with $\mathbf{z}=\boldsymbol{\varepsilon}$, $\boldsymbol{\mu}=\mathbf{0}$ and $\boldsymbol{\Sigma}=\sigma^2\mathbf{I}_n$, gives $\operatorname{\mathsf{E}}[\mathrm{SSE}]=\operatorname{tr}(\mathbf{M}\sigma^2\mathbf{I}_n)=\sigma^2\operatorname{tr}(\mathbf{M})=\sigma^2(n-k-1)$ by Theorem 2.9(iv). Dividing by $n-k-1$ gives $\operatorname{\mathsf{E}}[s^2]=\sigma^2$. ∎

**Remark** (The divisor; FREES §2.3.2, §3.3). The divisor is $n-k-1=\operatorname{tr}(\mathbf{M})$ because the residual vector lies in the $(n-k-1)$-dimensional space $\mathcal{C}(\mathbf{X})^\perp$: $k+1$ observations determine the plane exactly and leave no residual, and a further observation is needed before there is any variation about it. Dividing $\mathrm{SSE}$ by $n$ gives an estimator biased downward by the factor $(n-k-1)/n$.

**Theorem 2.23** (Variance of the residuals; CLM Theorem 3.8). Under A1–A4, $\operatorname{var}(\mathbf e)=\sigma^2\mathbf{M}$: $\operatorname{var}(e_i)=\sigma^2(1-h_{ii})$ and $\operatorname{cov}(e_i,e_j)=-\sigma^2h_{ij}$.

**Proof.** By (9) and **variance under a linear map**, $\operatorname{var}(\mathbf e)=\mathbf{M}(\sigma^2\mathbf{I}_n)\mathbf{M}'=\sigma^2\mathbf{M}^2=\sigma^2\mathbf{M}$; the $(i,i)$ entry of $\mathbf{I}-\mathbf{H}$ is $1-h_{ii}$ and the $(i,j)$ entry is $-h_{ij}$. ∎

**Definition 2.24** (Standard errors; FREES §2.4, §3.4, ISLR §3.1.2). The **standard error** of $\hat\beta_j$ is its estimated standard deviation,

$$
\mathrm{SE}(\hat\beta_j)=s\sqrt{[(\mathbf{X}'\mathbf{X})^{-1}]_{jj}},\qquad
\mathrm{SE}(b_1)=\frac{s}{s_x\sqrt{n-1}},\qquad
\mathrm{SE}(b_0)=s\sqrt{\frac1n+\frac{\bar x^2}{(n-1)s_x^2}} , \tag{15}
$$

the last two for $k=1$, by Theorem 2.19 and Corollary 2.20 with $\sigma$ replaced by $s$. ISLR writes $\mathrm{SE}(\hat\beta_j)$ for the same quantity and notes that, strictly, it is an estimate.

**Example 2.11** (Lottery sales, continued; FREES §2.4). With $s=3792$, $s_x=11{,}098$ and $n=50$, $\mathrm{SE}(b_1)=3792/(11{,}098\sqrt{49})=0.0488$.

In [ ]:
s_lot <- summary(fit_lot)$sigma
c(s = s_lot, se_b1 = s_lot / (sd(lottery$POP) * sqrt(49)),
  se_b1_lm = coef(summary(fit_lot))["POP", "Std. Error"])

**Example 2.12** (Advertising, continued; ISLR §3.1.2). The standard errors of ISLR Table 3.1, from the formulas (15) and from `lm`:

In [ ]:
s_tv <- summary(fit_tv)$sigma; xt <- adv$TV; n_adv <- nrow(adv)
rbind(formula = c(intercept = s_tv * sqrt(1/n_adv + mean(xt)^2 / sum((xt - mean(xt))^2)),
                  TV = s_tv / sqrt(sum((xt - mean(xt))^2))),
      lm = coef(summary(fit_tv))[, "Std. Error"])

**Example 2.13** (The population line and the least squares line; ISLR §3.1.2). ISLR Figure 3.3 draws $100$ observations from $Y=2+3X+\varepsilon$ and fits a line; repeated on ten independent samples, the fitted lines scatter about the population line, and averaged over many samples they reproduce it, which is Theorem 2.18.

In [ ]:
set.seed(2)
x_pop <- runif(100, -2, 2)
fits <- replicate(2000, coef(lm(I(2 + 3 * x_pop + rnorm(100, sd = 2)) ~ x_pop)))
Sxx_pop <- sum((x_pop - mean(x_pop))^2)
rbind(true = c(2, 3), mean_of_fits = rowMeans(fits), sd_of_fits = apply(fits, 1, sd),
      sd_by_formula = 2 * sqrt(c(1 / 100 + mean(x_pop)^2 / Sxx_pop, 1 / Sxx_pop)))

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
set.seed(3); y1 <- 2 + 3 * x_pop + rnorm(100, sd = 2)
plot(x_pop, y1, pch = 16, cex = 0.5, xlab = "X", ylab = "Y")
abline(2, 3, col = "#B45309", lwd = 2); abline(lm(y1 ~ x_pop), col = "steelblue", lwd = 2)
plot(x_pop, y1, type = "n", xlab = "X", ylab = "Y")
for (j in 1:10) abline(fits[, j], col = "lightblue")
abline(2, 3, col = "#B45309", lwd = 2); abline(lm(y1 ~ x_pop), col = "steelblue", lwd = 2)
par(op)

*Figure 2.6.* Left: the population line $2+3X$ (orange) and the least squares line (blue) for one sample. Right: ten least squares lines from ten samples (ISLR Figure 3.3).

The mean of $2000$ fitted intercepts and slopes agrees with $(2,3)$ within $0.01$, and the spread of the fits is close to the standard deviation (14) (row `sd_by_formula`).

## 4 Goodness of Fit

> **Reading.** FREES §2.3, §3.3 · ISLR §3.1.3, §3.2.2 · CLM §6. SRM learning outcome 2(e).

### 4.1 The analysis of variance

**Definition 2.25** (Centering matrix; CLM Definition 6.1). $\mathbf{M}_0=\mathbf{I}_n-\tfrac1n\mathbf{1}\mathbf{1}'=\mathbf{I}_n-\mathbf{H}_0$, so that $\mathbf{M}_0\mathbf{v}=\mathbf{v}-\bar v\mathbf{1}$. $\mathbf{M}_0$ is the residual maker of the intercept-only model: symmetric idempotent, with $\operatorname{tr}(\mathbf{M}_0)=n-1$, projecting onto $\mathbf{1}^\perp$.

**Theorem 2.26** (Variance decomposition; FREES §2.3.1, §3.3, ISLR §3.1.3; CLM Theorem 6.4, equation (24)). In a model with an intercept,

$$
\mathrm{SST}=\mathrm{SSR}+\mathrm{SSE} , \tag{16}
$$

with $\mathrm{SST}=\mathbf{Y}'\mathbf{M}_0\mathbf{Y}$, $\mathrm{SSR}=\hat{\mathbf{Y}}'\mathbf{M}_0\hat{\mathbf{Y}}=\|(\mathbf{H}-\mathbf{H}_0)\mathbf{Y}\|^2$ and $\mathrm{SSE}=\mathbf e'\mathbf e=\mathbf{Y}'\mathbf{M}\mathbf{Y}$.

**Proof.** **Step 1: the matrix forms.** For any $\mathbf{v}$, since $\mathbf{M}_0'\mathbf{M}_0=\mathbf{M}_0$, $\mathbf{v}'\mathbf{M}_0\mathbf{v}=\|\mathbf{M}_0\mathbf{v}\|^2=\|\mathbf{v}-\bar v\mathbf{1}\|^2=\sum_i(v_i-\bar v)^2$. With $\mathbf{v}=\mathbf{Y}$ this is $\mathrm{SST}$; with $\mathbf{v}=\hat{\mathbf{Y}}$ it is $\mathrm{SSR}$, because $\overline{\hat Y}=\bar Y$ by Theorem 2.11(ii). Also $\mathbf{M}_0\hat{\mathbf{Y}}=\mathbf{M}_0\mathbf{H}\mathbf{Y}=(\mathbf{H}-\mathbf{H}_0)\mathbf{Y}$, since $\mathbf{H}_0\mathbf{H}=\mathbf{H}_0$ by Lemma 2.10; and $\mathbf{Y}'\mathbf{M}\mathbf{Y}=\|\mathbf{M}\mathbf{Y}\|^2=\|\mathbf e\|^2$.

**Step 2: decompose.** Since $\bar e=0$, $\mathbf{M}_0\mathbf e=\mathbf e$ and

$$
\mathbf{M}_0\mathbf{Y}=\mathbf{M}_0(\hat{\mathbf{Y}}+\mathbf e)=\mathbf{M}_0\hat{\mathbf{Y}}+\mathbf e . \tag{17}
$$

**Step 3: expand.** $\mathrm{SST}=\|\mathbf{M}_0\hat{\mathbf{Y}}+\mathbf e\|^2 =\mathrm{SSR}+2\,\mathbf e'\mathbf{M}_0\hat{\mathbf{Y}}+\mathrm{SSE}$.

**Step 4: the cross term vanishes.** $\mathbf e'\mathbf{M}_0\hat{\mathbf{Y}}=\mathbf e'\hat{\mathbf{Y}}-\bar Y\,\mathbf e'\mathbf{1}=0-0$ by Theorem 2.11(iii) and (ii). ∎

**Remark** (FREES §2.3.1). Equation (16) is FREES's “total deviation equals unexplained deviation plus explained deviation”, $y_i-\bar y=(y_i-\hat y_i)+(\hat y_i-\bar y)$, squared and summed; the cross products sum to zero for the least squares fit because of the normal equations (Step 4); for an arbitrary line they need not vanish. Without an intercept $\bar e$ is in general nonzero (unless $\mathbf{1}\in\mathcal{C}(\mathbf{X})$), the cross term $-2n\bar Y\bar e$ survives, and (16) in general fails.

| Source | Sum of squares | $\operatorname{df}$ | Mean square |
|:--|:-:|:-:|:-:|
| Regression | $\mathrm{SSR}$ | $k$ | $\mathrm{MSR}=\mathrm{SSR}/k$ |
| Error | $\mathrm{SSE}$ | $n-k-1$ | $\mathrm{MSE}=\mathrm{SSE}/(n-k-1)=s^2$ |
| Total | $\mathrm{SST}$ | $n-1$ |  |

**Remark** (The ANOVA table; FREES §2.3.2, §3.3). The degrees of freedom are the dimensions of the three subspaces of Lemma 2.10: $\mathbf{M}_0\mathbf{Y}$ lies in $\mathbf{1}^\perp$ ($n-1$), and splits into $(\mathbf{H}-\mathbf{H}_0)\mathbf{Y}\in\mathcal{C}(\mathbf{X})\cap\mathbf{1}^\perp$ ($k$) and $\mathbf e\in\mathcal{C}(\mathbf{X})^\perp$ ($n-k-1$). For $k=1$ the regression sum of squares equals its mean square.

**Example 2.14** (Lottery sales, continued; FREES §2.3).

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
anova(fit_lot)
c(R2 = summary(fit_lot)$r.squared, s = s_lot, s_y = sd(lottery$SALES))

The regression explains $78.5\%$ of the variation and the typical deviation from the line is $s=3792$ against $s_y=8103$ without the explanatory variable.

### 4.2 The coefficient of determination

**Definition 2.27** ($R^2$ and adjusted $R^2$; FREES §2.3.1, §3.3, ISLR §3.1.3; CLM Definition 6.6).

$$
R^2=\frac{\mathrm{SSR}}{\mathrm{SST}}=1-\frac{\mathrm{SSE}}{\mathrm{SST}},
\qquad
R_a^2=1-\frac{\mathrm{SSE}/(n-k-1)}{\mathrm{SST}/(n-1)}=1-\frac{s^2}{s_y^2}=1-\frac{n-1}{n-k-1}(1-R^2), \tag{18}
$$

the proportion of the variation in $Y$ explained by the regression and its version adjusted for degrees of freedom; $R_a^2$ can be negative. $R=\sqrt{R^2}$ is the **multiple correlation coefficient**.

**Proposition 2.28** (Bounds and monotonicity; FREES §2.3, §3.3, ISLR §3.2.2; CLM Proposition 6.7, Theorem 6.8). In a model with an intercept and $\mathrm{SST}>0$, $0\leqslant R^2\leqslant1$; $R^2=1$ iff every residual is zero and $R^2=0$ iff $\hat{\mathbf{Y}}=\bar Y\mathbf{1}$. Adding an explanatory variable cannot decrease $R^2$, and can decrease $R_a^2$.

**Proof.** $\mathrm{SSR}\geqslant0$ and $\mathrm{SST}>0$ give $R^2\geqslant0$, with equality iff $\hat Y_i=\bar Y$ for every $i$; $\mathrm{SST}=\mathrm{SSR}+\mathrm{SSE}$ with $\mathrm{SSE}\geqslant0$ gives $R^2\leqslant1$, with equality iff $\mathrm{SSE}=0$. For monotonicity let $\mathcal{C}_k\subseteq\mathcal{C}_{k+1}$ be the column spaces of the smaller and the enlarged design. By Theorem 2.7, $\hat{\mathbf{Y}}_{k+1}$ is the point of $\mathcal{C}_{k+1}$ closest to $\mathbf{Y}$, and $\hat{\mathbf{Y}}_k\in\mathcal{C}_k\subseteq\mathcal{C}_{k+1}$, so $\mathrm{SSE}_{k+1}=\|\mathbf{Y}-\hat{\mathbf{Y}}_{k+1}\|^2\leqslant\|\mathbf{Y}-\hat{\mathbf{Y}}_k\|^2=\mathrm{SSE}_k$, while $\mathrm{SST}$ does not involve the design; hence $R^2_{k+1}\geqslant R^2_k$. For $R_a^2$, $\mathrm{SST}$ and $n-1$ are fixed and $s^2=\mathrm{SSE}/(n-k-1)$ rises whenever $\mathrm{SSE}$ falls by less than the factor $(n-k-2)/(n-k-1)$. ∎

**Proposition 2.29** ($R^2$ as a squared correlation; FREES §3.3, ISLR §3.2.2; CLM Proposition 6.12). In a model with an intercept, $\mathrm{SST}>0$ and $\mathrm{SSR}>0$, $R^2=\operatorname{cor}(Y,\hat Y)^2$, the squared sample correlation between the responses and the fitted values; and among all vectors $\mathbf{v}\in\mathcal{C}(\mathbf{X})$ that are not constant, $\operatorname{cor}(Y,v)^2$ is largest at $\mathbf{v}=\hat{\mathbf{Y}}$.

**Proof.** **Step 1: the correlation with $\hat{\mathbf{Y}}$.** By Theorem 2.11(ii), $\overline{\hat Y}=\bar Y$, so the numerator of $\operatorname{cor}(Y,\hat Y)$ is

$$
\sum_i(Y_i-\bar Y)(\hat Y_i-\bar Y)=\hat{\mathbf{Y}}'\mathbf{M}_0\mathbf{Y}
=\hat{\mathbf{Y}}'\mathbf{M}_0(\hat{\mathbf{Y}}+\mathbf e)=\mathrm{SSR}+\hat{\mathbf{Y}}'\mathbf e=\mathrm{SSR} ,
$$

using $\mathbf{M}_0\mathbf e=\mathbf e$ and Theorem 2.11(iii). The two sums of squares in the denominator are $\mathrm{SST}$ and $\mathrm{SSR}$, so $\operatorname{cor}(Y,\hat Y)^2=\mathrm{SSR}^2/(\mathrm{SST}\cdot\mathrm{SSR})=R^2$.

**Step 2: the maximum.** Let $\mathbf{v}\in\mathcal{C}(\mathbf{X})$ and put $\tilde{\mathbf{Y}}=\mathbf{M}_0\mathbf{Y}$, $\tilde{\mathbf{v}}=\mathbf{M}_0\mathbf{v}$. Since $\mathbf{v}\in\mathcal{C}(\mathbf{X})$ and $\mathbf{H}_0\mathbf{v}\in\mathcal{C}(\mathbf{1})\subseteq\mathcal{C}(\mathbf{X})$, $\tilde{\mathbf{v}}=\mathbf{v}-\mathbf{H}_0\mathbf{v}$ lies in $\mathcal{C}(\mathbf{X})\cap\mathbf{1}^\perp$, the column space of $\mathbf{P}=\mathbf{H}-\mathbf{H}_0$ (Lemma 2.10), so $\mathbf{P}\tilde{\mathbf{v}}=\tilde{\mathbf{v}}$ and $\tilde{\mathbf{Y}}'\tilde{\mathbf{v}}=\tilde{\mathbf{Y}}'\mathbf{P}\tilde{\mathbf{v}}=(\mathbf{P}\tilde{\mathbf{Y}})'\tilde{\mathbf{v}}$. By the Cauchy–Schwarz inequality of the prerequisites,

$$
\operatorname{cor}(Y,v)^2=\frac{(\tilde{\mathbf{Y}}'\tilde{\mathbf{v}})^2}{\|\tilde{\mathbf{Y}}\|^2\|\tilde{\mathbf{v}}\|^2}
\leqslant\frac{\|\mathbf{P}\tilde{\mathbf{Y}}\|^2\|\tilde{\mathbf{v}}\|^2}{\|\tilde{\mathbf{Y}}\|^2\|\tilde{\mathbf{v}}\|^2}
=\frac{\|\mathbf{P}\mathbf{M}_0\mathbf{Y}\|^2}{\mathrm{SST}}=\frac{\|(\mathbf{H}-\mathbf{H}_0)\mathbf{Y}\|^2}{\mathrm{SST}}=R^2 ,
$$

using $\mathbf{P}\mathbf{M}_0=\mathbf{P}-\mathbf{P}\mathbf{H}_0=\mathbf{P}$ (as $\mathbf{H}\mathbf{H}_0=\mathbf{H}_0$ and $\mathbf{H}_0^2=\mathbf{H}_0$) and Theorem 2.26. Equality holds when $\tilde{\mathbf{v}}$ is proportional to $\mathbf{P}\tilde{\mathbf{Y}}=\mathbf{M}_0\hat{\mathbf{Y}}$, which is the case $\mathbf{v}=\hat{\mathbf{Y}}$ by Step 1. ∎

**Lemma 2.30** (Adding one explanatory variable; CLM Lemma 6.9, equation (28)). Let $\mathbf{X}$ satisfy A4, let $\mathbf{X}_+=[\mathbf{X}\ \ \mathbf{z}]$ with $\mathbf{z}\notin\mathcal{C}(\mathbf{X})$, so that $\mathbf{z}'\mathbf{M}\mathbf{z}>0$, and let $\hat\gamma$ be the coefficient of $\mathbf{z}$ in the enlarged fit, with residual sum of squares $\mathrm{SSE}_+$. Then

$$
\hat\gamma=\frac{\mathbf{z}'\mathbf{M}\mathbf{Y}}{\mathbf{z}'\mathbf{M}\mathbf{z}},\qquad
\mathrm{SSE}-\mathrm{SSE}_+=\hat\gamma^2\,\mathbf{z}'\mathbf{M}\mathbf{z}=\frac{(\mathbf{z}'\mathbf{M}\mathbf{Y})^2}{\mathbf{z}'\mathbf{M}\mathbf{z}},\qquad
\operatorname{var}(\hat\gamma)=\frac{\sigma^2}{\mathbf{z}'\mathbf{M}\mathbf{z}}, \tag{19}
$$

the last under A1–A4; and for $n>k+2$ the standard error of $\hat\gamma$ in the enlarged model is $\mathrm{SE}(\hat\gamma)=s_+/\sqrt{\mathbf{z}'\mathbf{M}\mathbf{z}}$ with $s_+^2=\mathrm{SSE}_+/(n-k-2)$.

**Proof.** **Step 1: the partitioned normal equations.** Write the enlarged coefficient vector as $(\hat{\boldsymbol{\beta}}_+',\hat\gamma)'$. The normal equations $\mathbf{X}_+'\mathbf{X}_+(\hat{\boldsymbol{\beta}}_+',\hat\gamma)'=\mathbf{X}_+'\mathbf{Y}$ split into

$$
\text{(a)}\ \ \mathbf{X}'\mathbf{X}\hat{\boldsymbol{\beta}}_++\mathbf{X}'\mathbf{z}\hat\gamma=\mathbf{X}'\mathbf{Y},\qquad
\text{(b)}\ \ \mathbf{z}'\mathbf{X}\hat{\boldsymbol{\beta}}_++\mathbf{z}'\mathbf{z}\hat\gamma=\mathbf{z}'\mathbf{Y} .
$$

From (a), $\hat{\boldsymbol{\beta}}_+=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'(\mathbf{Y}-\mathbf{z}\hat\gamma)=\hat{\boldsymbol{\beta}}-(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{z}\hat\gamma$. Substituting into (b), with $\mathbf{H}=\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'$,

$$
\mathbf{z}'\mathbf{H}\mathbf{Y}-\mathbf{z}'\mathbf{H}\mathbf{z}\hat\gamma+\mathbf{z}'\mathbf{z}\hat\gamma=\mathbf{z}'\mathbf{Y}
\;\Longleftrightarrow\; \mathbf{z}'(\mathbf{I}-\mathbf{H})\mathbf{z}\,\hat\gamma=\mathbf{z}'(\mathbf{I}-\mathbf{H})\mathbf{Y} ,
$$

which is the first formula.

**Step 2: the residuals.** Multiplying the expression for $\hat{\boldsymbol{\beta}}_+$ by $\mathbf{X}$, $\mathbf{X}\hat{\boldsymbol{\beta}}_+=\hat{\mathbf{Y}}-\mathbf{H}\mathbf{z}\hat\gamma$, so $\hat{\mathbf{Y}}_+=\mathbf{X}\hat{\boldsymbol{\beta}}_++\mathbf{z}\hat\gamma=\hat{\mathbf{Y}}+\mathbf{M}\mathbf{z}\hat\gamma$ and $\mathbf e_+=\mathbf e-\hat\gamma\mathbf{M}\mathbf{z}$. Taking squared lengths, with $\mathbf e=\mathbf{M}\mathbf{Y}$ and $\mathbf{M}^2=\mathbf{M}$,

$$
\mathrm{SSE}_+=\mathrm{SSE}-2\hat\gamma\,\mathbf{Y}'\mathbf{M}\mathbf{z}+\hat\gamma^2\mathbf{z}'\mathbf{M}\mathbf{z}
=\mathrm{SSE}-2\hat\gamma^2\mathbf{z}'\mathbf{M}\mathbf{z}+\hat\gamma^2\mathbf{z}'\mathbf{M}\mathbf{z}=\mathrm{SSE}-\hat\gamma^2\mathbf{z}'\mathbf{M}\mathbf{z} ,
$$

substituting $\mathbf{Y}'\mathbf{M}\mathbf{z}=\mathbf{z}'\mathbf{M}\mathbf{Y}=\hat\gamma\,\mathbf{z}'\mathbf{M}\mathbf{z}$.

**Step 3: the variance.** $\hat\gamma=\mathbf{z}'\mathbf{M}\mathbf{Y}/(\mathbf{z}'\mathbf{M}\mathbf{z})$ is a linear map of $\mathbf{Y}$, so by **variance under a linear map** and A3, $\operatorname{var}(\hat\gamma)=\sigma^2\mathbf{z}'\mathbf{M}\mathbf{M}'\mathbf{z}/(\mathbf{z}'\mathbf{M}\mathbf{z})^2=\sigma^2/(\mathbf{z}'\mathbf{M}\mathbf{z})$; the standard error replaces $\sigma^2$ by the enlarged model's $s_+^2$. ∎

**Theorem 2.31** (Adjusted $R^2$ rises iff $|t|>1$; CLM Theorem 6.10). Let $n>k+2$ and $\mathrm{SST}>0$, and add one explanatory variable $\mathbf{z}\notin\mathcal{C}(\mathbf{X})$ with enlarged residual sum of squares $\mathrm{SSE}_+>0$. The addition raises $R_a^2$ iff the $t$ statistic $t=\hat\gamma/\mathrm{SE}(\hat\gamma)$ of the new variable in the enlarged model satisfies $|t|>1$, and lowers it iff $|t|<1$.

**Proof.** By (18), $R_a^2=1-s^2/s_y^2$ with $s_y^2$ fixed, so $R_a^2$ rises iff $s_+^2<s^2$, that is

$$
\begin{align*}
\frac{\mathrm{SSE}_+}{n-k-2}<\frac{\mathrm{SSE}}{n-k-1}
&\;\Longleftrightarrow\; (n-k-1)\mathrm{SSE}_+<(n-k-2)\mathrm{SSE}\\
&\;\Longleftrightarrow\; \mathrm{SSE}_+<(n-k-2)(\mathrm{SSE}-\mathrm{SSE}_+) ,
\end{align*}
$$

subtracting $(n-k-2)\mathrm{SSE}_+$ from both sides. Dividing by $\mathrm{SSE}_+=(n-k-2)s_+^2>0$ gives $1<(\mathrm{SSE}-\mathrm{SSE}_+)/s_+^2$, and by Lemma 2.30

$$
\frac{\mathrm{SSE}-\mathrm{SSE}_+}{s_+^2}=\frac{\hat\gamma^2\,\mathbf{z}'\mathbf{M}\mathbf{z}}{s_+^2}
=\frac{\hat\gamma^2}{\mathrm{SE}(\hat\gamma)^2}=t^2 .
$$

The same equivalences hold with $<$ replaced by $>$ or $=$, which gives the second statement. ∎

**Corollary 2.32** (The case $k=1$; FREES §2.5.1, ISLR §3.1.3; CLM Corollary 6.13). With one explanatory variable whose values are not all equal, an intercept and $\mathrm{SST}>0$, $R^2=r^2$, $\mathrm{SSR}=b_1^2(n-1)s_x^2$, and, for $n>2$,

$$
s_y^2=b_1^2s_x^2+s^2\,\frac{n-2}{n-1} . \tag{20}
$$

**Proof.** By Corollary 2.14, $\hat y_i-\bar y=b_1(x_i-\bar x)$, so $\mathrm{SSR}=b_1^2\sum_i(x_i-\bar x)^2=b_1^2(n-1)s_x^2$; substituting $b_1=r\,s_y/s_x$ gives $\mathrm{SSR}=r^2(n-1)s_y^2=r^2\mathrm{SST}$, that is $R^2=r^2$. Dividing (16) by $n-1$ and using $\mathrm{SSE}=(n-2)s^2$ gives (20). ∎

**Remark** (ISLR §3.1.3, §3.2.2). For $k=1$ the sign of the association is carried by $r$ and its strength by $R^2=r^2$, so ISLR's $R^2=0.61$ for sales on television advertising is $r=0.78$ from Example 2.5. For $k>1$, $R^2$ is the squared correlation of $Y$ with the fitted combination of all the explanatory variables (Proposition 2.29), not with any one of them. What counts as a good $R^2$ depends on the field: near $1$ in a physical law with small measurement error, well below $0.1$ in a social or marketing setting where the linear model is a rough approximation.

**Example 2.15** (Term life insurance, continued; FREES §3.3).

In [ ]:
a <- anova(fit_term)
SSR_t <- sum(a[["Sum Sq"]][1:3]); SSE_t <- a[["Sum Sq"]][4]; SST_t <- SSR_t + SSE_t
rbind(Regression = c(SS = SSR_t, df = 3, MS = SSR_t / 3),
      Error      = c(SS = SSE_t, df = 271, MS = SSE_t / 271),
      Total      = c(SS = SST_t, df = 274, MS = NA))
c(R2 = SSR_t / SST_t, s = sqrt(SSE_t / 271), s_y = sqrt(SST_t / 274),
  R2_adj = 1 - (SSE_t / 271) / (SST_t / 274))

`anova` prints one row per variable, sequential sums of squares whose total is $\mathrm{SSR}$. The three explanatory variables explain $34.3\%$ of the variation in the logarithmic face amount; the typical error falls from $s_y=1.871$ to $s=1.525$, and $R_a^2=1-s^2/s_y^2=0.335$.

**Remark** (Why do females live longer than males?; FREES §3.3, Table 3.4). Lemaire's regression of the female advantage in life expectancy, $4.51$ years on average over $n=169$ countries, chose $k=4$ of $45$ candidate measures and reached $R^2=61\%$: the logarithm of persons per physician ($-0.473$), fertility ($-0.444$), the percentage of Hindus and Buddhists ($-0.018$) and an indicator of the former Soviet Union ($4.922$), each with $|t|>3$. The four stand for wider groups of correlated variables, economic modernisation, education, religion and geography; with so much collinearity among the candidates another analyst could pick another four. The data are not published, so the table is quoted, not reproduced; its point is that a parsimonious model can explain most of the variation.

**Example 2.16** (Advertising, continued; ISLR §3.1.3, §3.2.2).

In [ ]:
fit_two <- lm(sales ~ TV + radio, data = adv)
fit_stats <- function(m) c(RSE = summary(m)$sigma, R2 = summary(m)$r.squared,
                           adj_R2 = summary(m)$adj.r.squared)
round(rbind(TV_only = fit_stats(fit_tv), TV_radio = fit_stats(fit_two),
            all_three = fit_stats(fit_all)), 5)
sprintf("R2: TV+radio %.5f, all three %.5f", summary(fit_two)$r.squared, summary(fit_all)$r.squared)
c(mean_sales = mean(adv$sales))

Television alone explains $61\%$ of the variation with a residual standard error of $3.26$, so a market's sales deviate from the line by about $3{,}260$ units, $23\%$ of the mean sales of about $14{,}000$. Adding radio raises $R^2$ to $0.897$ and lowers the residual standard error to $1.68$. Adding newspaper raises $R^2$ from $0.89719$ to $0.89721$, the increase Proposition 2.28 guarantees, while the residual standard error **rises** from $1.681$ to $1.686$: the decrease in $\mathrm{SSE}$ is smaller than the loss of a degree of freedom, and by Theorem 2.31 the newspaper $t$ statistic must be below $1$ in absolute value, which Example 2.10's coefficient of $-0.001$ against its standard error confirms.

## 5 Exact Inference Under Normality

> **Reading.** FREES §2.5, §2.7, §3.2.4 · ISLR §3.1.2, §3.2.2 · CLM §4–5, §7, §9. SRM learning outcomes 2(e), 2(f). A5 is in force throughout.

### 5.1 Sampling distributions

**Theorem 2.33** (Normality of $\hat{\boldsymbol{\beta}}$; FREES §3.2.4; CLM Theorem 5.1). Under A1–A5, $\hat{\boldsymbol{\beta}}\sim N\bigl(\boldsymbol{\beta},\,\sigma^2(\mathbf{X}'\mathbf{X})^{-1}\bigr)$, and for any constant $\mathbf{c}$, $\mathbf{c}'\hat{\boldsymbol{\beta}}\sim N(\mathbf{c}'\boldsymbol{\beta},\,\sigma^2\mathbf{c}'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{c})$.

**Proof.** By (13), $\hat{\boldsymbol{\beta}}=\boldsymbol{\beta}+\mathbf{A}\boldsymbol{\varepsilon}$ with $\mathbf{A}=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'$ and $\boldsymbol{\varepsilon}\sim N(\mathbf{0},\sigma^2\mathbf{I}_n)$. By the prerequisites' theorem **closure under affine maps**, $\hat{\boldsymbol{\beta}}\sim N(\boldsymbol{\beta},\,\sigma^2\mathbf{A}\mathbf{A}')$, and $\mathbf{A}\mathbf{A}'=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}=(\mathbf{X}'\mathbf{X})^{-1}$. Applying the same theorem to the map $\hat{\boldsymbol{\beta}}\mapsto\mathbf{c}'\hat{\boldsymbol{\beta}}$ gives the second statement. ∎

**Theorem 2.34** (Distribution of $\mathrm{SSE}$ and independence from $\hat{\boldsymbol{\beta}}$; CLM Theorem 5.2). Under A1–A5, $\mathrm{SSE}/\sigma^2\sim\chi^2_{n-k-1}$, and $\mathrm{SSE}$ is independent of $\hat{\boldsymbol{\beta}}$.

**Proof.** **Step 1.** With $\mathbf{z}=\boldsymbol{\varepsilon}/\sigma\sim N(\mathbf{0},\mathbf{I}_n)$ and (9), $\mathrm{SSE}/\sigma^2=\boldsymbol{\varepsilon}'\mathbf{M}\boldsymbol{\varepsilon}/\sigma^2=\mathbf{z}'\mathbf{M}\mathbf{z}$.

**Step 2.** $\mathbf{M}$ is symmetric idempotent with $\operatorname{tr}(\mathbf{M})=n-k-1$ (Theorem 2.9), so by the prerequisites' theorem **chi-square quadratic forms**, $\mathbf{z}'\mathbf{M}\mathbf{z}\sim\chi^2_{n-k-1}$.

**Step 3.** The stacked vector

$$
\begin{pmatrix}\hat{\boldsymbol{\beta}}-\boldsymbol{\beta}\\ \mathbf e\end{pmatrix}
=\begin{pmatrix}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\\ \mathbf{M}\end{pmatrix}\boldsymbol{\varepsilon}
$$

is one affine map of one normal vector, hence jointly normal by **closure under affine maps**, with cross-covariance $(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'(\sigma^2\mathbf{I}_n)\mathbf{M}'=\sigma^2(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{M}=\mathbf{0}$, since $\mathbf{X}'\mathbf{M}=(\mathbf{M}\mathbf{X})'=\mathbf{0}$ by Theorem 2.9(ii). By the prerequisites' theorem **uncorrelated blocks are independent**, $\hat{\boldsymbol{\beta}}$ and $\mathbf e$ are independent, and $\mathrm{SSE}=\mathbf e'\mathbf e$ is a function of $\mathbf e$ alone. ∎

**Remark**. A $t$ statistic is a standard normal divided by $\sqrt{\chi^2_\nu/\nu}$ with numerator and denominator **independent**. Here the numerator comes from $\hat{\boldsymbol{\beta}}$ and the denominator from $\mathrm{SSE}$; Step 3 supplies the independence, and without A5 zero covariance would not give it.

### 5.2 Tests and confidence intervals for a coefficient

**Theorem 2.35** (The $t$ statistic; FREES §2.4–2.5, §3.4, ISLR §3.1.2; CLM Theorem 7.1). Under A1–A5, for any constant $\mathbf{c}\neq\mathbf{0}$,

$$
\frac{\mathbf{c}'\hat{\boldsymbol{\beta}}-\mathbf{c}'\boldsymbol{\beta}}{s\sqrt{\mathbf{c}'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{c}}}\sim t_{n-k-1} ; \tag{21}
$$

in particular $(\hat\beta_j-\beta_j)/\mathrm{SE}(\hat\beta_j)\sim t_{n-k-1}$, and for $k=1$, $(b_1-\beta_1)/\mathrm{SE}(b_1)\sim t_{n-2}$.

**Proof.** **Step 1: a standard normal.** By Theorem 2.33, $Z=(\mathbf{c}'\hat{\boldsymbol{\beta}}-\mathbf{c}'\boldsymbol{\beta})/\{\sigma\sqrt{\mathbf{c}'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{c}}\}\sim N(0,1)$, so $Z$ is standard normal by the prerequisites' corollary **standardising a normal variable**.

**Step 2: an independent chi-square.** By Theorem 2.34, $V=\mathrm{SSE}/\sigma^2\sim\chi^2_{n-k-1}$ is independent of $\hat{\boldsymbol{\beta}}$, hence of $Z$, which is a function of $\hat{\boldsymbol{\beta}}$ alone.

**Step 3: the ratio.** By the prerequisites' definition of Student's $t$, $Z/\sqrt{V/(n-k-1)}\sim t_{n-k-1}$, and

$$
\frac{Z}{\sqrt{V/(n-k-1)}}
=\frac{\mathbf{c}'\hat{\boldsymbol{\beta}}-\mathbf{c}'\boldsymbol{\beta}}{\sigma\sqrt{\mathbf{c}'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{c}}}\cdot
 \frac{\sigma}{\sqrt{\mathrm{SSE}/(n-k-1)}}
=\frac{\mathbf{c}'\hat{\boldsymbol{\beta}}-\mathbf{c}'\boldsymbol{\beta}}{s\sqrt{\mathbf{c}'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{c}}} ,
$$

since $s=\sqrt{\mathrm{SSE}/(n-k-1)}$. Taking $\mathbf{c}$ the $j$-th coordinate vector gives the statement for $\hat\beta_j$, and $k=1$ gives $n-2$ degrees of freedom. ∎

**Definition 2.36** ($t$ tests, $p$-values and confidence intervals; FREES §2.5.1–2.5.2, Tables 2.3–2.4 and 3.5–3.6, ISLR §3.1.2; CLM Definition 7.11, Theorem 7.9). To test $H_0\colon\beta_j=d$ at significance level $\alpha$, compute the **$t$ ratio** $t(\hat\beta_j)=(\hat\beta_j-d)/\mathrm{SE}(\hat\beta_j)$ and, with $t_{\nu,q}$ the $q$-th quantile of $t_\nu$ and $\nu=n-k-1$:

| Alternative $H_a$ | Reject $H_0$ if | $p$-value |
|:--|:--|:--|
| $\beta_j>d$ | $t(\hat\beta_j)>t_{\nu,1-\alpha}$ | $\operatorname{\mathsf{P}}(t_\nu>t(\hat\beta_j))$ |
| $\beta_j<d$ | $t(\hat\beta_j)<-t_{\nu,1-\alpha}$ | $\operatorname{\mathsf{P}}(t_\nu<t(\hat\beta_j))$ |
| $\beta_j\neq d$ | $\vert t(\hat\beta_j)\vert >t_{\nu,1-\alpha/2}$ | $\operatorname{\mathsf{P}}(\vert t_\nu\vert >\vert t(\hat\beta_j)\vert )$ |

The $p$-value is the probability, under $H_0$, of a $t$ ratio at least as extreme as the one observed, and $H_0$ is rejected at level $\alpha$ iff $p<\alpha$. The usual test is $H_0\colon\beta_j=0$ against $\beta_j\neq0$, whose $t$ ratio $\hat\beta_j/\mathrm{SE}(\hat\beta_j)$ and $p$-value software prints for every coefficient. A $100(1-\alpha)\%$ **confidence interval** for $\beta_j$ is

$$
\hat\beta_j\pm t_{n-k-1,1-\alpha/2}\,\mathrm{SE}(\hat\beta_j) , \tag{22}
$$

and for $\mathbf{c}'\boldsymbol{\beta}$ it is $\mathbf{c}'\hat{\boldsymbol{\beta}}\pm t_{n-k-1,1-\alpha/2}\,s\sqrt{\mathbf{c}'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{c}}$.

**Proposition 2.37** (Validity of the procedures; CLM Proposition 7.12). Under A1–A5 each test of Definition 2.36 rejects a true $H_0$ with probability exactly $\alpha$, and the interval (22) contains $\beta_j$ with probability exactly $1-\alpha$.

**Proof.** Under $H_0\colon\beta_j=d$ the ratio $t(\hat\beta_j)$ is (21) with $\mathbf{c}$ the $j$-th coordinate vector, hence $t_{n-k-1}$ by Theorem 2.35. For the two-sided test, $\operatorname{\mathsf{P}}(|t_\nu|>t_{\nu,1-\alpha/2})=2\operatorname{\mathsf{P}}(t_\nu>t_{\nu,1-\alpha/2})=\alpha$ by the symmetry of $t_\nu$, which follows from the symmetry of $Z$ in its definition; the one-sided cases are $\operatorname{\mathsf{P}}(t_\nu>t_{\nu,1-\alpha})=\alpha$ and its mirror image. For the interval, with $T=(\hat\beta_j-\beta_j)/\mathrm{SE}(\hat\beta_j)\sim t_\nu$,

$$
1-\alpha=\operatorname{\mathsf{P}}\bigl(-t_{\nu,1-\alpha/2}\leqslant T\leqslant t_{\nu,1-\alpha/2}\bigr)
=\operatorname{\mathsf{P}}\bigl(\hat\beta_j-t_{\nu,1-\alpha/2}\mathrm{SE}(\hat\beta_j)\leqslant\beta_j
\leqslant\hat\beta_j+t_{\nu,1-\alpha/2}\mathrm{SE}(\hat\beta_j)\bigr),
$$

rearranging the inequalities inside the probability. The same argument with (21) covers $\mathbf{c}'\boldsymbol{\beta}$. ∎

**Remark** (ISLR's factor of two; ISLR §3.1.2). ISLR writes the $95\%$ interval as $\hat\beta_j\pm2\,\mathrm{SE}(\hat\beta_j)$; the exact multiplier is $t_{n-k-1,0.975}$, which is $2.01$ at $48$ degrees of freedom and $1.97$ at $198$, and the $t$ law itself rests on A5. Rejecting $H_0\colon\beta_1=0$ is ISLR's “there is a relationship between $X$ and $Y$”; the $p$-value cutoffs $5\%$ and $1\%$ correspond, at $n=30$, to $|t|$ of about $2$ and $2.75$.

In [ ]:
round(c(t48_975 = qt(0.975, 48), t198_975 = qt(0.975, 198),
        t28_975 = qt(0.975, 28), t28_995 = qt(0.995, 28)), 3)

**Example 2.17** (Lottery sales, continued; FREES §2.5). With $b_1=0.647$ and $\mathrm{SE}(b_1)=0.0488$, $t(b_1)=13.3$ on $n-2=48$ degrees of freedom: the slope is $13$ standard errors above zero, far beyond $t_{48,0.975}=2.011$, so $H_0\colon\beta_1=0$ is rejected against $\beta_1\neq0$ at $5\%$, with $p$-value $10^{-17}$. The $95\%$ confidence interval $0.647\pm2.011(0.0488)=(0.549,0.745)$ says that ZIP codes differing by $1{,}000$ in population are expected to differ in sales by between \$549 and \$745; the $90\%$ interval is narrower.

In [ ]:
coef(summary(fit_lot))
c(t_crit = qt(0.975, 48))
confint(fit_lot, level = 0.95)
confint(fit_lot, level = 0.90)

**Example 2.18** (Advertising, continued; ISLR §3.1.2, §3.2.1). For sales on television advertising both $t$ ratios are large and both $p$-values are below $10^{-4}$; the $95\%$ intervals are $[6.13,7.94]$ for the intercept, so sales without advertising fall between $6{,}130$ and $7{,}935$ units on average, and $[0.042,0.053]$ for the slope. In the three-medium model the newspaper $p$-value is $0.86$: with television and radio held fixed there is no evidence that newspaper advertising is associated with sales.

In [ ]:
round(coef(summary(fit_tv)), 4)
round(confint(fit_tv), 4)
round(coef(summary(fit_all)), 4)

### 5.3 The overall $F$ test

**Theorem 2.38** (Overall $F$ test; ISLR §3.2.2; CLM Theorem 7.3, Proposition 7.14, equation (34)). Under A1–A5, for $H_0\colon\beta_1=\cdots=\beta_k=0$,

$$
F=\frac{\mathrm{MSR}}{\mathrm{MSE}}=\frac{\mathrm{SSR}/k}{\mathrm{SSE}/(n-k-1)}
=\frac{(\mathrm{SST}-\mathrm{SSE})/k}{\mathrm{SSE}/(n-k-1)}
=\frac{R^2/k}{(1-R^2)/(n-k-1)} \tag{23}
$$

is distributed as $F_{k,\,n-k-1}$ under $H_0$; it is defined with probability one, since $\mathrm{SSE}/\sigma^2\sim\chi^2_{n-k-1}$ (Theorem 2.34) is positive with probability one and $\mathrm{SST}\geqslant\mathrm{SSE}$; and $\operatorname{\mathsf{E}}[\mathrm{SSR}/k]=\sigma^2+\boldsymbol{\beta}_1'\mathbf{X}_1'\mathbf{M}_0\mathbf{X}_1\boldsymbol{\beta}_1/k\geqslant\sigma^2$ in general, with equality iff $H_0$ holds, where $\mathbf{X}_1$ holds the non-intercept columns and $\boldsymbol{\beta}_1$ the slopes.

**Proof.** **Step 1: $\mathrm{SSR}$ as a quadratic form in the errors.** By Theorem 2.26, $\mathrm{SSR}=\mathbf{Y}'(\mathbf{H}-\mathbf{H}_0)\mathbf{Y}$ with $\mathbf{P}=\mathbf{H}-\mathbf{H}_0$ symmetric idempotent (Lemma 2.10). Since $\mathbf{P}\mathbf{1}=\mathbf{H}\mathbf{1}-\mathbf{H}_0\mathbf{1}=\mathbf{1}-\mathbf{1}=\mathbf{0}$, $\mathbf{P}\mathbf{X}\boldsymbol{\beta}=\mathbf{P}(\beta_0\mathbf{1}+\mathbf{X}_1\boldsymbol{\beta}_1)=\mathbf{P}\mathbf{X}_1\boldsymbol{\beta}_1$, and under $H_0$ this is $\mathbf{0}$, so $\mathbf{P}\mathbf{Y}=\mathbf{P}\boldsymbol{\varepsilon}$ and $\mathrm{SSR}=\boldsymbol{\varepsilon}'\mathbf{P}\boldsymbol{\varepsilon}$.

**Step 2: the two chi-squares.** With $\mathbf{z}=\boldsymbol{\varepsilon}/\sigma\sim N(\mathbf{0},\mathbf{I}_n)$, under $H_0$, $\mathrm{SSR}/\sigma^2=\mathbf{z}'\mathbf{P}\mathbf{z}$ and $\mathrm{SSE}/\sigma^2=\mathbf{z}'\mathbf{M}\mathbf{z}$. Both $\mathbf{P}$ and $\mathbf{M}$ are symmetric idempotent, with traces $k$ and $n-k-1$, so by **chi-square quadratic forms** they are $\chi^2_k$ and $\chi^2_{n-k-1}$.

**Step 3: independence.** $\mathbf{P}\mathbf{M}=\mathbf{0}$ by Lemma 2.10, so by the prerequisites' **Cochran theorem** the two quadratic forms are independent.

**Step 4: the ratio.** By the prerequisites' definition of the $F$ law, the ratio of independent chi-squares each divided by its degrees of freedom is $F_{k,n-k-1}$. The other forms of (23) substitute $\mathrm{SSR}=\mathrm{SST}-\mathrm{SSE}$ and then divide numerator and denominator by $\mathrm{SST}$.

**Step 5: the expectation.** Without $H_0$, $\mathbf{P}\mathbf{Y}=\mathbf{P}\mathbf{X}_1\boldsymbol{\beta}_1+\mathbf{P}\boldsymbol{\varepsilon}$, so by **expectation of a quadratic form** with mean $\mathbf{P}\mathbf{X}_1\boldsymbol{\beta}_1$ and variance $\sigma^2\mathbf{I}_n$, $\operatorname{\mathsf{E}}[\mathrm{SSR}]=\operatorname{\mathsf{E}}[\mathbf{Y}'\mathbf{P}\mathbf{Y}]=\sigma^2\operatorname{tr}(\mathbf{P})+\boldsymbol{\beta}_1'\mathbf{X}_1'\mathbf{P}\mathbf{X}_1\boldsymbol{\beta}_1 =k\sigma^2+\boldsymbol{\beta}_1'\mathbf{X}_1'\mathbf{M}_0\mathbf{X}_1\boldsymbol{\beta}_1$, using $\mathbf{P}\mathbf{X}_1=\mathbf{M}_0\mathbf{X}_1-\mathbf{M}\mathbf{X}_1=\mathbf{M}_0\mathbf{X}_1$, which holds since $\mathbf{M}\mathbf{X}_1=\mathbf{0}$ and $\mathbf{P}=\mathbf{H}-\mathbf{H}_0=(\mathbf{I}-\mathbf{H}_0)-(\mathbf{I}-\mathbf{H})=\mathbf{M}_0-\mathbf{M}$. The quadratic form $\|\mathbf{M}_0\mathbf{X}_1\boldsymbol{\beta}_1\|^2$ is zero iff $\mathbf{X}_1\boldsymbol{\beta}_1$ is constant, which by A4 happens iff $\boldsymbol{\beta}_1=\mathbf{0}$. ∎

**Corollary 2.39** ($R^2$ from $F$; CLM Corollary 7.15). If $\mathrm{SST}>0$ and $\mathrm{SSE}>0$, then $\displaystyle R^2=\frac{kF}{kF+n-k-1}$, so $F$ determines $R^2$ without $\mathrm{SSE}$ and $\mathrm{SST}$.

**Proof.** Solving the last form of (23), $F(1-R^2)(n-k-1)=R^2k$, for $R^2$. ∎

**Remark** (Why $F$ and not the $k$ separate $t$ tests; ISLR §3.2.2). $\operatorname{\mathsf{E}}[\mathrm{MSE}]=\sigma^2$ always and $\operatorname{\mathsf{E}}[\mathrm{MSR}]=\sigma^2$ under $H_0$, so $F$ near $1$ is consistent with no relation and $F$ well above $1$ is evidence of one; how far above depends on $n$ and $k$ through the $F_{k,n-k-1}$ quantile, and a value barely above $1$ can be decisive when $n$ is large. With $k=100$ true null slopes, about five of the individual $p$-values fall below $0.05$ by chance and the smallest very probably does (in $98\%$ of the $2000$ simulated data sets with $n=250$ printed here), so declaring a relation whenever some $t$ test rejects has a false-alarm rate far above $5\%$; the $F$ test rejects a true $H_0$ with probability exactly $5\%$ whatever $k$. When $k+1\geqslant n$ no residual degrees of freedom are left (and for $k\geqslant n$ the least squares coefficients are not unique), so neither statistic exists.

In [ ]:
set.seed(5); n_s <- 250; k_s <- 100
X_s <- cbind(1, matrix(rnorm(n_s * k_s), n_s)); A_s <- solve(crossprod(X_s))
sims <- replicate(2000, {
  y_s <- rnorm(n_s); b_s <- drop(A_s %*% crossprod(X_s, y_s))
  s_s <- sqrt(sum((y_s - X_s %*% b_s)^2) / (n_s - k_s - 1))
  p_s <- 2 * pt(-abs(b_s[-1] / (s_s * sqrt(diag(A_s)[-1]))), n_s - k_s - 1)
  c(n_below = sum(p_s < 0.05), some_below = any(p_s < 0.05))
})
round(rowMeans(sims), 3)

**Proposition 2.40** ($t^2=F$ for one explanatory variable; ISLR §3.2.2; CLM Corollary 7.17). Let $\mathbf{z}$ be one of the explanatory variables in a fitted model, $t$ its $t$ ratio for $H_0\colon\gamma=0$, $\mathrm{SSE}_0$ the residual sum of squares of the model without $\mathbf{z}$, and $\mathrm{SSE}$ that of the full model, with $n-k-1$ residual degrees of freedom. Then, under A4 and $\mathrm{SSE}>0$,

$$
\frac{(\mathrm{SSE}_0-\mathrm{SSE})/1}{\mathrm{SSE}/(n-k-1)}=t^2 , \tag{24}
$$

and under A1–A5 and $H_0$ this statistic is $F_{1,n-k-1}$. For $k=1$ the overall $F$ of (23) is $t(b_1)^2$.

**Proof.** Let $\mathbf{X}_{(\mathbf{z})}$ be the design without the column $\mathbf{z}$ and $\mathbf{M}_{(\mathbf{z})}$ its residual maker. By A4 the columns of $\mathbf{X}_{(\mathbf{z})}$ are linearly independent and $\mathbf{z}\notin\mathcal{C}(\mathbf{X}_{(\mathbf{z})})$, so Lemma 2.30 with $\mathbf{X}_{(\mathbf{z})}$ as the base gives $\mathrm{SSE}_0-\mathrm{SSE}=\hat\gamma^2\mathbf{z}'\mathbf{M}_{(\mathbf{z})}\mathbf{z}$ and $\mathrm{SE}(\hat\gamma)^2=s^2/(\mathbf{z}'\mathbf{M}_{(\mathbf{z})}\mathbf{z})$, and the left side of (24) is $\hat\gamma^2\mathbf{z}'\mathbf{M}_{(\mathbf{z})}\mathbf{z}/s^2=\hat\gamma^2/\mathrm{SE}(\hat\gamma)^2=t^2$. Under $H_0$, $t\sim t_{n-k-1}$ by Theorem 2.35, and by the prerequisites' corollary **the square of a $t$ is an $F$**, $t^2\sim F_{1,n-k-1}$. For $k=1$, $\mathbf{X}_{(\mathbf{z})}=\mathbf{1}$, the intercept alone, whose residual sum of squares is $\mathrm{SST}$, and (24) is (23). ∎

**Example 2.19** (Advertising, continued; ISLR §3.2.2). For the three-medium model $F=570$ on $(3,196)$ degrees of freedom, with a $p$-value of order $10^{-96}$: at least one medium is associated with sales.

In [ ]:
sm <- summary(fit_all); Fst <- sm$fstatistic
SSE_a <- sum(residuals(fit_all)^2); SST_a <- sum((adv$sales - mean(adv$sales))^2)
c(F_lm = Fst[["value"]], F_formula = ((SST_a - SSE_a) / 3) / (SSE_a / 196),
  F_from_R2 = (sm$r.squared / 3) / ((1 - sm$r.squared) / 196),
  p = pf(Fst[["value"]], 3, 196, lower.tail = FALSE))
c(t2_newspaper = coef(sm)["newspaper", "t value"]^2,
  F_drop_newspaper = anova(fit_two, fit_all)[["F"]][2])

Dropping newspaper from the full model gives an $F$ equal to the square of its $t$ ratio, which is Proposition 2.40.

### 5.4 Prediction

**Theorem 2.41** (Confidence and prediction intervals; FREES §2.5.3, ISLR §3.2.2; CLM Theorem 9.1, Theorem 9.2, Corollary 9.4, Corollary 3.10). Let $\mathbf{x}_0$ be a new vector of explanatory variables, $\hat Y_0=\mathbf{x}_0'\hat{\boldsymbol{\beta}}$, and $h_0=\mathbf{x}_0'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_0$. Under A1–A5:

(i) $\operatorname{var}(\hat Y_0)=\sigma^2h_0$, and a $100(1-\alpha)\%$ confidence interval for the mean response $\operatorname{\mathsf{E}}[Y_0]=\mathbf{x}_0'\boldsymbol{\beta}$ is $\hat Y_0\pm t_{n-k-1,1-\alpha/2}\,s\sqrt{h_0}$;

(ii) for a new response $Y_0=\mathbf{x}_0'\boldsymbol{\beta}+\varepsilon_0$ with $\varepsilon_0\sim N(0,\sigma^2)$ independent of $\boldsymbol{\varepsilon}$, $\operatorname{var}(Y_0-\hat Y_0)=\sigma^2(1+h_0)$, and a $100(1-\alpha)\%$ **prediction interval** for $Y_0$ is $\hat Y_0\pm t_{n-k-1,1-\alpha/2}\,s\sqrt{1+h_0}$;

(iii) for $k=1$, $\mathrm{SE}(\mathrm{pred})=s\sqrt{1+h_0}=s\sqrt{1+\dfrac1n+\dfrac{(x_0-\bar x)^2}{(n-1)s_x^2}}$;

(iv) at an observed point $\mathbf{x}_0=\mathbf{x}_i$, $\operatorname{var}(\hat Y_i)=\sigma^2h_{ii}$ and $\operatorname{var}(\hat Y_i)+\operatorname{var}(e_i)=\sigma^2$.

**Proof.** **(i)** $\hat Y_0=\mathbf{x}_0'\hat{\boldsymbol{\beta}}$ is the case $\mathbf{c}=\mathbf{x}_0$ of Theorem 2.33: $\hat Y_0\sim N(\mathbf{x}_0'\boldsymbol{\beta},\sigma^2h_0)$, and the interval is Definition 2.36's interval for $\mathbf{c}'\boldsymbol{\beta}$ with Proposition 2.37.

**(ii)** $Y_0-\hat Y_0=\varepsilon_0-\mathbf{x}_0'(\hat{\boldsymbol{\beta}}-\boldsymbol{\beta})$, and $\hat{\boldsymbol{\beta}}-\boldsymbol{\beta}=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\boldsymbol{\varepsilon}$ is a function of $\boldsymbol{\varepsilon}$, independent of $\varepsilon_0$. Hence the variance is the sum $\sigma^2+\mathbf{x}_0'\operatorname{var}(\hat{\boldsymbol{\beta}})\mathbf{x}_0=\sigma^2+\sigma^2h_0$, and the difference of two independent normals is normal: $Y_0-\hat Y_0\sim N(0,\sigma^2(1+h_0))$. It is independent of $\mathrm{SSE}$, which is a function of $\mathbf e=\mathbf{M}\boldsymbol{\varepsilon}$: $(\hat{\boldsymbol{\beta}}-\boldsymbol{\beta},\mathbf e)$ are independent by Theorem 2.34, and $\varepsilon_0$ is independent of the pair $(\hat{\boldsymbol{\beta}},\mathbf e)$, both being functions of $\boldsymbol{\varepsilon}$. So $(Y_0-\hat Y_0)/\{s\sqrt{1+h_0}\}$ is $(Y_0-\hat Y_0)/\{\sigma\sqrt{1+h_0}\}$, standard normal by the prerequisites' corollary **standardising a normal variable**, over an independent $\sqrt{\chi^2_{n-k-1}/(n-k-1)}$, hence $t_{n-k-1}$, and the interval follows as in Proposition 2.37.

**(iii)** Equation (10) with $x_0$ in place of $x_i$ and $\sum_j(x_j-\bar x)^2=(n-1)s_x^2$.

**(iv)** $h_0=h_{ii}$ by Definition 2.12, and $\operatorname{var}(e_i)=\sigma^2(1-h_{ii})$ by Theorem 2.23. ∎

**Remark** (ISLR §3.2.2, FREES §2.5.3). The confidence interval measures the reducible error, the uncertainty in the estimated plane, and shrinks to nothing as $n$ grows whenever $h_0\to0$, as for $k=1$ when $\bar x$ stays bounded and $s_x$ stays away from $0$ (by (10)); the prediction interval adds the irreducible error $\varepsilon_0$, is always the wider, and cannot be narrower than $2t\,s$. Both are centred at $\hat Y_0$. Leverage is the share of $\sigma^2$ that a point's own fitted value carries, and $1-h_{ii}$ the share left in its residual. Model bias, the error from a linear model being only an approximation, is excluded from both intervals.

**Example 2.20** (Lottery sales, continued; FREES §2.5.3). At $x_0=10{,}000$ the point prediction is $6{,}941$ and

$$
\mathrm{SE}(\mathrm{pred})=3792\sqrt{1+\frac1{50}+\frac{(10{,}000-9{,}311)^2}{49(11{,}098)^2}}=3830 ,
$$

so the $95\%$ prediction interval is $6941\pm2.011(3830)$, which `predict` gives as $(-759,\,14{,}641)$: of many ZIP codes with population $10{,}000$, about $19$ in $20$ would have sales between $0$ and \$14,641, the negative lower limit being truncated at zero.

In [ ]:
x0 <- 10000; xp <- lottery$POP
h0 <- 1/50 + (x0 - mean(xp))^2 / sum((xp - mean(xp))^2)
c(se_pred = s_lot * sqrt(1 + h0), se_mean = s_lot * sqrt(h0))
predict(fit_lot, data.frame(POP = x0), interval = "prediction")
predict(fit_lot, data.frame(POP = x0), interval = "confidence")

**Example 2.21** (Advertising, continued; ISLR §3.2.2). With \$100,000 on television and \$20,000 on radio, the fitted sales are $11{,}256$ units; the $95\%$ confidence interval for the **average** sales of such markets is $[10{,}985,\,11{,}528]$ and the prediction interval for one such market is $[7{,}930,\,14{,}583]$.

In [ ]:
nd <- data.frame(TV = 100, radio = 20)
round(1000 * rbind(confidence = predict(fit_two, nd, interval = "confidence"),
                   prediction = predict(fit_two, nd, interval = "prediction")))

**Example 2.22** (The capital asset pricing model; FREES §2.7). Monthly returns of the Lincoln National Insurance Corporation ($y$) and of the equally weighted S&P 500 index ($x$) over $1986$–$1990$, $n=60$. The CAPM posits $\operatorname{\mathsf{E}}[r]=\beta_0+\beta_1r_m$; $\beta_1$ measures the return attributed to the market, and securities with $\beta_1>1$ are called **aggressive**, with $\beta_1<1$ **defensive**.

In [ ]:
capm <- read.csv("data/frees/csv/CAPM.csv")
round(rbind(LINCOLN = descr(capm$LINCOLN), MARKET = descr(capm$MARKET)), 4)
fit_capm <- lm(LINCOLN ~ MARKET, data = capm)
round(coef(summary(fit_capm)), 4)
c(s = summary(fit_capm)$sigma, R2 = summary(fit_capm)$r.squared,
  crash_month = which.min(capm$MARKET), z_crash = (min(capm$MARKET) - mean(capm$MARKET)) / sd(capm$MARKET))

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
plot(1:60, capm$LINCOLN, type = "l", xlab = "month (from January 1986)", ylab = "monthly return")
lines(1:60, capm$MARKET, lty = 2)
legend("topright", c("LINCOLN", "MARKET"), lty = 1:2, bty = "n", cex = 0.7)
plot(capm$MARKET, capm$LINCOLN, xlab = "MARKET", ylab = "LINCOLN", pch = 16, cex = 0.6)
abline(fit_capm)
par(op)

*Figure 2.7.* Time series plots of the two returns, and Lincoln's return against the market's with the fitted line (FREES Figures 2.10–2.11).

The fitted line is $\widehat{\texttt{LINCOLN}}=-0.0021+0.973\,\texttt{MARKET}$ with $t(b_1)=5.64$: the market is clearly a determinant, yet $R^2=35.4\%$, so it explains only a third of the variation in Lincoln's return. Lincoln is riskier than the market, $s_y=0.0859>0.0525=s_x$, and its five-year mean return, $0.51\%$ a month, is below the market's $0.74\%$. The smallest market return, $-0.221$, is $4.3$ standard deviations below its mean; it is October 1987, the month of the crash, a high leverage point. Corollary 2.32 splits the risk of the security into a market component and a diversifiable one, $s_y^2=b_1^2s_x^2+s^2(n-2)/(n-1)$:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
b1c <- coef(fit_capm)[["MARKET"]]; sc <- summary(fit_capm)$sigma
c(s_y2 = var(capm$LINCOLN), market = b1c^2 * var(capm$MARKET),
  diversifiable = sc^2 * 58 / 59, sum = b1c^2 * var(capm$MARKET) + sc^2 * 58 / 59)

The CAPM question is whether $\beta_1=1$, the market's own sensitivity. By Definition 2.36 with $d=1$, $t=(0.973-1)/\mathrm{SE}(b_1)$, and the hypothesis is not rejected:

In [ ]:
se_b1c <- coef(summary(fit_capm))["MARKET", "Std. Error"]
c(t_beta1_equals_1 = (b1c - 1) / se_b1c, p = 2 * pt(-abs((b1c - 1) / se_b1c), 58))

Refitting without the crash month gives $\widehat{\texttt{LINCOLN}}=-0.00181+0.956\,\texttt{MARKET}$ with $R^2=26.4\%$: removing the high leverage point **lowers** $R^2$, because that month showed the relation over a range the other months do not cover.

In [ ]:
fit_capm1 <- lm(LINCOLN ~ MARKET, data = capm[-22, ])
c(coef(fit_capm1), R2 = summary(fit_capm1)$r.squared, s = summary(fit_capm1)$sigma,
  t = coef(summary(fit_capm1))["MARKET", "t value"])

### 5.5 Maximum likelihood (beyond the syllabus)

> **Reading.** CLM §5. Not in the SRM or PA reading lists.

**Proposition 2.42** (Least squares is maximum likelihood; CLM Theorem 5.3). Under A1–A5, $\hat{\boldsymbol{\beta}}$ is the maximum likelihood estimator of $\boldsymbol{\beta}$, and $\hat\sigma^2_{\mathrm{MLE}}=\mathrm{SSE}/n$, which is biased downward by the factor $(n-k-1)/n$.

**Proof.** $\mathbf{Y}\sim N(\mathbf{X}\boldsymbol{\beta},\sigma^2\mathbf{I}_n)$ has density $(2\pi\sigma^2)^{-n/2}\exp\{-(\mathbf{Y}-\mathbf{X}\boldsymbol{\beta})'(\mathbf{Y}-\mathbf{X}\boldsymbol{\beta})/(2\sigma^2)\}$, so

$$
\ell(\boldsymbol{\beta},\sigma^2)=-\frac n2\ln(2\pi)-\frac n2\ln\sigma^2-\frac{Q(\boldsymbol{\beta})}{2\sigma^2} . \tag{25}
$$

For fixed $\sigma^2$ only the last term involves $\boldsymbol{\beta}$, with the negative coefficient $-1/(2\sigma^2)$, so $\ell$ is maximised over $\boldsymbol{\beta}$ where $Q$ is minimised, at $\hat{\boldsymbol{\beta}}$ by Theorem 2.6, whatever $\sigma^2$. Substituting $Q(\hat{\boldsymbol{\beta}})=\mathrm{SSE}$,

$$
\frac{\partial\ell}{\partial\sigma^2}=-\frac{n}{2\sigma^2}+\frac{\mathrm{SSE}}{2\sigma^4}=0
\;\Longleftrightarrow\; \sigma^2=\frac{\mathrm{SSE}}{n},
$$

and $\partial^2\ell/\partial(\sigma^2)^2=n/(2\sigma^4)-\mathrm{SSE}/\sigma^6$ is negative there, equal to $-n^3/(2\mathrm{SSE}^2)$, and $\partial\ell/\partial\sigma^2=(\mathrm{SSE}-n\sigma^2)/(2\sigma^4)$ changes sign from $+$ to $-$ there, so it is the global maximum ($\mathrm{SSE}>0$ with probability one when $n>k+1$). Its expectation is $(n-k-1)\sigma^2/n$ by Theorem 2.22. ∎

## 6 Residual Analysis and Unusual Points

> **Reading.** FREES §2.6 · ISLR §3.3.3 (outliers, high leverage points) · CLM §2–3. SRM learning outcome 2(d).

**Definition 2.43** (Standardised residuals, outliers, high leverage points; FREES §2.6; CLM Definition 3.11). The **standardised residual** of observation $i$ is, for $s>0$ and $h_{ii}<1$, $e_i/\{s\sqrt{1-h_{ii}}\}$, the residual divided by its estimated standard deviation (Theorem 2.23); FREES's approximation $e_i/s$ ignores the leverage. An observation is an **outlier** if it is unusual in the vertical direction, its standardised residual exceeding $2$ in absolute value, and a **high leverage point** if it is unusual in the horizontal direction, flagged here by $h_{ii}>2(k+1)/n$. A point can be both.

**Remark** (Model discrepancies; FREES §2.6). Residual analysis checks the assumptions of Definition 2.4 against the residuals, which stand in for the unobservable errors, and uses any pattern found to respecify the model. FREES lists five discrepancies: lack of independence among the errors; heteroscedasticity, a variance that changes with the observation; a relation between the errors and an explanatory variable, which means the variable could predict better; non-normal errors, which invalidate the exact inference of §5; and unusual points, single observations that move the fit. Standardised residuals have approximately a standard normal distribution under A5 when $n-k-1$ is large ($e_i/(\sigma\sqrt{1-h_{ii}})$ is exactly $N(0,1)$, $e_i$ being a linear function of the normal errors with variance $\sigma^2(1-h_{ii})$ by Theorem 2.23), and are dimensionless, so experience of their size transfers between data sets.

**Example 2.23** (Outliers and high leverage points; FREES §2.6). FREES's fictitious data: $19$ base points and three unusual points A, B and C.

In [ ]:
ol <- read.csv("data/frees/csv/OutlierExample.csv")
one_fit <- function(rows) {
  m <- lm(Y ~ X, data = ol[rows, ])
  c(b0 = coef(m)[[1]], b1 = coef(m)[[2]], s = summary(m)$sigma,
    R2_pct = 100 * summary(m)$r.squared, t_b1 = coef(summary(m))["X", "t value"])
}
round(rbind(base = one_fit(1:19), base_A = one_fit(1:20),
            base_B = one_fit(c(1:19, 21)), base_C = one_fit(c(1:19, 22))), 3)
round(rbind(std_resid = sapply(20:22, function(i) rstandard(lm(Y ~ X, ol[c(1:19, i), ]))[20]),
            leverage  = sapply(20:22, function(i) hatvalues(lm(Y ~ X, ol[c(1:19, i), ]))[20]),
            flag_lev  = 2 * 2 / 20), 3)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
plot(ol$X, ol$Y, xlim = c(0, 10), ylim = c(2, 9), xlab = "X", ylab = "Y", pch = 16, cex = 0.7)
text(c(4.5, 9.7, 9.7), c(8.0, 8.0, 2.5), c("A", "B", "C"))
abline(lm(Y ~ X, ol[1:19, ]))
par(op)

*Figure 2.8.* Nineteen base points with the line fitted to them, and the three unusual points (FREES Figure 2.7).

The base points are well fitted: $R^2=89\%$, $s=0.29$, $t(b_1)=11.7$. Adding A, an outlier at an ordinary $x$, leaves the line much where it was but drops $R^2$ to $54\%$ and triples $s$. B has A's $y$-value at a large $x$ and lies near the base line: it is a high leverage point but not an outlier, and it **raises** $R^2$ to $95\%$ by enlarging $\mathrm{SST}$ while leaving $\mathrm{SSE}$ nearly unchanged. C has B's $x$ and a low $y$-value, so it is both: the slope falls from $0.61$ to $0.16$, $R^2$ to $10\%$, and the $t$ ratio to $1.4$. One point in twenty can do this, because the coefficients are weighted sums of the responses (Proposition 2.17) with weights that grow with $|x_i-\bar x|$. Here removing the outlier A would restore the fit, while removing the high leverage point B would lower $R^2$.

**Example 2.24** (Lottery sales, continued: Kenosha; FREES §2.6). The point at the upper left of the scatter plot is the ZIP code containing Kenosha, on the Illinois border, whose sales are far above what its population predicts.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
i_ken <- which.max(residuals(fit_lot))
c(row = i_ken, POP = lottery$POP[i_ken], SALES = lottery$SALES[i_ken],
  std_resid = rstandard(fit_lot)[i_ken], leverage = hatvalues(fit_lot)[i_ken])
fit_ken <- lm(SALES ~ POP, data = lottery[-i_ken, ])
round(rbind(with_Kenosha = one_fit_lot <- c(coef(fit_lot), s = s_lot,
                R2_pct = 100 * summary(fit_lot)$r.squared,
                t_b1 = coef(summary(fit_lot))["POP", "t value"]),
            without = c(coef(fit_ken), s = summary(fit_ken)$sigma,
                R2_pct = 100 * summary(fit_ken)$r.squared,
                t_b1 = coef(summary(fit_ken))["POP", "t value"])), 3)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 1.6, 0.6), mgp = c(1.9, 0.6, 0))
qqnorm(rstandard(fit_lot), main = "all 50 ZIP codes", cex = 0.6); qqline(rstandard(fit_lot))
qqnorm(rstandard(fit_ken), main = "without Kenosha", cex = 0.6); qqline(rstandard(fit_ken))
par(op)

*Figure 2.9.* Normal quantile plots of the standardised residuals with and without Kenosha (FREES Figure 2.9).

Kenosha is an outlier with a standardised residual of $4.9$ and ordinary leverage. The slope barely changes when it is removed, from $0.647$ to $0.662$, but its $t$ ratio rises from $13.3$ to $18.8$ because $s$ falls by more than \$1,000 (from $3792$ to $2728$), and $R^2$ rises from $78.5\%$ to $88.3\%$; the smaller $s$ matters for prediction intervals. The quantile plot with Kenosha climbs away from the line at the upper end; without it the points follow the line much more closely, with mild departures in both tails, so the apparent non-normality was one observation, not skewness shared by all.

## 7 Computation in R

> **Reading.** FREES §2.8 · ISLR §3.6.1–3.6.3.

### 7.1 Reading regression output

FREES §2.8 annotates the output of Excel, SAS and R for the lottery regression; every printed quantity is a formula of this unit.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
summary(fit_lot)
anova(fit_lot)

In [ ]:
X_l <- model.matrix(fit_lot); y_l <- lottery$SALES; n_l <- nrow(X_l)
b_l <- solve(crossprod(X_l), crossprod(X_l, y_l))           # (X'X)^{-1} X'y
e_l <- y_l - X_l %*% b_l
SSE_l <- sum(e_l^2); SST_l <- sum((y_l - mean(y_l))^2); SSR_l <- SST_l - SSE_l
s2_l <- SSE_l / (n_l - 2)
se_l <- sqrt(s2_l * diag(solve(crossprod(X_l))))
t_l <- drop(b_l) / se_l
c(n = n_l, b0 = b_l[1], b1 = b_l[2], se_b0 = se_l[1], se_b1 = se_l[2], t_b0 = t_l[1], t_b1 = t_l[2])
c(p_b0 = 2 * pt(-abs(t_l[1]), n_l - 2), p_b1 = 2 * pt(-abs(t_l[2]), n_l - 2),
  s = sqrt(s2_l), R2 = SSR_l / SST_l, R2_adj = 1 - s2_l / (SST_l / (n_l - 1)),
  multiple_R = sqrt(SSR_l / SST_l))
c(SSR = SSR_l, SSE = SSE_l, SST = SST_l, MSR = SSR_l / 1, MSE = s2_l,
  F = (SSR_l / 1) / s2_l, p_F = pf((SSR_l / 1) / s2_l, 1, n_l - 2, lower.tail = FALSE))

The coefficient table holds $\hat\beta_j$, $\mathrm{SE}(\hat\beta_j)$, the $t$ ratio for $H_0\colon\beta_j=0$ and its two-sided $p$-value (Definition 2.36); the residual standard error is $s$ on $n-k-1$ degrees of freedom; `Multiple R-squared` and `Adjusted R-squared` are (18); the $F$ statistic is (23) on $(k,n-k-1)$ degrees of freedom; and `anova` prints the regression and error rows of the ANOVA table, the total row being their sum.

### 7.2 The ISLR lab

The `Boston` data of `ISLR2`: the median house value `medv` of $506$ census tracts with $12$ predictors, among them `lstat`, the percentage of households of low socioeconomic status.

In [ ]:
lm.fit <- lm(medv ~ lstat, data = Boston)
summary(lm.fit)
names(lm.fit)
coef(lm.fit)
confint(lm.fit)

`predict` gives the confidence interval for the mean response and the prediction interval for a new response at chosen values of the predictor (Theorem 2.41): at $\texttt{lstat}=10$ the fitted value is $25.05$, the confidence interval $(24.47,25.63)$ and the prediction interval $(12.83,37.28)$.

In [ ]:
predict(lm.fit, data.frame(lstat = c(5, 10, 15)), interval = "confidence")
predict(lm.fit, data.frame(lstat = c(5, 10, 15)), interval = "prediction")

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
plot(Boston$lstat, Boston$medv, xlab = "lstat", ylab = "medv", pch = 16, cex = 0.4)
abline(lm.fit, lwd = 2, col = "#B45309")
plot(predict(lm.fit), residuals(lm.fit), xlab = "fitted", ylab = "residual", pch = 16, cex = 0.4)
abline(h = 0, lty = 2)
plot(hatvalues(lm.fit), xlab = "observation", ylab = "leverage", pch = 16, cex = 0.4)
abline(h = 2 * 2 / 506, lty = 2)
par(op)

*Figure 2.10.* `medv` against `lstat` with the least squares line; residuals against fitted values; leverages with the flag $2(k+1)/n$.

The residuals bend: negative in the middle, positive at both ends, so the relation is not linear. Leverage is computed by `hatvalues`, and `which.max` names the observation with the largest.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
c(max_leverage = max(hatvalues(lm.fit)), at = which.max(hatvalues(lm.fit)),
  mean_leverage = 2 / 506, lstat_there = Boston$lstat[which.max(hatvalues(lm.fit))],
  max_lstat = max(Boston$lstat))

Several predictors are joined by `+`, all of them by `.`, and all but one by `- age`; `update` refits with a changed formula.

In [ ]:
lm.fit2 <- lm(medv ~ lstat + age, data = Boston)
round(coef(summary(lm.fit2)), 5)
lm.fit3 <- lm(medv ~ ., data = Boston)
summary(lm.fit3)

In [ ]:
c(R2 = summary(lm.fit3)$r.squared, RSE = summary(lm.fit3)$sigma)
lm.fit4 <- update(lm.fit3, ~ . - age)
c(R2_without_age = summary(lm.fit4)$r.squared, RSE_without_age = summary(lm.fit4)$sigma)

Dropping `age`, whose $p$-value is $0.79$, leaves $R^2$ unchanged to three decimals and lowers the residual standard error, as Theorem 2.31 requires of a variable with $|t|<1$.

### 7.3 Formulas against package output

The fit is `medv` on `lstat`, `age` and `rm`, $k=3$.

In [ ]:
fit <- lm(medv ~ lstat + age + rm, data = Boston)
X <- model.matrix(fit); y <- Boston$medv; e <- residuals(fit)
n <- nrow(X); k <- ncol(X) - 1L
c(n = n, k = k, df = n - k - 1L)

In [ ]:
XtX   <- t(X) %*% X
b_man <- drop(solve(XtX, t(X) %*% y))         # (X'X)^{-1} X'Y without forming the inverse
cbind(manual = b_man, lm = coef(fit), diff = b_man - coef(fit))

`lm` does not form $\mathbf{X}'\mathbf{X}$ at all; it factors $\mathbf{X}=\mathbf{Q}\mathbf{R}$, which avoids squaring the condition number. Equation (5) is how least squares is reasoned about, not how it is computed.

In [ ]:
s2    <- sum(e^2) / (n - k - 1)               # MSE
V_man <- s2 * solve(XtX)                      # s^2 (X'X)^{-1}
c(max_abs_diff = max(abs(V_man - vcov(fit))), s2_manual = s2, s2_lm = summary(fit)$sigma^2)

In [ ]:
H <- X %*% solve(XtX) %*% t(X); M <- diag(n) - H
c(symmetric = max(abs(H - t(H))), idempotent = max(abs(H %*% H - H)),
  HM_zero = max(abs(H %*% M)), MX_zero = max(abs(M %*% X)),
  tr_H = sum(diag(H)), tr_M = sum(diag(M)))
c(fitted = max(abs(drop(H %*% y) - fitted(fit))), resid = max(abs(drop(M %*% y) - e)),
  leverage = max(abs(diag(H) - hatvalues(fit))),
  Xt_e = max(abs(t(X) %*% e)), sum_e = sum(e), yhat_t_e = sum(fitted(fit) * e))

In [ ]:
r_man <- e / (sqrt(s2) * sqrt(1 - hatvalues(fit)))
c(max_abs_diff = max(abs(r_man - rstandard(fit))), h_min = min(hatvalues(fit)),
  h_max = max(hatvalues(fit)))

In [ ]:
se_man <- sqrt(diag(V_man)); t_man <- coef(fit) / se_man
p_man  <- 2 * pt(abs(t_man), df = n - k - 1, lower.tail = FALSE)
tab    <- coef(summary(fit))
round(cbind(se_man, se_lm = tab[, 2], t_man, t_lm = tab[, 3], p_man, p_lm = tab[, 4]), 8)

In [ ]:
SSE <- sum(e^2); SST <- sum((y - mean(y))^2); SSR <- SST - SSE
seq_ss <- anova(fit)[["Sum Sq"]]                # sequential sums of squares
c(SSR = SSR, sum_sequential = sum(seq_ss[1:k]), SSE = SSE, anova_residual = seq_ss[k + 1L],
  SSR_projection = sum(((H - matrix(1/n, n, n)) %*% y)^2))
R2 <- 1 - SSE / SST; adjR2 <- 1 - (n - 1) / (n - k - 1) * (1 - R2)
F_man <- (SSR / k) / (SSE / (n - k - 1))
c(R2 = R2, R2_lm = summary(fit)$r.squared, R2_cor2 = cor(y, fitted(fit))^2,
  adj = adjR2, adj_lm = summary(fit)$adj.r.squared,
  F = F_man, F_lm = summary(fit)$fstatistic[["value"]],
  F_anova_null = anova(lm(medv ~ 1, data = Boston), fit)[["F"]][2],
  R2_from_F = k * F_man / (k * F_man + n - k - 1))

In [ ]:
x0 <- c(1, lstat = 10, age = 60, rm = 6.5); nd0 <- data.frame(lstat = 10, age = 60, rm = 6.5)
lev0 <- drop(t(x0) %*% solve(XtX) %*% x0); tq <- qt(0.975, n - k - 1)
pr <- predict(fit, nd0, se.fit = TRUE)
c(yhat_manual = sum(x0 * coef(fit)), yhat_R = unname(pr$fit),
  se_fit_manual = sqrt(s2 * lev0), se_fit_R = pr$se.fit, width_ratio = sqrt((1 + lev0) / lev0))
rbind(manual = c(pr$fit + c(-1, 1) * tq * sqrt(s2 * lev0), pr$fit + c(-1, 1) * tq * sqrt(s2 * (1 + lev0))),
      R      = c(predict(fit, nd0, interval = "confidence")[2:3],
                 predict(fit, nd0, interval = "prediction")[2:3]))

At this point the prediction interval is $21$ times as wide as the confidence interval, the ratio being $\sqrt{(1+h_0)/h_0}$ by Theorem 2.41.

In [ ]:
m_no <- lm(medv ~ lstat - 1, data = Boston)
c(R2_with = summary(lm.fit)$r.squared, R2_without = summary(m_no)$r.squared,
  SSE_with = sum(residuals(lm.fit)^2), SSE_without = sum(residuals(m_no)^2),
  sum_resid_without = sum(residuals(m_no)))

Without an intercept the residuals do not sum to zero, Theorem 2.26 does not apply, and R switches to an uncentred $R^2$, $1-\mathrm{SSE}/\sum_iY_i^2$, which is not comparable with the centred one (here $0.449$ against $0.544$, with an $\mathrm{SSE}$ $8.5$ times larger).

### 7.4 Maximum likelihood in R (beyond the syllabus)

In [ ]:
s2_mle <- SSE / n
ll_man <- -n/2 * log(2*pi) - n/2 * log(s2_mle) - SSE / (2 * s2_mle)
c(s2_unbiased = s2, s2_mle = s2_mle, ratio = s2_mle / s2, factor = (n - k - 1) / n,
  loglik_manual = ll_man, loglik_R = as.numeric(logLik(fit)), npar_R = attr(logLik(fit), "df"))

`logLik` uses $\hat\sigma^2_{\mathrm{MLE}}$ of Proposition 2.42 and counts $k+2$ parameters, the coefficients and $\sigma$.

### 7.5 The sampling theory by simulation

The design is held fixed, as A1–A5 require, and only the errors are redrawn: the means of the estimates, their variances, the moments of $\mathrm{SSE}/\sigma^2$ and its independence from $\hat{\boldsymbol{\beta}}$ are Theorems 2.18, 2.19, 2.34.

In [ ]:
set.seed(1)
Xs <- X[1:60, ]; ns <- nrow(Xs); ks <- ncol(Xs) - 1L
beta_true <- c(10, -0.5, 0.02, 3); sigma <- 4; B <- 5000
bhat <- matrix(NA_real_, B, ks + 1L); sse <- numeric(B)
for (bb in seq_len(B)) {
  ysim <- drop(Xs %*% beta_true) + rnorm(ns, sd = sigma)
  f <- lm.fit(Xs, ysim)
  bhat[bb, ] <- f$coefficients; sse[bb] <- sum(f$residuals^2)
}
rbind(true = beta_true, mean_bhat = colMeans(bhat))
round(cbind(empirical = apply(bhat, 2, var), theory = sigma^2 * diag(solve(t(Xs) %*% Xs))), 5)
c(mean_SSE_over_s2 = mean(sse) / sigma^2, df = ns - ks - 1L,
  var_SSE_over_s2 = var(sse / sigma^2), two_df = 2 * (ns - ks - 1L))
round(cor(bhat, sse), 4)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
op <- par(mar = c(3.2, 3.4, 1, 1), mgp = c(1.9, 0.6, 0))
hist(sse / sigma^2, breaks = 40, freq = FALSE, col = "grey88", border = "white",
     main = "", xlab = expression(SSE/sigma^2))
curve(dchisq(x, df = ns - ks - 1), add = TRUE, lwd = 2, col = "#B45309")
par(op)

*Figure 2.11.* Simulated $\mathrm{SSE}/\sigma^2$ over $5000$ replications on a fixed design of $60$ rows against the $\chi^2_{56}$ density.

The mean and variance of $\mathrm{SSE}/\sigma^2$ are those of $\chi^2_{56}$, $56$ and $112$, and its correlations with the four estimates are zero to Monte-Carlo error.

## 8 Exercises

**Exercise 2.1** (FREES exercise 2.1). For the data $x=(2,-6,7)$, $y=(3,4,6)$, fit a line by least squares: determine $r$, $b_1$ and $b_0$.

**Solution.** $\bar x=1$ and $\bar y=13/3$; the deviations are $(1,-7,6)$ and $(-4/3,-1/3,5/3)$, so $\sum_i(x_i-\bar x)(y_i-\bar y)=-4/3+7/3+10=11$, $\sum_i(x_i-\bar x)^2=1+49+36=86$ and $\sum_i(y_i-\bar y)^2=(16+1+25)/9=14/3$. By (1) and Corollary 2.14,

$$
r=\frac{11}{\sqrt{86\cdot14/3}}=0.549,\qquad b_1=\frac{11}{86}=0.128,\qquad
b_0=\frac{13}{3}-\frac{11}{86}=4.205 .
$$

■

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
xa <- c(2, -6, 7); ya <- c(3, 4, 6)
c(r = cor(xa, ya), b1 = 11/86, b0 = 13/3 - 11/86); coef(lm(ya ~ xa))

**Exercise 2.2** (FREES exercise 2.2). For $x=(-2,-1,0,1,2)$ and $y=x^2=(4,1,0,1,4)$: (a) graph the data; (b) check that $r=0$.

**Solution.**

In [ ]:
options(repr.plot.width = 3, repr.plot.height = 2.4)
xq <- -2:2; yq <- xq^2
c(r = cor(xq, yq), cross_products = sum((xq - mean(xq)) * (yq - mean(yq))))
plot(xq, yq, pch = 16, xlab = "x", ylab = "y")

(b) $\bar x=0$ and $\bar y=2$, so $\sum_i(x_i-\bar x)(y_i-\bar y)=\sum_ix_i(y_i-2) =(-2)(2)+(-1)(-1)+0+(1)(-1)+(2)(2)=0$ and $r=0$: a perfect relation that is not linear has zero correlation, since by Proposition 2.2 $|r|=1$ needs a straight line. ■

**Exercise 2.3** (FREES exercise 2.3). With $a$ and $c$ constants and standardised values $u_i=(x_i-\bar x)/s_x$, $v_i=(y_i-\bar y)/s_y$: (a) verify $0\leqslant(n-1)^{-1}\sum_i(au_i-cv_i)^2=a^2+c^2-2acr$; (b) deduce $2ac(r-1)\leqslant(a-c)^2$; (c) take $a=c$ to show $r\leqslant1$; (d) take $a=-c$ to show $r\geqslant-1$; (e) when is $r=-1$, and when $r=1$?

**Solution.** (a) The left side is a mean of squares, hence non-negative. Expanding, $(n-1)^{-1}\sum_i(au_i-cv_i)^2=a^2(n-1)^{-1}\sum_iu_i^2+c^2(n-1)^{-1}\sum_iv_i^2 -2ac(n-1)^{-1}\sum_iu_iv_i$, and $(n-1)^{-1}\sum_iu_i^2=s_x^2/s_x^2=1$, likewise for $v$, while $(n-1)^{-1}\sum_iu_iv_i=r$ by (1). (b) $a^2+c^2-2acr\geqslant0$ is $2acr\leqslant a^2+c^2$; subtracting $2ac$ from both sides, $2ac(r-1)\leqslant a^2-2ac+c^2=(a-c)^2$. (c) With $a=c\neq0$, (b) reads $2a^2(r-1)\leqslant0$, so $r\leqslant1$. (d) With $a=-c\neq0$, (b) reads $-2a^2(r-1)\leqslant4a^2$, that is $r-1\geqslant-2$, so $r\geqslant-1$. (e) $r=1$ iff the sum in (a) vanishes at $a=c$, that is $u_i=v_i$ for every $i$: each $y_i$ is the same number of standard deviations from its mean as $x_i$, the points lying on a line of positive slope $s_y/s_x$. $r=-1$ iff $u_i=-v_i$ for every $i$, a line of negative slope. This is Proposition 2.2 by another route. ■

**Exercise 2.4** (FREES exercise 2.4). Show that $b_0=\sum_iw_{i,0}y_i$ is a weighted sum of the responses and express $w_{i,0}$ through the slope weights $w_i$ of Proposition 2.17.

**Solution.** $b_0=\bar y-b_1\bar x=\sum_i\tfrac1ny_i-\bar x\sum_iw_iy_i=\sum_i(\tfrac1n-\bar xw_i)y_i$, so $w_{i,0}=1/n-\bar xw_i$. The weights sum to $1$, since $\sum_iw_i=0$, and $b_0$ is the mean of the responses when $\bar x=0$. ■

**Exercise 2.5** (FREES exercise 2.5). (a) With $\mathit{slope}_i=(y_i-\bar y)/(x_i-\bar x)$ the slope from $(\bar x,\bar y)$ to $(x_i,y_i)$, show $b_1=\sum_i\mathit{weight}_i\,\mathit{slope}_i/\sum_i\mathit{weight}_i$ and identify $\mathit{weight}_i$. (b) With $\bar x=4$, $\bar y=3$, $x_1=2$, $y_1=6$, find $\mathit{slope}_1$ and $\mathit{weight}_1$.

**Solution.** (a) With $\mathit{weight}_i=(x_i-\bar x)^2$, $\sum_i\mathit{weight}_i\,\mathit{slope}_i=\sum_i(x_i-\bar x)(y_i-\bar y)$ and $\sum_i\mathit{weight}_i=\sum_i(x_i-\bar x)^2$, whose ratio is $b_1$ by Corollary 2.14: the least squares slope is the average of the point slopes weighted by squared distance from $\bar x$, which is why distant points dominate. (b) $\mathit{slope}_1=(6-3)/(2-4)=-1.5$ and $\mathit{weight}_1=(2-4)^2=4$. ■

**Exercise 2.6** (FREES exercise 2.6). Let $b_{y|x}$ be the slope from regressing $y$ on $x$ and $b_{x|y}$ that from regressing $x$ on $y$, both with intercepts. Show $|r|=\sqrt{b_{y|x}\,b_{x|y}}$.

**Solution.** By Corollary 2.14, $b_{y|x}=r\,s_y/s_x$ and, exchanging the roles of the variables, $b_{x|y}=r\,s_x/s_y$. Their product is $r^2$, so $|r|=\sqrt{b_{y|x}b_{x|y}}$; the two slopes share the sign of $r$. For $r\neq0$, drawn on the same axes the second line has slope $1/b_{x|y}=s_y/(rs_x)$, and the ratio of the first slope to it is $r^2$: the lines coincide iff $r^2=1$, and otherwise the regression of $y$ on $x$ is the flatter. In standard units the fitted line is $\hat v=ru$, so a value one standard deviation above average in $x$ is predicted $|r|<1$ standard deviations above average in $y$, the regression toward the mean of Unit 1. ■

**Exercise 2.7** (FREES exercise 2.7). For regression through the origin, $y_i=\beta_1x_i+\varepsilon_i$: (a) verify $b_1=\sum_ix_iy_i/\sum_ix_i^2$; (b) find the least squares estimate in the model $y_i=\beta_1z_i^2+\varepsilon_i$.

**Solution.** (a) $\mathrm{d}\sum_i(y_i-bx_i)^2/\mathrm{d}b=-2\sum_ix_i(y_i-bx_i)=0$ gives $b\sum_ix_i^2=\sum_ix_iy_i$, and the second derivative $2\sum_ix_i^2>0$ makes it the minimum; this is Corollary 2.15. (b) The model is linear in $\beta_1$ with explanatory variable $x_i=z_i^2$, so $b_1=\sum_iz_i^2y_i/\sum_iz_i^4$. ■

**Exercise 2.8** (FREES exercise 2.8). Show (a) $\sum_i(y_i-\bar y)^2=\sum_iy_i^2-n\bar y^2$; (b) $\sum_i(y_i-\bar y)(x_i-\bar x)=\sum_ix_iy_i-n\bar x\bar y$; (c) $b_1=\sum_i(y_i-\bar y)(x_i-\bar x)/\sum_i(x_i-\bar x)^2$; and (d) $b_1=(\sum_ix_iy_i-n\bar x\bar y)/(\sum_ix_i^2-n\bar x^2)$.

**Solution.** (a) $\sum_i(y_i-\bar y)^2=\sum_iy_i^2-2\bar y\sum_iy_i+n\bar y^2=\sum_iy_i^2-2n\bar y^2+n\bar y^2$. (b) $\sum_i(y_i-\bar y)(x_i-\bar x)=\sum_ix_iy_i-\bar x\sum_iy_i-\bar y\sum_ix_i+n\bar x\bar y =\sum_ix_iy_i-n\bar x\bar y$. (c) is the first form of (11), from the normal equations. (d) Substitute (a), applied to $x$, and (b) into (c). ■

**Exercise 2.9** (FREES exercise 2.9). Suppose $x_i$ takes only the values $0$ and $1$: $n_1$ observations have $x=0$ with mean response $\bar y_1$, and $n_2=n-n_1$ have $x=1$ with mean $\bar y_2$. Show $b_1=\bar y_2-\bar y_1$ and $b_0=\bar y_1$, and give $\mathrm{SE}(b_1)$ in terms of $n_1$, $n_2$ and $s$.

**Solution.** $\bar x=n_2/n$ and $x_i^2=x_i$, so $\sum_i(x_i-\bar x)^2=\sum_ix_i^2-n\bar x^2=n_2-n_2^2/n=n_1n_2/n$, and $\sum_i(x_i-\bar x)(y_i-\bar y)=\sum_ix_iy_i-n\bar x\bar y =n_2\bar y_2-\tfrac{n_2}{n}(n_1\bar y_1+n_2\bar y_2)=\tfrac{n_1n_2}{n}(\bar y_2-\bar y_1)$. By Corollary 2.14 the ratio is $b_1=\bar y_2-\bar y_1$, and $b_0=\bar y-b_1\bar x=\tfrac1n(n_1\bar y_1+n_2\bar y_2)-\tfrac{n_2}{n}(\bar y_2-\bar y_1)=\bar y_1$. The fitted values are the two group means, so $\mathrm{SSE}$ is the within-group sum of squares and $s^2$ the pooled variance on $n-2$ degrees of freedom; by Corollary 2.20,

$$
\mathrm{SE}(b_1)=\frac{s}{\sqrt{n_1n_2/n}}=s\sqrt{\frac1{n_1}+\frac1{n_2}} ,
$$

so $t(b_1)$ is the pooled two-sample $t$ statistic: regression on one indicator is the two-sample comparison of means. ■

**Exercise 2.10** (FREES exercise 2.10). Nursing home data of 2000 (`WiscNursingHome`, Unit 1). Part 1: (a) the correlations (i) between `TPY` and $\texttt{LOGTPY}=\ln\texttt{TPY}$, (ii) among `TPY`, `NUMBED` and `SQRFOOT`, (iii) between `TPY` and $\texttt{NUMBED}/10$; (b) scatter plots of `TPY` against `NUMBED` and against `SQRFOOT`; (c) regressions of `TPY` on `NUMBED`, of `TPY` on `SQRFOOT`, of `LOGTPY` on $\ln\texttt{NUMBED}$ and of `LOGTPY` on $\ln\texttt{SQRFOOT}$, each summarised by $R^2$ and the $t$ ratio of the slope. Part 2: refit `TPY` on `NUMBED` for 2001.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
nh <- read.csv("data/frees/csv/WiscNursingHome.csv")
nh0 <- subset(nh, CRYEAR == 2000); nh1 <- subset(nh, CRYEAR == 2001)
c(cor_TPY_logTPY = cor(nh0$TPY, log(nh0$TPY)), cor_TPY_NUMBED_over_10 = cor(nh0$TPY, nh0$NUMBED / 10))
round(cor(nh0[, c("TPY", "NUMBED", "SQRFOOT")], use = "complete.obs"), 3)
one_slr <- function(f, d) { m <- lm(f, d)
  c(n = nobs(m), b1 = coef(m)[[2]], R2 = summary(m)$r.squared, t_b1 = coef(summary(m))[2, 3]) }
round(rbind(TPY_NUMBED = one_slr(TPY ~ NUMBED, nh0), TPY_SQRFOOT = one_slr(TPY ~ SQRFOOT, nh0),
            logTPY_logNUMBED = one_slr(log(TPY) ~ log(NUMBED), nh0),
            logTPY_logSQRFOOT = one_slr(log(TPY) ~ log(SQRFOOT), nh0),
            TPY_NUMBED_2001 = one_slr(TPY ~ NUMBED, nh1)), 3)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
plot(nh0$NUMBED, nh0$TPY, xlab = "NUMBED", ylab = "TPY", pch = 16, cex = 0.5)
plot(nh0$SQRFOOT, nh0$TPY, xlab = "SQRFOOT", ylab = "TPY", pch = 16, cex = 0.5)
par(op)

(a)(i) The correlation between `TPY` and its logarithm is $0.937$: the logarithm is increasing, so the association is strong, but it is not linear, so $r<1$. (ii) `TPY` and `NUMBED` are correlated at $0.979$, the square footage with either at about $0.82$. (iii) Dividing `NUMBED` by $10$ leaves the correlation at $0.979$, by the scale invariance of Proposition 2.2. (b) Patient years rise linearly with beds along a tight band; against square footage the band is wider. (c) On `NUMBED`, $R^2=0.959$ with $t=91$; on `SQRFOOT`, $R^2=0.680$ with $t=27$ on the $357$ facilities that report it, so beds fit far better. In logarithms the slope on $\ln\texttt{NUMBED}$ is $1.01$, an elasticity of one, with $R^2=0.948$; on $\ln\texttt{SQRFOOT}$, $R^2=0.677$. Part 2: in 2001 the slope on `NUMBED` is $0.932$ against $0.921$ in 2000, with $R^2=0.976$: the relation is stable. ■

**Exercise 2.11** (FREES exercise 2.11). For $n=3$, $e_2=24$ and $e_3=-1$; determine $e_1$.

**Solution.** By Theorem 2.11(ii) the residuals of a fit with an intercept sum to zero, so $e_1=-(24-1)=-23$. ■

**Exercise 2.12** (FREES exercise 2.12). With $r=0$, $n=15$ and $s_y=10$, determine $s$.

**Solution.** By Corollary 2.32, $R^2=r^2=0$, so $\mathrm{SSR}=0$ and $\mathrm{SSE}=\mathrm{SST}=(n-1)s_y^2=14(100)=1400$; then $s^2=\mathrm{SSE}/(n-2)=1400/13=107.7$ and $s=10.38$. The line explains nothing, and $s$ exceeds $s_y$ because a degree of freedom was spent on the useless slope. ■

**Exercise 2.13** (FREES exercise 2.13). (a) Show $\hat y_i-\bar y=b_1(x_i-\bar x)$; (b) deduce $\mathrm{SSR}=b_1^2s_x^2(n-1)$; (c) establish $R^2=r^2$.

**Solution.** (a) $\hat y_i-\bar y=b_0+b_1x_i-\bar y=(\bar y-b_1\bar x)+b_1x_i-\bar y=b_1(x_i-\bar x)$. (b) Squaring and summing, $\mathrm{SSR}=b_1^2\sum_i(x_i-\bar x)^2=b_1^2(n-1)s_x^2$. (c) With $b_1=r\,s_y/s_x$, $\mathrm{SSR}=r^2(n-1)s_y^2=r^2\mathrm{SST}$, so $R^2=\mathrm{SSR}/\mathrm{SST}=r^2$; this is Corollary 2.32. ■

**Exercise 2.14** (FREES exercise 2.14). Show that the average residual is zero.

**Solution.** The first normal equation, $\sum_i(y_i-b_0-b_1x_i)=0$, is $\sum_ie_i=0$; in general it is the first row of $\mathbf{X}'\mathbf e=\mathbf{0}$, Theorem 2.11(ii), and holds because the intercept column $\mathbf{1}$ is in the design. ■

**Exercise 2.15** (FREES exercise 2.15). (a) If $\bar x=0$ or $\bar y=0$, show that $r(y,x)=0$ iff $\sum_iy_ix_i=0$. (b) Show that the residuals are uncorrelated with the explanatory variable, and (c) with the fitted values.

**Solution.** (a) $\sum_i(y_i-\bar y)(x_i-\bar x)=\sum_iy_ix_i-n\bar x\bar y$ by Exercise 2.8(b), and the second term vanishes when either mean is zero; so the numerator of $r$ is $\sum_iy_ix_i$ and, the denominator being positive, $r=0$ iff the sum is zero. (b) $\bar e=0$ by Exercise 2.14, and $\sum_ix_ie_i=0$ is the second normal equation (Theorem 2.11(i)), so by (a) with $y$ replaced by $e$, $r(e,x)=0$. (c) $\sum_i\hat y_ie_i=0$ is Theorem 2.11(iii), and $\bar e=0$ again gives $r(e,\hat y)=0$. ■

**Exercise 2.16** (FREES exercise 2.16). Show (a) $R^2=1-\frac{n-2}{n-1}\,\frac{s^2}{s_y^2}$; (b) $s=s_y\sqrt{(1-r^2)\frac{n-1}{n-2}}$; (c) $t(b_1)=\sqrt{n-2}\,\dfrac{r}{\sqrt{1-r^2}}$.

**Solution.** (a) $R^2=1-\mathrm{SSE}/\mathrm{SST}$ with $\mathrm{SSE}=(n-2)s^2$ and $\mathrm{SST}=(n-1)s_y^2$. (b) Solve (a) for $s^2$ using $R^2=r^2$: $s^2=s_y^2(1-r^2)(n-1)/(n-2)$. (c) By Definition 2.24 and Corollary 2.14,

$$
\begin{align*}
t(b_1)&=\frac{b_1}{\mathrm{SE}(b_1)}=\frac{r\,s_y/s_x}{s/(s_x\sqrt{n-1})}=\frac{r\,s_y\sqrt{n-1}}{s}\\
&=\frac{r\,s_y\sqrt{n-1}}{s_y\sqrt{(1-r^2)(n-1)/(n-2)}}=\sqrt{n-2}\,\frac{r}{\sqrt{1-r^2}} .
\end{align*}
$$

The $t$ statistic is a function of $r$ and $n$ alone, symmetric in $x$ and $y$, so regressing $x$ on $y$ gives the same $t$; for the lottery data, $\sqrt{48}\,(0.8863)/\sqrt{1-0.7855}=13.3$. ■

**Exercise 2.17** (FREES exercise 2.17). In a regression with one predictor on $n=100$ observations the tenth residual is large. What percentage of $\mathrm{SSE}$ is due to it if (a) $e_{10}=8s$; (b) $e_{10}=4s$; (c) $e_{10}=4s$ with $n=20$?

**Solution.** $\mathrm{SSE}=(n-2)s^2$, so $e_{10}^2/\mathrm{SSE}=c^2/(n-2)$ when $e_{10}=cs$: (a) $64/98=65.3\%$; (b) $16/98=16.3\%$; (c) $16/18=88.9\%$. A residual of $8s$ at $n=100$ owns two thirds of the error sum of squares, and $c^2\leqslant n-2$ is forced, since one squared residual cannot exceed their sum. ■

**Exercise 2.18** (FREES exercise 2.18). With $n=20$, $\bar x=0$, $\bar y=9$, $s_x=1$, $s_y=10$ and $s=7$, determine the standard error of a prediction at $x_\ast=1$.

**Solution.** By Theorem 2.41(iii),

$$
\mathrm{SE}(\mathrm{pred})=7\sqrt{1+\frac1{20}+\frac{(1-0)^2}{19(1)}}=7\sqrt{1.1026}=7.35 ,
$$

almost all of it the $\sigma$ of the new observation; $\bar y$ and $s_y$ are not needed. ■

**Exercise 2.19** (FREES exercise 2.19). Anscombe's data (`AnscombesData`): (a) compute the means and standard deviations of each column; (b) regress $y_1$, $y_2$, $y_3$ on $x_1$ and $y_4$ on $x_2$, and verify $b_0\approx3.0$, $b_1\approx0.5$, $s\approx1.237$, $R^2\approx0.677$ in each case; (c) draw the four scatter plots; (d) discuss.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
an <- read.csv("data/frees/csv/AnscombesData.csv")
round(sapply(an[, -1], function(v) c(mean = mean(v), sd = sd(v))), 3)
one_fit_a <- function(f) { m <- lm(f, an)
  c(b0 = coef(m)[[1]], b1 = coef(m)[[2]], s = summary(m)$sigma, R2 = summary(m)$r.squared) }
round(rbind(y1_x1 = one_fit_a(y1 ~ x1), y2_x1 = one_fit_a(y2 ~ x1),
            y3_x1 = one_fit_a(y3 ~ x1), y4_x2 = one_fit_a(y4 ~ x2)), 3)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.2)
op <- par(mfrow = c(1, 4), mar = c(3.2, 3.2, 1.2, 0.4), mgp = c(1.9, 0.6, 0))
for (p in list(c("x1", "y1"), c("x1", "y2"), c("x1", "y3"), c("x2", "y4"))) {
  plot(an[[p[1]]], an[[p[2]]], xlab = p[1], ylab = p[2], pch = 16, cex = 0.7)
  abline(lm(an[[p[2]]] ~ an[[p[1]]]))
}
par(op)

(a)–(b) The two $x$ columns share mean $9.00$ and standard deviation $3.32$, the four $y$ columns mean $7.50$ and standard deviation $2.03$, and the four fits agree in $b_0$, $b_1$, $s$ and $R^2$ to two decimals, with $R^2=0.666$ or $0.667$ in each case: the exercise's $0.677$ is a misprint in FREES. (c)–(d) The plots differ entirely: a linear relation with scatter; a smooth curve that a line misrepresents; a linear relation with one outlier that tilts the line; and a single high leverage point that alone determines the slope, the other ten $x$ values being equal. Summary statistics and the fitted line cannot distinguish the four, so a regression is read with its scatter plot. ■

**Exercise 2.20** (FREES exercise 2.20). Nursing home data of 2001, $\texttt{LOGTPY}=\ln\texttt{TPY}$ on $\texttt{LOGNUMBED}=\ln\texttt{NUMBED}$: (a) summary statistics, the correlation and a scatter plot; (b) the fit, with $R^2$, the slope and its $t$ ratio; (c) at the $5\%$ level, with $p$-values, test $H_0\colon\beta_1=0$ against $\beta_1\neq0$ and $H_0\colon\beta_1=1$ against $\beta_1\neq1$, $\beta_1>1$ and $\beta_1<1$; (d) a point estimate and $95\%$ and $99\%$ confidence intervals for the expected change in `LOGTPY` when `LOGNUMBED` changes by $2$; (e) at $100$ beds, the predicted `LOGTPY`, its standard error of prediction, a $95\%$ prediction interval, the same converted to patient years, and a $90\%$ interval.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
nh1$LOGTPY <- log(nh1$TPY); nh1$LOGNUMBED <- log(nh1$NUMBED)
round(rbind(LOGTPY = descr(nh1$LOGTPY), LOGNUMBED = descr(nh1$LOGNUMBED)), 3)
fit_nh <- lm(LOGTPY ~ LOGNUMBED, data = nh1)
c(cor = cor(nh1$LOGTPY, nh1$LOGNUMBED), R2 = summary(fit_nh)$r.squared, n = nobs(fit_nh))
round(coef(summary(fit_nh)), 4)
b1n <- coef(fit_nh)[["LOGNUMBED"]]; se1n <- coef(summary(fit_nh))["LOGNUMBED", 2]; t1 <- (b1n - 1) / se1n
c(t_beta1_is_1 = t1, crit_two = qt(0.975, 353), crit_one = qt(0.95, 353),
  p_two = 2 * pt(-abs(t1), 353), p_greater = pt(t1, 353, lower.tail = FALSE), p_less = pt(t1, 353))
rbind(estimate = 2 * b1n, CI95 = 2 * (b1n + c(-1, 1) * qt(0.975, 353) * se1n),
      CI99 = 2 * (b1n + c(-1, 1) * qt(0.995, 353) * se1n))
nd_nh <- data.frame(LOGNUMBED = log(100))
pr_nh <- predict(fit_nh, nd_nh, se.fit = TRUE)
c(fit = pr_nh$fit, se_pred = sqrt(summary(fit_nh)$sigma^2 + pr_nh$se.fit^2))
rbind(log_scale_95 = predict(fit_nh, nd_nh, interval = "prediction"),
      patient_years_95 = exp(predict(fit_nh, nd_nh, interval = "prediction")),
      patient_years_90 = exp(predict(fit_nh, nd_nh, interval = "prediction", level = 0.90)))

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
op <- par(mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
plot(nh1$LOGNUMBED, nh1$LOGTPY, xlab = "LOGNUMBED", ylab = "LOGTPY", pch = 16, cex = 0.5)
abline(fit_nh)
par(op)

(a)–(b) The two logarithms are correlated at $0.983$; the slope is $1.019$ with standard error $0.010$, $t=101$ and $R^2=0.966$. (c) $H_0\colon\beta_1=0$ is rejected at any level. For $H_0\colon\beta_1=1$ the $t$ ratio is $(1.019-1)/0.010=1.90$: against $\beta_1\neq1$ it is below $t_{353,0.975}=1.97$, so $H_0$ is not rejected, $p=0.058$; against $\beta_1>1$ it exceeds $t_{353,0.95}=1.65$, so $H_0$ is rejected, $p=0.029$; against $\beta_1<1$ it is not rejected, $p=0.971$. (d) The expected change is $2b_1=2.04$, with intervals $(2.00,2.08)$ and $(1.99,2.09)$, twice those for $\beta_1$ by Definition 2.36 with $\mathbf{c}=(0,2)'$. (e) At $\ln100=4.605$ the prediction is $4.52$ with standard error of prediction $0.094$, nearly all of it $s$; the $95\%$ interval $(4.33,4.70)$ becomes $(76.3,110.4)$ patient years around $91.7$ after exponentiation, and the $90\%$ interval is $(78.6,107.1)$. ■

**Exercise 2.21** (FREES exercise 2.21). Initial public offerings (`IPO`): one-year `RETURN` and 1997 revenue `REV` in millions of dollars for the $110$ of $116$ firms reporting both, with $\texttt{LnREV}=\ln\texttt{REV}$. (a) With $r(\texttt{RETURN},\texttt{REV})=-0.0175$: (i) fit `RETURN` on `REV` from the summary statistics; (ii) the fitted return of Hyperion Telecommunications, $\texttt{REV}=95.55$. (b) With the fit on `LnREV`: (i) the fitted return of Hyperion, compared with a(ii); (ii) test at $10\%$ whether logarithmic revenue affects returns; (iii) test at $10\%$ the null of no relation against a positive relation. (c) The correlation between `LnREV` and `RETURN`. (d) At $\texttt{LnREV}=2$, the point prediction and a $95\%$ prediction interval. (e) Inktomi, $\texttt{LnREV}=1.76$ and $\texttt{RETURN}=4.333$: (i) its residual; (ii) its share of $\mathrm{SSE}$; (iii) the definition of a high leverage point; (iv) whether Inktomi is one.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
ipo <- read.csv("data/frees/csv/IPO.csv"); ipo <- ipo[complete.cases(ipo[, c("RETURN", "REV")]), ]
round(t(sapply(ipo[, c("RETURN", "REV", "LnREV")], descr)), 3)
b1_rev <- cor(ipo$RETURN, ipo$REV) * sd(ipo$RETURN) / sd(ipo$REV)
b0_rev <- mean(ipo$RETURN) - b1_rev * mean(ipo$REV)
c(r = cor(ipo$RETURN, ipo$REV), b1 = b1_rev, b0 = b0_rev, hyperion_REV = b0_rev + b1_rev * 95.55)
fit_ipo <- lm(RETURN ~ LnREV, data = ipo)
round(coef(summary(fit_ipo)), 4)
c(s = summary(fit_ipo)$sigma, R2 = summary(fit_ipo)$r.squared, r_LnREV = cor(ipo$LnREV, ipo$RETURN),
  hyperion_LnREV = unname(predict(fit_ipo, data.frame(LnREV = log(95.55)))),
  crit_two_10 = qt(0.95, 108), crit_one_10 = qt(0.90, 108))
predict(fit_ipo, data.frame(LnREV = 2), interval = "prediction")
i_ink <- which.max(ipo$RETURN)
c(company = ipo$COMPANY[i_ink], LnREV = ipo$LnREV[i_ink], RETURN = ipo$RETURN[i_ink])
c(residual = residuals(fit_ipo)[[i_ink]], share_of_SSE = residuals(fit_ipo)[[i_ink]]^2 / sum(residuals(fit_ipo)^2),
  leverage = hatvalues(fit_ipo)[[i_ink]], mean_leverage = 2 / 110, flag = 4 / 110,
  std_resid = rstandard(fit_ipo)[[i_ink]])

(a) $b_1=r\,s_y/s_x=-0.0175(0.824)/261.9=-5.5\times10^{-5}$ and $b_0=0.106-b_1(134.5)=0.113$; Hyperion's fitted return is $0.113-5.5\times10^{-5}(95.55)=0.108$. (b)(i) On the logarithmic scale the fit is $0.4381-0.0902\,\texttt{LnREV}$ and Hyperion's fitted return is $0.4381-0.0902\ln95.55=0.027$: two models, two fitted values. (ii) $H_0\colon\beta_1=0$ against $\beta_1\neq0$ at $10\%$: $|t|=1.97$ exceeds $t_{108,0.95}=1.66$, so $H_0$ is rejected, with $p=0.052$. (iii) Against $\beta_1>0$ the rejection region is $t>t_{108,0.90}=1.29$, and $t=-1.97$ is not in it: $H_0$ is not rejected, the evidence pointing the other way. (c) $r=-0.186$, negative like the slope, and $r^2=0.0345=R^2$. (d) $0.438-0.090(2)=0.258$, with prediction interval $(-1.37,1.89)$: the interval is $3.3$ wide because $s=0.81$ is large relative to the fitted values. (e)(i) Inktomi's residual is $4.333-\{0.438-0.090(1.76)\}=4.05$; (ii) its square is $23\%$ of $\mathrm{SSE}$; (iii) a high leverage point is unusual in the explanatory variable, with $h_{ii}$ above $2(k+1)/n$ (Definition 2.43); (iv) its leverage $0.021$ is barely above the mean $0.018$ and below the flag $0.036$, since $1.76$ is one standard deviation below the mean of `LnREV`: Inktomi is an outlier, with standardised residual $5.0$, and not a high leverage point. ■

**Exercise 2.22** (FREES exercise 2.22). Regress `LIFEEXP` on `FERTILITY` (`UNLifeExpectancy`, Unit 1). (a) The fitted life expectancy of the United States, $\texttt{FERTILITY}=2.0$; (b) a $95\%$ prediction interval for Dominica, not in the regression, at $2.0$; (c) China, $\texttt{FERTILITY}=1.7$ and $\texttt{LIFEEXP}=72.5$: its residual and the residual as a multiple of $s$; (d) test at $5\%$ the prior hypothesis that the slope is $-6.0$ against the alternative that it is greater, with an approximate $p$-value.

**Solution.**

In [ ]:
un <- read.csv("data/frees/csv/UNLifeExpectancy.csv")
fit_un <- lm(LIFEEXP ~ FERTILITY, data = un)
round(coef(summary(fit_un)), 4)
c(n = nobs(fit_un), s = summary(fit_un)$sigma, R2 = summary(fit_un)$r.squared)
rbind(US = predict(fit_un, data.frame(FERTILITY = 2.0)),
      Dominica = predict(fit_un, data.frame(FERTILITY = 2.0), interval = "prediction"))
res_china <- 72.5 - predict(fit_un, data.frame(FERTILITY = 1.7))
c(residual = unname(res_china), multiple_of_s = unname(res_china) / summary(fit_un)$sigma)
b1u <- coef(fit_un)[["FERTILITY"]]; se1u <- coef(summary(fit_un))["FERTILITY", 2]
c(b1 = b1u, t = (b1u + 6) / se1u, crit = qt(0.95, nobs(fit_un) - 2), p = pt((b1u + 6) / se1u, nobs(fit_un) - 2, lower.tail = FALSE))

The regression uses the $181$ countries reporting fertility: $\widehat{\texttt{LIFEEXP}}=83.7-5.27\,\texttt{FERTILITY}$, $s=6.6$, $R^2=0.65$. (a) At $2.0$ births per woman the fitted life expectancy is $73.2$ years. (b) Dominica's prediction interval is $(60.1,86.3)$, about $\pm2s$. (c) China's fitted value is $74.8$, so its residual is $-2.3$ years, $0.34$ of $s$: unremarkable. (d) $H_0\colon\beta_1=-6$ against $\beta_1>-6$: $t=(-5.2735+6)/0.2887=2.52$ exceeds $t_{179,0.95}=1.65$, so $H_0$ is rejected with $p=0.006$; the slope is less steep than $-6$. ■

**Exercise 2.23** (FREES exercise 3.1). A data set of $n=100$ observations has $s_y=80$; a regression on three explanatory variables gives $s=50$. (a) Calculate $R_a^2$; (b) complete the ANOVA table; (c) calculate $R^2$.

**Solution.** Two mean squares determine everything: $\mathrm{SST}=(n-1)s_y^2$ and $\mathrm{SSE}=(n-k-1)s^2$, and $\mathrm{SSR}=\mathrm{SST}-\mathrm{SSE}$ by Theorem 2.26.

In [ ]:
nA <- 100; kA <- 3; syA <- 80; sA <- 50
SSTa <- (nA - 1) * syA^2; SSEa <- (nA - kA - 1) * sA^2; SSRa <- SSTa - SSEa
c(SST = SSTa, SSR = SSRa, SSE = SSEa, MSR = SSRa / kA, MSE = sA^2, F = (SSRa / kA) / sA^2,
  R2 = SSRa / SSTa, R2_adj = 1 - sA^2 / syA^2)

(a) $R_a^2=1-s^2/s_y^2=1-2500/6400=0.609$, from (18) without any sum of squares. (b)

| Source | Sum of squares | $\operatorname{df}$ | Mean square |
|:--|--:|--:|--:|
| Regression | $393{,}600$ | $3$ | $131{,}200$ |
| Error | $240{,}000$ | $96$ | $2{,}500$ |
| Total | $633{,}600$ | $99$ |  |

(c) $R^2=393{,}600/633{,}600=0.621$, above $R_a^2$ as always when $k\geqslant1$. ■

**Exercise 2.24** (FREES exercise 3.2). With $n=100$, $k=3$, $s=50$ and

$$
(\mathbf{X}'\mathbf{X})^{-1}=\begin{pmatrix}
100 & 20 & 20 & 20\\ 20 & 90 & 30 & 40\\ 20 & 30 & 80 & 50\\ 20 & 40 & 50 & 70
\end{pmatrix},
$$

determine (a) $\mathrm{SE}(b_3)$; (b) the estimated $\operatorname{cov}(b_2,b_3)$; (c) the estimated $\operatorname{cor}(b_2,b_3)$; (d) the estimated variance of $4b_2+3b_3$.

**Solution.** By Theorem 2.19 the estimated covariance matrix is $s^2(\mathbf{X}'\mathbf{X})^{-1}$, so $\mathrm{SE}(b_j)=s\sqrt{[(\mathbf{X}'\mathbf{X})^{-1}]_{jj}}$ and $\widehat{\operatorname{cov}}(b_i,b_j)=s^2[(\mathbf{X}'\mathbf{X})^{-1}]_{ij}$; the correlation cancels $s^2$ and can be read off $(\mathbf{X}'\mathbf{X})^{-1}$ alone. For the linear combination, **variance under a linear map** with $\mathbf{c}=(0,0,4,3)'$ gives $\widehat{\operatorname{var}}(\mathbf{c}'\hat{\boldsymbol{\beta}})=s^2\mathbf{c}'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{c}=s^2\{16(80)+9(70)+2(4)(3)(50)\}$.

In [ ]:
XtXi <- rbind(c(100, 20, 20, 20), c(20, 90, 30, 40), c(20, 30, 80, 50), c(20, 40, 50, 70))
sF <- 50; cF <- c(0, 0, 4, 3)
c(se_b3 = sF * sqrt(XtXi[4, 4]), cov_b23 = sF^2 * XtXi[3, 4],
  cor_b23 = XtXi[3, 4] / sqrt(XtXi[3, 3] * XtXi[4, 4]),
  var_lin = sF^2 * drop(t(cF) %*% XtXi %*% cF), se_lin = sF * sqrt(drop(t(cF) %*% XtXi %*% cF)))

(a) $\mathrm{SE}(b_3)=50\sqrt{70}=418$; (b) $2500(50)=125{,}000$; (c) $50/\sqrt{80\cdot70}=0.668$; (d) $2500(1280+630+1200)=7{,}775{,}000$, of which the covariance term contributes $3{,}000{,}000$: ignoring it would understate the standard error of $4b_2+3b_3$ by the factor $\sqrt{7775/4775}=1.28$. ■

**Exercise 2.25** (FREES exercise 3.3). For the data

$$
\mathbf{x}_1=(-1,2,4,6)',\quad \mathbf{x}_2=(0,0,1,1)',\quad \mathbf{y}=(0,1,5,8)',
$$

the fit of $y=\beta_0+\beta_1x_1+\beta_2x_2+\varepsilon$ has $s=1.373$, $\mathbf b=(0.15,0.692,2.88)'$ and

$$
(\mathbf{X}'\mathbf{X})^{-1}=\begin{pmatrix}
0.53846 & -0.07692 & -0.15385\\ -0.07692 & 0.15385 & -0.69231\\ -0.15385 & -0.69231 & 4.11538
\end{pmatrix}.
$$

(a) Write $\mathbf{y}$ and $\mathbf{X}$; (b) determine $\hat y_3$; (c) $\mathrm{SE}(b_2)$; (d) $t(b_1)$.

**Solution.** (a) $\mathbf{X}$ has the intercept column first: it is $4\times3$ with rows $(1,-1,0)$, $(1,2,0)$, $(1,4,1)$, $(1,6,1)$, and $n-k-1=1$.

In [ ]:
Xf <- cbind(1, x1 = c(-1, 2, 4, 6), x2 = c(0, 0, 1, 1)); yf <- c(0, 1, 5, 8)
Af <- solve(crossprod(Xf)); bf <- drop(Af %*% crossprod(Xf, yf))
ef <- yf - Xf %*% bf; sf <- sqrt(sum(ef^2) / (4 - 3))
round(Af, 5)
sprintf("b%d = %.5f", 0:2, bf)
c(b0 = bf[1], b1 = bf[2], b2 = bf[3], s = sf, yhat3 = drop(Xf[3, ] %*% bf),
  se_b2 = sf * sqrt(Af[3, 3]), t_b1 = bf[2] / (sf * sqrt(Af[2, 2])),
  p_b1 = 2 * pt(-abs(bf[2] / (sf * sqrt(Af[2, 2]))), df = 1))

(b) $\hat y_3=0.154+0.692(4)+2.885(1)=5.81$; (c) with $s=1.3728$, $\mathrm{SE}(b_2)=1.3728\sqrt{4.11538}=2.785$; (d) $t(b_1)=0.69231/(1.3728\sqrt{0.15385})=1.29$ on one degree of freedom, whose two-sided $p$-value is $0.42$ and $5\%$ critical value $12.7$: four observations and three parameters leave almost nothing with which to estimate $\sigma^2$. ■

**Exercise 2.26** (FREES exercise 3.4, parts (a)–(f), (i), (j)). The lottery data with the eight explanatory variables of `WiscLottery` (`PERPERHH`, `MEDSCHYR`, `MEDHVL`, `PRCRENT`, `PRC55P`, `HHMEDAGE`, `MEDINC`, `POP`). (a) Summary statistics; how many standard deviations above the mean are `MEDSCHYR` and `MEDHVL` for observation 11 (Shorewood)? (b) Which three variables are most correlated with `SALES`? (c) A scatterplot matrix; where is observation 11 in the plot of `SALES` against `MEDSCHYR`? (d) Fit `SALES` on all eight variables and report $s$, $R^2$, $R_a^2$. (e) Is `MEDSCHYR` significant? State the hypotheses, the criterion and the decision. (f) Fit `SALES` on `MEDSCHYR`, `MEDHVL` and `POP`; compare $s$, $R^2$, $R_a^2$ with (d). (i) Refit (f) without observation 11: is `MEDSCHYR` significant? (j) Refit (f) without observation 9.

**Solution.**

In [ ]:
round(t(sapply(lottery[, -1], descr)), 2)
c(z_MEDSCHYR_11 = (lottery$MEDSCHYR[11] - mean(lottery$MEDSCHYR)) / sd(lottery$MEDSCHYR),
  z_MEDHVL_11 = (lottery$MEDHVL[11] - mean(lottery$MEDHVL)) / sd(lottery$MEDHVL))
round(sort(cor(lottery[, -1])[, "SALES"], decreasing = TRUE), 3)
fit_stats3 <- function(m) c(n = nobs(m), s = summary(m)$sigma, R2 = summary(m)$r.squared,
                            R2_adj = summary(m)$adj.r.squared)
fit_l8 <- lm(SALES ~ . - ZIP, data = lottery)
fit_l3 <- lm(SALES ~ MEDSCHYR + MEDHVL + POP, data = lottery)
fit_l3_no11 <- update(fit_l3, data = lottery[-11, ]); fit_l3_no9 <- update(fit_l3, data = lottery[-9, ])
round(rbind(eight = fit_stats3(fit_l8), three = fit_stats3(fit_l3),
            three_no_11 = fit_stats3(fit_l3_no11), three_no_9 = fit_stats3(fit_l3_no9)), 3)
round(coef(summary(fit_l8))["MEDSCHYR", ], 4); c(crit = qt(0.975, 41))
round(coef(summary(fit_l3)), 4)
round(coef(summary(fit_l3_no11))["MEDSCHYR", ], 4); c(crit = qt(0.975, 45))

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3.6)
pairs(lottery[, c("SALES", "POP", "MEDHVL", "MEDSCHYR", "PRCRENT", "MEDINC")],
      pch = 16, cex = 0.35, gap = 0.3, cex.labels = 0.9)

(a) Observation 11 is $5.8$ standard deviations above the mean in `MEDSCHYR` and $3.4$ in `MEDHVL`. (b) `POP` ($0.886$), `MEDHVL` ($0.572$) and `PRCRENT` ($0.496$). (c) In the panel of `SALES` against `MEDSCHYR` observation 11 sits alone at the far right with moderate sales. (d) $s=3580$, $R^2=0.837$, $R_a^2=0.805$ on $41$ residual degrees of freedom. (e) $H_0\colon\beta_{\texttt{MEDSCHYR}}=0$ against $\neq0$; reject at $5\%$ if $|t|>t_{41,0.975}=2.02$; $t=-2.26$ with $p=0.029$, so $H_0$ is rejected. (f) The three-variable model has $s=3445$, $R^2=0.830$ and $R_a^2=0.819$: a lower $R^2$, as Proposition 2.28 requires of a submodel, but a smaller $s$, hence by (18) a larger $R_a^2$. (i) Without Shorewood the `MEDSCHYR` $t$ ratio is $-0.81$ with $p=0.42$, so $H_0$ is no longer rejected: the significance in (e) rested on one high leverage observation. (j) Without Kenosha, $s$ falls to $2585$ and $R^2$ rises to $0.899$, the outlier of Example 2.24 again. ■

**Exercise 2.27** (FREES exercise 3.5, parts (a)–(f)). Insurer expenses (`NAICExpense`, Unit 1), with the outputs `LONGLOSS`, `SHORTLOSS`, `GPWPERSONAL`, `GPWCOMM`, the size measures `ASSETS`, `CASH`, the indicators `GROUP`, `STOCK`, `MUTUAL`, and the wages `STAFFWAGE`, `AGENTWAGE`. (a) Summary statistics of the response and the non-binary explanatory variables, with their skewness; note the negative values. (b) Transform each non-binary variable by $\ln(1+x)$, with $\texttt{LNEXPENSES}=\ln(1+\texttt{EXPENSES})$. (c) Which three variables are most correlated with `LNEXPENSES`? (d) A box plot of `LNEXPENSES` by `GROUP`: which level has higher expenses? (e) Fit `LNEXPENSES` on all eleven explanatory variables and report $s$, $R^2$, $R_a^2$. (f) Fit the eight-variable model dropping `CASH`, `STOCK` and `MUTUAL`: (i) $s$, $R^2$, $R_a^2$; (ii) interpret the coefficient of `GPWCOMM`; (iii) the expected increase in `EXPENSES` when `GPWCOMM` increases by \$1, at the medians.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
naic <- read.csv("data/frees/csv/NAICExpense.csv")
skew <- function(v) mean((v - mean(v))^3) / mean((v - mean(v))^2)^1.5
vars <- c("EXPENSES", "LONGLOSS", "SHORTLOSS", "GPWPERSONAL", "GPWCOMM", "ASSETS", "CASH",
          "STAFFWAGE", "AGENTWAGE")
round(t(sapply(naic[, vars], function(v) { v <- v[!is.na(v)]
  c(n = length(v), median = median(v), mean = mean(v), sd = sd(v), min = min(v), skew = skew(v),
    negative = sum(v < 0)) })), 3)
L <- as.data.frame(lapply(naic[, vars], function(v) log(1 + v))); names(L) <- paste0("LN", vars)
L$GROUP <- naic$GROUP; L$STOCK <- naic$STOCK; L$MUTUAL <- naic$MUTUAL
round(sort(cor(L[, paste0("LN", vars)], use = "complete.obs")[, "LNEXPENSES"], decreasing = TRUE), 3)
tapply(L$LNEXPENSES, L$GROUP, median)
fit_n11 <- lm(LNEXPENSES ~ ., data = L)
fit_n8 <- lm(LNEXPENSES ~ LNASSETS + GROUP + LNLONGLOSS + LNSHORTLOSS + LNGPWPERSONAL + LNGPWCOMM +
               LNSTAFFWAGE + LNAGENTWAGE, data = L)
round(rbind(eleven = fit_stats3(fit_n11), eight = fit_stats3(fit_n8)), 4)
round(coef(summary(fit_n8)), 4)
mG <- median(naic$GPWCOMM); mE <- median(naic$EXPENSES)
c(median_GPWCOMM = mG, median_EXPENSES = mE,
  dEXPENSES_per_dollar = coef(fit_n8)[["LNGPWCOMM"]] * (1 + mE) / (1 + mG))

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
op <- par(mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
boxplot(LNEXPENSES ~ GROUP, data = L, col = "grey85", xlab = "GROUP", ylab = "LNEXPENSES")
par(op)

(a) Every dollar variable is heavily skewed to the right, with skewness between $5.6$ and $7.2$ and means several times the medians; expenses, both losses and both premium variables take negative values for a few companies, so the plain logarithm is undefined for them, and `AGENTWAGE` is missing for $19$ companies. (b) The shifted logarithm $\ln(1+x)$ is defined for every value here, the smallest being $-0.071$. (c) `LNSHORTLOSS` ($0.930$), `LNLONGLOSS` ($0.918$) and `LNGPWCOMM` ($0.864$). (d) Affiliated companies, $\texttt{GROUP}=1$, have the higher expenses, a median of $0.016$ against $0.002$. (e) On the $365$ companies with complete data, $s=0.0224$, $R^2=0.942$, $R_a^2=0.940$. (f)(i) The eight-variable model has $s=0.0224$, $R^2=0.941$ and $R_a^2=0.940$: dropping three variables costs a thousandth of $R^2$ and nothing in $R_a^2$. (ii) The coefficient $0.069$ of `LNGPWCOMM` is an elasticity between the shifted variables: a $1\%$ increase in $1+\texttt{GPWCOMM}$ is associated with a $0.069\%$ increase in $1+\texttt{EXPENSES}$, other variables fixed. (iii) Differentiating $\ln(1+E)=\cdots+0.069\ln(1+G)$ gives $\mathrm dE=0.069\,(1+E)/(1+G)\,\mathrm dG$; at the medians, $E=0.0085$ and $G=0.0238$ million, a dollar more of commercial premium goes with $0.068$ dollars more expense. ■

**Exercise 2.28** (FREES exercise 3.6, parts (a)–(c)). Regress life expectancy, `LIFEEXP`, on the three variables `FERTILITY`, `PUBLICEDUCATION` and $\texttt{lnHEALTH}=\ln\texttt{PRIVATEHEALTH}$. (a) Interpret the coefficient of public education. (b) Interpret the coefficient of health expenditure without the logarithmic scale. (c) Is `PUBLICEDUCATION` significant? State the hypotheses, the criterion and the decision.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
un$lnHEALTH <- log(un$PRIVATEHEALTH)
fit_un3 <- lm(LIFEEXP ~ FERTILITY + PUBLICEDUCATION + lnHEALTH, data = un)
round(coef(summary(fit_un3)), 4)
c(n = nobs(fit_un3), R2 = summary(fit_un3)$r.squared, crit = qt(0.975, nobs(fit_un3) - 4),
  cor_PUBED_LIFEEXP = cor(un$PUBLICEDUCATION, un$LIFEEXP, use = "complete.obs"))

The fit uses the $152$ countries with all four variables. (a) Holding fertility and private health expenditure fixed, a country spending one more percentage point of GDP on public education has an expected life expectancy $0.18$ years **lower**, although the simple correlation of the two variables is positive ($0.07$). (b) The coefficient $-1.03$ of $\ln\texttt{PRIVATEHEALTH}$ means that a $1\%$ increase in private health expenditure, as a share of GDP, is associated with a change of $-1.03/100=-0.010$ years in expected life expectancy, the other variables fixed; doubling it, a change of $\ln2$ in the logarithm, with $-0.71$ years. (c) $H_0\colon\beta_{\texttt{PUBED}}=0$ against $\neq0$; reject at $5\%$ if $|t|>t_{148,0.975}=1.98$; $t=-0.69$ with $p=0.49$, so $H_0$ is not rejected. ■

**Exercise 2.29** (ISLR §3.7, exercise 1). Describe the null hypotheses of the $p$-values in ISLR Table 3.4, the regression of `sales` on `TV`, `radio` and `newspaper` (Example 2.18), and the conclusions they support, in terms of the variables rather than the coefficients.

**Solution.** Each $p$-value tests, by Definition 2.36, $H_0\colon\beta_j=0$ against $\beta_j\neq0$ with the other two media held fixed: for `TV`, that television advertising has no association with sales in markets with given radio and newspaper budgets; likewise for `radio` and for `newspaper`; and for the intercept, that sales are zero when nothing is spent. The $p$-values below $10^{-4}$ for `TV` and `radio` reject their nulls: each medium is associated with sales after adjusting for the other two. The $p$-value $0.86$ for `newspaper` does not: given the television and radio budgets, newspaper spending carries no evidence of an association, its simple association in Example 2.10 being borrowed from radio. ■

**Exercise 2.30** (ISLR §3.7, exercise 4). With $n=100$ observations on one predictor, a linear and a cubic regression $Y=\beta_0+\beta_1X+\beta_2X^2+\beta_3X^3+\varepsilon$ are fitted. (a) If the true relation is linear, which has the lower training $\mathrm{RSS}$? (b) The lower test $\mathrm{RSS}$? (c)–(d) The same when the truth is non-linear to an unknown degree.

**Solution.** (a) The cubic, or a tie: the linear design's column space is contained in the cubic's, so by Proposition 2.28 the cubic's training $\mathrm{RSS}$ cannot exceed the linear one's, whatever the truth. (b) The linear model is expected to have the lower test $\mathrm{RSS}$: both are unbiased for a linear truth, since the cubic contains it, and the cubic estimates two more coefficients, which adds variance and no reduction in bias (the decomposition of Unit 1). (c) The cubic again, by the same containment; the training $\mathrm{RSS}$ says nothing about the truth. (d) Not enough information: if the truth is far from linear the cubic's lower bias outweighs its extra variance and it wins on test data; if the truth is nearly linear the linear model wins as in (b). ■

**Exercise 2.31** (ISLR §3.7, exercise 5). For regression without an intercept, $\hat y_i=x_i\hat\beta$ with $\hat\beta=\sum_ix_iy_i/\sum_jx_j^2$. Write $\hat y_i=\sum_ja_jy_j$ and identify $a_j$.

**Solution.** Substituting Corollary 2.15,

$$
\hat y_i=x_i\frac{\sum_jx_jy_j}{\sum_lx_l^2}=\sum_ja_jy_j,\qquad a_j=\frac{x_ix_j}{\sum_lx_l^2}:
$$

the fitted values are linear combinations of the responses with weights fixed by the design. In general $\hat{\mathbf{Y}}=\mathbf{H}\mathbf{Y}$, so the weight of $Y_j$ in $\hat Y_i$ is $h_{ij}$; here $\mathbf{H}=\mathbf{x}\mathbf{x}'/(\mathbf{x}'\mathbf{x})$, whose $(i,j)$ entry is $a_j$. With an intercept, $\mathbf{H}\mathbf{1}=\mathbf{1}$ makes $\sum_jh_{ij}=1$ for every $i$, and the weight an observation gives itself is its leverage. ■

**Exercise 2.32** (ISLR §3.7, exercise 6). Show that the least squares line always passes through $(\bar x,\bar y)$.

**Solution.** By Corollary 2.14, $b_0=\bar y-b_1\bar x$, so the line's height at $\bar x$ is $b_0+b_1\bar x=\bar y$. It is the first normal equation, $\sum_ie_i=0$, and needs the intercept: regression through the origin passes through $(0,0)$ instead. ■

**Exercise 2.33** (ISLR §3.7, exercise 7). Prove $R^2=r^2$ for simple linear regression, assuming $\bar x=\bar y=0$.

**Solution.** With zero means, $b_1=\sum_ix_iy_i/\sum_ix_i^2$ and $b_0=0$, so $\mathrm{TSS}=\sum_iy_i^2$ and

$$
\mathrm{RSS}=\sum_i(y_i-b_1x_i)^2=\sum_iy_i^2-2b_1\sum_ix_iy_i+b_1^2\sum_ix_i^2
=\sum_iy_i^2-\frac{(\sum_ix_iy_i)^2}{\sum_ix_i^2} ,
$$

hence

$$
R^2=\frac{\mathrm{TSS}-\mathrm{RSS}}{\mathrm{TSS}}=\frac{(\sum_ix_iy_i)^2}{\sum_ix_i^2\sum_iy_i^2}=r^2
$$

by (1) with the means zero. Without the assumption, replace $x_i$ and $y_i$ by their deviations, which changes neither $R^2$ (Corollary 2.32) nor $r$ (Proposition 2.2). ■

**Exercise 2.34** (ISLR §3.7, exercise 8). Regress `mpg` on `horsepower` in `Auto`. (a) Is there a relation; how strong; positive or negative; the predicted `mpg` at $98$ horsepower with $95\%$ confidence and prediction intervals. (b) Plot the data with the line. (c) Diagnostic plots.

**Solution.**

In [ ]:
fit_hp <- lm(mpg ~ horsepower, data = Auto)
round(coef(summary(fit_hp)), 4)
c(p_slope = coef(summary(fit_hp))["horsepower", "Pr(>|t|)"])
c(R2 = summary(fit_hp)$r.squared, RSE = summary(fit_hp)$sigma, mean_mpg = mean(Auto$mpg),
  RSE_over_mean = summary(fit_hp)$sigma / mean(Auto$mpg))
rbind(confidence = predict(fit_hp, data.frame(horsepower = 98), interval = "confidence"),
      prediction = predict(fit_hp, data.frame(horsepower = 98), interval = "prediction"))

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.4, 1.4, 0.6), mgp = c(1.9, 0.6, 0))
plot(Auto$horsepower, Auto$mpg, xlab = "horsepower", ylab = "mpg", pch = 16, cex = 0.4)
abline(fit_hp, col = "#B45309", lwd = 2)
plot(fitted(fit_hp), residuals(fit_hp), xlab = "fitted", ylab = "residual", pch = 16, cex = 0.4)
abline(h = 0, lty = 2)
plot(fitted(fit_hp), rstandard(fit_hp), xlab = "fitted", ylab = "standardised residual", pch = 16, cex = 0.4)
abline(h = c(-2, 0, 2), lty = 2)
par(op)

(a) The slope $-0.158$ has $t=-24.5$ and a $p$-value of $7\times10^{-81}$: there is a relation, and it is negative, each additional unit of horsepower lowering the expected mileage by $0.16$ miles per gallon. $R^2=0.61$, and the residual standard error $4.9$ is $21\%$ of the mean mileage $23.4$. At $98$ horsepower the fitted value is $24.5$, the confidence interval $(24.0,25.0)$ and the prediction interval $(14.8,34.1)$. (b)–(c) The line underestimates mileage at low and high horsepower and overestimates it in the middle: the residuals against the fitted values form a U, the mark of a relation that is not linear; the standardised residuals also spread more at high fitted values. ■

**Exercise 2.35** (ISLR §3.7, exercise 9, parts (a)–(d)). For `Auto`: (a) a scatterplot matrix; (b) the correlation matrix excluding `name`; (c) regress `mpg` on all other variables: is there a relation, which predictors are significant, what does the `year` coefficient say; (d) diagnostic plots: large residuals, high leverage points.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
A <- Auto[, names(Auto) != "name"]
round(cor(A), 2)
fit_auto <- lm(mpg ~ ., data = A)
round(coef(summary(fit_auto)), 4)
c(R2 = summary(fit_auto)$r.squared, F = summary(fit_auto)$fstatistic[["value"]],
  max_std_resid = max(abs(rstandard(fit_auto))), at = which.max(abs(rstandard(fit_auto))),
  max_leverage = max(hatvalues(fit_auto)), at_lev = which.max(hatvalues(fit_auto)), flag = 2 * 8 / 392)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3.6)
pairs(A, pch = 16, cex = 0.25, gap = 0.2, cex.labels = 0.8)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 1.4, 0.6), mgp = c(1.9, 0.6, 0))
plot(fitted(fit_auto), rstandard(fit_auto), xlab = "fitted", ylab = "standardised residual", pch = 16, cex = 0.4)
abline(h = c(-2, 0, 2), lty = 2)
plot(hatvalues(fit_auto), rstandard(fit_auto), xlab = "leverage", ylab = "standardised residual", pch = 16, cex = 0.4)
abline(v = 2 * 8 / 392, lty = 2)
par(op)

(a)–(b) Mileage falls with cylinders, displacement, horsepower and weight, whose mutual correlations are $0.84$ to $0.95$, and rises with year and origin. (c) $F=252$ on $(7,384)$ degrees of freedom: a relation exists. At $5\%$ the significant predictors are `displacement`, `weight`, `year` and `origin`; `cylinders`, `horsepower` and `acceleration` are not, their simple associations being absorbed by the correlated size variables. The `year` coefficient $0.75$ says that, with the other variables fixed, a car one model year newer is expected to travel $0.75$ more miles per gallon. (d) The standardised residuals curve upward at high fitted values, so the relation is again not linear; observation $321$ has the largest standardised residual, $4.0$, and observation $14$ the largest leverage, $0.19$, nearly five times the flag $2(k+1)/n=0.04$. ■

**Exercise 2.36** (ISLR §3.7, exercise 11). Generate $x_i$ standard normal and $y_i=2x_i+\varepsilon_i$, $n=100$, with `set.seed(1)`. (a) Regress $y$ on $x$ without an intercept and report the coefficient, its standard error, $t$ and $p$-value. (b) The same for $x$ on $y$. (c) The relation between the two. (d) Show that for regression through the origin

$$
t=\frac{\sqrt{n-1}\,\sum_ix_iy_i}{\sqrt{(\sum_ix_i^2)(\sum_iy_i^2)-(\sum_ix_iy_i)^2}} ,
$$

and confirm it numerically. (e) Deduce that the $t$ statistics in (a) and (b) are equal. (f) Show numerically that the same holds with an intercept.

**Solution.** (d) With $\hat\beta=\sum_ix_iy_i/\sum_ix_i^2$ and, for one parameter and no intercept, $\mathrm{SE}(\hat\beta)=\sqrt{\sum_i(y_i-x_i\hat\beta)^2/\{(n-1)\sum_ix_i^2\}}$ (the proofs of Theorems 2.9(iv) and 2.22 and Definition 2.24 hold for any $\mathbf{X}$ of full column rank $p$, with $n-p$ in place of $n-k-1$; here $\mathbf{X}=\mathbf{x}$ has one column, so the divisor is $n-1$),

$$
t=\frac{\hat\beta}{\mathrm{SE}(\hat\beta)}
=\frac{\sum_ix_iy_i}{\sum_ix_i^2}\cdot\frac{\sqrt{n-1}\sqrt{\sum_ix_i^2}}{\sqrt{\sum_i(y_i-x_i\hat\beta)^2}} ,
\qquad
\sum_i(y_i-x_i\hat\beta)^2=\sum_iy_i^2-\frac{(\sum_ix_iy_i)^2}{\sum_ix_i^2} ,
$$

the second by expanding the square and substituting $\hat\beta$; multiplying numerator and denominator by $\sqrt{\sum_ix_i^2}$ gives the display. (e) The expression is symmetric in $x$ and $y$, so exchanging their roles cannot change it.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
set.seed(1); xa <- rnorm(100); ya <- 2 * xa + rnorm(100)
tf <- sqrt(99) * sum(xa * ya) / sqrt(sum(xa^2) * sum(ya^2) - sum(xa * ya)^2)
round(rbind(y_on_x = coef(summary(lm(ya ~ xa + 0)))[1, ], x_on_y = coef(summary(lm(xa ~ ya + 0)))[1, ]), 5)
c(closed_form = tf, with_intercept_y_on_x = coef(summary(lm(ya ~ xa)))[2, 3],
  with_intercept_x_on_y = coef(summary(lm(xa ~ ya)))[2, 3])

(a)–(c) The coefficient of $y$ on $x$ is $1.99$, that of $x$ on $y$ is $0.39$, and the two $t$ statistics are the same, $18.7$, as (d)–(e) require; the coefficients differ because $\sum_ix_i^2\neq\sum_iy_i^2$. (f) With an intercept the two $t$ statistics again coincide, by Exercise 2.16(c): $t=\sqrt{n-2}\,r/\sqrt{1-r^2}$ depends on the symmetric $r$ alone, so testing $\beta_1=0$ is testing $r=0$, a statement about the pair and not about a direction. ■

**Exercise 2.37** (ISLR §3.7, exercise 12). For regression through the origin: (a) when is the coefficient of $X$ on $Y$ equal to that of $Y$ on $X$? (b) Generate $n=100$ observations for which they differ; (c) for which they coincide.

**Solution.** (a) By Corollary 2.15 the two are $\sum_ix_iy_i/\sum_ix_i^2$ and $\sum_ix_iy_i/\sum_iy_i^2$, equal iff $\sum_ix_i^2=\sum_iy_i^2$ (or the numerator is zero).

In [ ]:
set.seed(4); xb <- rnorm(100); yb <- 3 * xb + rnorm(100)
c(y_on_x = coef(lm(yb ~ xb + 0))[[1]], x_on_y = coef(lm(xb ~ yb + 0))[[1]], sum_x2 = sum(xb^2), sum_y2 = sum(yb^2))
yc <- sample(xb)                                  # a permutation: the same squares
c(y_on_x = coef(lm(yc ~ xb + 0))[[1]], x_on_y = coef(lm(xb ~ yc + 0))[[1]], sum_x2 = sum(xb^2), sum_y2 = sum(yc^2))

(b) With $y=3x+\varepsilon$ the sums of squares differ and so do the coefficients. (c) A permutation of $x$ has the same sum of squares, and the two coefficients agree. ■

**Exercise 2.38** (ISLR §3.7, exercise 13). With `set.seed(1)`: (a) $x$, $100$ draws from $N(0,1)$; (b) `eps`, $100$ draws from $N(0,0.25)$; (c) $y=-1+0.5x+\varepsilon$: the length of $y$, and $\beta_0$, $\beta_1$; (d) a scatter plot; (e) the least squares fit against the truth; (f) the fitted and the population lines on the plot, with a legend; (g) a quadratic term: does it improve the fit? (h)–(i) repeat with less noise (variance $0.01$) and more noise (variance $1$); (j) the confidence intervals for $\beta_0$, $\beta_1$ in the three cases.

**Solution.**

In [ ]:
sim_fit <- function(sd_eps) {
  set.seed(1); x <- rnorm(100); eps <- rnorm(100, sd = sd_eps); y <- -1 + 0.5 * x + eps
  m <- lm(y ~ x)
  list(m = m, x = x, y = y, sd = sd_eps, ci = confint(m),
       quad = coef(summary(lm(y ~ x + I(x^2))))["I(x^2)", ])
}
s_mid <- sim_fit(0.5); s_low <- sim_fit(0.1); s_high <- sim_fit(1)
c(length_y = length(s_mid$y), beta0 = -1, beta1 = 0.5)
round(coef(summary(s_mid$m)), 4)
round(s_mid$quad, 4)
round(rbind(noise_0.25 = c(s_mid$ci[1, ], s_mid$ci[2, ]), noise_0.01 = c(s_low$ci[1, ], s_low$ci[2, ]),
            noise_1 = c(s_high$ci[1, ], s_high$ci[2, ])), 4)
round(rbind(noise_0.25 = coef(s_mid$m), noise_0.01 = coef(s_low$m), noise_1 = coef(s_high$m)), 4)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.4, 1.4, 0.6), mgp = c(1.9, 0.6, 0))
for (s in list(s_low, s_mid, s_high)) {
  plot(s$x, s$y, pch = 16, cex = 0.4, xlab = "x", ylab = "y", main = paste("sd(eps) =", s$sd))
  abline(s$m, col = "steelblue", lwd = 2); abline(-1, 0.5, col = "#B45309", lwd = 2, lty = 2)
  legend("topleft", c("least squares", "population"), col = c("steelblue", "#B45309"), lty = 1:2, bty = "n", cex = 0.7)
}
par(op)

(c) $y$ has length $100$, $\beta_0=-1$ and $\beta_1=0.5$. (d) A linear relation with scatter. (e) $\hat\beta_0=-1.02$ and $\hat\beta_1=0.50$, each within one standard error of the truth. (g) The quadratic term has $t=-1.40$ and $p=0.16$: no evidence of curvature, as expected of a linear truth. (h)–(j) With error variance $0.01$ the fit is $(-1.004,0.500)$ and the slope interval $(0.479,0.521)$; with variance $1$ the fit is $(-1.04,0.50)$ and the slope interval $(0.29,0.71)$: the interval width scales with $\sigma$ exactly as (14) says, the design being the same in all three cases, and all three intervals cover the truth. ■

**Exercise 2.39** (ISLR §3.7, exercise 15, parts (a)–(c)). Predict the per-capita crime rate `crim` in `Boston`. (a) Regress it on each predictor alone: which associations are significant? (b) Regress it on all predictors: for which can $H_0\colon\beta_j=0$ be rejected? (c) Plot the simple coefficients against the multiple ones.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
preds <- setdiff(names(Boston), "crim")
uni <- t(sapply(preds, function(v) { m <- lm(as.formula(paste("crim ~", v)), data = Boston)
  c(coef = coef(m)[[2]], p = coef(summary(m))[2, 4]) }))
fit_crim <- lm(crim ~ ., data = Boston)
multi <- coef(summary(fit_crim))[-1, c(1, 4)]
round(cbind(simple = uni[, "coef"], p_simple = uni[, "p"], multiple = multi[, 1], p_multiple = multi[, 2]), 4)
c(R2_multiple = summary(fit_crim)$r.squared)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
plot(uni[, "coef"], multi[, 1], xlab = "simple regression coefficient", ylab = "multiple regression coefficient",
     pch = 16)
text(uni[, "coef"], multi[, 1], preds, pos = 4, cex = 0.6)
abline(0, 1, lty = 2)
par(op)

(a) Alone, every predictor but `chas` ($p=0.21$) is significantly associated with the crime rate. (b) Together, $H_0$ is rejected at $5\%$ for `zn`, `dis`, `rad` and `medv` only, with `nox` and `lstat` near the boundary; the model explains $45\%$ of the variation. (c) The two sets of coefficients are unrelated, and `nox` moves from $31$ alone to $-10$ in the joint model: its simple association is carried by the predictors it is correlated with, Example 2.10 on a larger scale. ■

**Exercise 2.40** (SOA sample question 11). A regression model gives, for five observations, $y=(2,5,6,8,4)$ and $\hat f(x)=(4,3,9,3,6)$. Calculate the sum of squared errors: (A) $-35$; (B) $-5$; (C) $5$; (D) $35$; (E) $46$.

**Solution.** By Definition 2.16, $\mathrm{SSE}=\sum_i(y_i-\hat y_i)^2=(-2)^2+2^2+(-3)^2+5^2+(-2)^2 =4+4+9+25+4=46$, which is (E). (A) and (B) are impossible, a sum of squares being non-negative; the residuals themselves sum to $-2+2-3+5-2=0$, and neither $5$, the number of observations, nor $35$ is a sum of squares of these residuals. ■

**Exercise 2.41** (SOA sample question 18). For a simple linear regression on $25$ observations, $\sum_ie_i^2=230$ and $R^2=0.64$. Calculate the total sum of squares: (A) $605.94$; (B) $638.89$; (C) $690.77$; (D) $701.59$; (E) $750.87$.

**Solution.** By (18), $R^2=1-\mathrm{SSE}/\mathrm{SST}$, so $\mathrm{SST}=\mathrm{SSE}/(1-R^2)=230/0.36=638.89$, which is (B); $n=25$ is not needed, and $\mathrm{SSR}=\mathrm{SST}-\mathrm{SSE}=408.89$ checks as $0.64\times638.89$. None of the other options equals $230/(1-R^2)$; they would require $1-R^2$ of $0.380$, $0.333$, $0.328$ or $0.306$. ■

**Exercise 2.42** (SOA sample question 23). A cafeteria sells coffee at $1.00$ for $12$ ounces, $1.20$ for $16$ and $1.40$ for $20$, and will price any size at the least squares regression of price on size. Toby and Karen each buy a $24$-ounce coffee; calculate what they save by sharing one $48$-ounce coffee instead: (A) it costs $0.40$ more; (B) the same; (C) they save $0.40$; (D) $0.80$; (E) $1.20$.

**Solution.** The three points are collinear, so by Proposition 2.2 $r=1$ and the least squares line passes through all three with $\mathrm{SSE}=0$: the slope is $(1.40-1.00)/(20-12)=0.05$ per ounce and the intercept $1.00-0.05(12)=0.40$; the same comes from (11), with $\bar x=16$, $\bar y=1.20$, $b_1=\{(-4)(-0.2)+0+(4)(0.2)\}/\{16+0+16\}=1.6/32=0.05$. Two $24$-ounce cups cost $2(0.40+1.20)=3.20$; one $48$-ounce cup costs $0.40+2.40=2.80$. They save $0.40$, the fixed charge of the second cup, which is (C). (A) would follow from adding, not subtracting, the intercept; (B) from a line through the origin, which the data reject; (D) and (E) from doubling or tripling the intercept. ■

**Exercise 2.43** (SOA sample question 24). A regression of a mutual fund's return on four predictors plus an intercept over $105$ months reports $F=20.0$; the software failed before computing $R^2$. Which is true: (A) insufficient information, but $R^2$ could be calculated given $\mathrm{RSS}$; (B) insufficient information, but it could be calculated given $\mathrm{TSS}$ and $\mathrm{RSS}$; (C) $R^2=0.44$; (D) $R^2=0.56$; (E) $R^2=0.80$.

**Solution.** By Corollary 2.39, $R^2$ depends on $\mathrm{SSE}$ and $\mathrm{SST}$ only through their ratio, and $F$ fixes that ratio: with $k=4$ and $n-k-1=100$,

$$
F=\frac{(\mathrm{SST}-\mathrm{SSE})/4}{\mathrm{SSE}/100}=20\;\Longleftrightarrow\;\frac{\mathrm{SST}-\mathrm{SSE}}{\mathrm{SSE}}=0.80
\;\Longleftrightarrow\;\frac{\mathrm{SST}}{\mathrm{SSE}}=1.80 ,
$$

so $R^2=1-1/1.80=0.444$, which is (C); the corollary gives it directly as $kF/(kF+n-k-1)=80/180$. (A) and (B) are false because nothing beyond $F$, $n$ and $k$ is needed. (D) is $1-R^2=\mathrm{SSE}/\mathrm{SST}=0.556$, the unexplained share, and (E) is $kF/(n-k-1)=\mathrm{SSR}/\mathrm{SSE}=0.80$, the ratio of explained to unexplained variation. ■

**Exercise 2.44** (SOA sample question 44). Dental claims of $n=100$ participants with the predictor sex, coded $0$ and $1$. Actuary 1 fits $Y=\beta+\varepsilon$ and Actuary 2 fits $Y=\beta_0+\beta_1\,\mathrm{Sex}+\varepsilon$, with $\mathrm{RSS}=250{,}000$ and $\mathrm{TSS}=490{,}000$. Calculate the $F$ statistic testing whether Actuary 2's model is a significant improvement: (A) $92$; (B) $93$; (C) $94$; (D) $95$; (E) $96$.

**Solution.** Actuary 1's model is the intercept-only model, whose residual sum of squares is $\mathrm{TSS}$, so nothing further about it is needed. By (23) with $k=1$,

$$
F=\frac{(\mathrm{TSS}-\mathrm{RSS})/1}{\mathrm{RSS}/(n-2)}=\frac{240{,}000}{250{,}000/98}=94.08 \quad\text{on }F_{1,98},
$$

which is (C). The other options are the same ratio with the wrong residual degrees of freedom, $96$, $97$, $99$ and $100$ in place of $n-2=98$. Two checks: by Proposition 2.40, $t(b_1)=\sqrt{94.08}=9.70$, and by Exercise 2.9 this is the pooled two-sample $t$ statistic comparing the mean claims of the two sexes; and $R^2=1-250/490=0.4898$ reproduces $F$ through (23). ■

**Exercise 2.45** (SOA sample question 53). Which statement about $Y=\beta_0+\beta_1X+\varepsilon$ is NOT true: (A) $\beta_0$ is the expected value of $Y$; (B) $\beta_1$ is the average increase in $Y$ associated with a one-unit increase in $X$; (C) $\varepsilon$ is typically assumed independent of $X$; (D) the equation defines the population regression line; (E) least squares is commonly used to estimate $\beta_0$ and $\beta_1$.

**Solution.** (A) is false, hence the answer: by A1, $\operatorname{\mathsf{E}}[Y\mid X]=\beta_0+\beta_1X$, so $\beta_0=\operatorname{\mathsf{E}}[Y\mid X=0]$, the expected response when $X=0$, and the unconditional expectation is $\beta_0+\beta_1\operatorname{\mathsf{E}}[X]$. (B) is the interpretation of a coefficient in §1.2, $\beta_1=\partial\operatorname{\mathsf{E}}[Y\mid X]/\partial X$; (C) is a stronger form of A2: independence of $\varepsilon$ and $X$ together with $\operatorname{\mathsf{E}}[\varepsilon]=0$ implies A2; (D) is the population regression line of the same remark; (E) is Definition 2.5. ■